<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_08_visual_input_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 · Sector 8 — Visual input preparation

Consumes verified S7 outputs, preserves originals by reference, and generates only new visual products. All code is included here; no upstream notebooks are imported or executed.


Published tables verified when this notebook was built: **66,127 files**, **47,239 annotations**, **110,666 temporal references**. S7 retains `final_split = null`, `split_assignment_status = pending`, and `training_ready = false`; S8 preserves the same semantics. `image_usable_for_modelling = false` does not exclude preparation of a technically usable image.


## 1. Products and variants

| Product | What it generates | When it is available |
|---|---|---|
| `full_frame` | Reference to the original, without a new copy | Technically usable S7 source |
| `roi_crop` | Cropped JPEG with a configurable margin | Valid bbox within the image dimensions |
| `bbox_overlay` | Full frame with all valid bounding boxes drawn | At least one valid bbox |
| `evidence_panel` | Full frame + crop in one JPEG | Valid bbox |
| `temporal_collage` | Neighboring frames in order, with the target and offsets | S7 window containing a target and a neighbor |
| `progression_panel` | Verified anchors along the segment | At least two distinct verified frames |
| `quality_panel` | Target and nearby frames showing variation in QC metrics | At least two distinct frames from the same video |
| `comparison_panel` | Finding and an explicitly labeled normal reference | Verified normal reference in the same video |
| `combined_panel` | Frame, available crop/overlay, and temporal context | At least two available views |

Panels use neutral A/B/C markers or frame/time identifiers. They do not print diagnoses, classes, QC values, or answers. Normal/abnormal positions are permuted deterministically. Bbox-derived overlays/crops are marked `annotation_assisted`.

Here, “progression” means comparing **the appearance of observed anchors**, without claims about clinical progression, onset, end, or continuous visibility. QC metrics are uncalibrated technical indicators; a QC panel does not automatically yield a correct clinical answer. An unlabeled frame is not normal by default.

### Common manifest
`visual_manifest` indexes originals and products. `visual_source_manifest` preserves each product's links to all its sources; `source_evidence_manifest` keeps labels/verification records/QC separate from the object passed to the model. Originals are not physically duplicated.

### What the model receives
`benchmark_input_manifest` contains **ordered recipes**, not invented QA:

| Variant | Images actually loaded |
|---|---|
| `full` | Original |
| `crop` | Crop |
| `fullcrop` | Two images: original, crop |
| `fullcrop_panel` | One composite JPEG: original + crop |
| `overlay` | Original with bounding boxes |
| `temporal_multi` | List of originals in temporal order |
| `temporal_collage` | One temporal JPEG |
| `progression`, `qc`, `comparison`, `combined` | Corresponding panel |

`s8_load_model_input(example, question, CONFIG, run_dir)` returns **only** `question`, `images` (PIL RGB objects), and `image_roles`. It does not pass paths containing class names, labels, or ground truth. Validated questions and answers are added later. An actual loading example is included in section 10.

`same_cohort_eligible` indicates variant availability for the same annotation; question equivalence is validated separately. `shared_source_groups` links physical frames, files with the same SHA, and sources used together. The groups are conservative across all generated variants and may become large. After the final variants are selected, the benchmark stage assigns splits and may recompute groups on the selected subset.

## 2. Dependencies, imports, and configuration

The recorded successful S7 run used an NVIDIA L4, Python 3.13.15, CuPy 14.2.0, CUDA runtime 12.9, and a driver supporting CUDA 13. It decoded all 66,127 files on the GPU. S8 uses that CUDA 12 package family: `cupy-cuda12x==14.2.0`, `nvidia-nvimgcodec-cu12==0.9.0.20`, and `nvidia-nvjpeg-cu12==12.4.0.76`.

The installer inventories the environment and removes conflicting CuPy/nvImageCodec variants before installing one coherent family. Python 3.13 does not imply CUDA 13; the CUDA maximum printed by `nvidia-smi` is driver capability, not the active runtime. Installation errors are shown, not suppressed. Set `force_repair=True` only if a single installed distribution is itself damaged.

**If the installer reports that already-loaded binary packages changed, use Colab → Runtime → Restart session, then rerun from this section.** Repeating the cell without restarting is insufficient. Do not use **Disconnect and delete runtime**: a session restart preserves files on `/content`. If installing into a fresh session requires no restart, continue directly. All imports are confined to this section: standard-library imports in the dependency cell, then numerical/GPU/Drive imports in the next cell.

The early GPU preflight in section 5 validates actual JPEG decoding and reports device, runtime, package versions, backend warnings, and complete exception messages before any dataset copying or long resume verification. A GPU VM can still have a broken decoder installation.

`require_gpu_decode=True` requires NVIDIA GPU-accelerated JPEG decoding and device-resident pixels. `HW_GPU_ONLY`/`GPU_ONLY` are tried first. `allow_nvidia_hybrid=True` additionally permits nvJPEG's explicitly recorded `HYBRID_CPU_GPU` backend, which can use CPU header/entropy work while returning CUDA pixels; GPU cropping is retained. Set `require_gpu_only_backend=True` to keep the original pure-backend restriction. CPU-only JPEG decoding remains disabled by `allow_cpu_fallback=False`. Panel composition and JPEG encoding use the CPU, as before.

A logical batch contains at most **1,000 unique source files**. Adaptive subbatches and host spill limit GPU memory. Repeated uses read SSD, without another Drive copy. `progress_bars=True` shows aggregate copy, SHA, rendering, API inventory, upload, and confirmation progress. Hash and cloud-confirmation rules are unchanged.

References: [successful S7 notebook](https://github.com/bogdanparvu18/msc-graduate-project/blob/8645420b2d86d97ae536ceea9a8fa846564e936f/notebooks/phase2/phase2_07_image_quality_control.ipynb), [S7 lineage](https://github.com/bogdanparvu18/msc-graduate-project/blob/8645420b2d86d97ae536ceea9a8fa846564e936f/outputs/phase2/sector7/configs/sector7_lineage.json), [CuPy installation](https://docs.cupy.dev/en/stable/install.html), [NVIDIA nvImageCodec API](https://docs.nvidia.com/cuda/nvimagecodec/py_api.html).


In [1]:
"""Dependency repair before importing numerical/CUDA packages into this session.

CUDA 12 is the default because it is the recorded working S7 stack. Python 3.13
and the CUDA maximum displayed by nvidia-smi do not imply CUDA 13 packages.
"""
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import subprocess
import sys


def s8_dependency_inventory():
    return {re.sub(r'[-_.]+', '-', dist.metadata['Name']).lower(): dist.version
            for dist in importlib.metadata.distributions() if dist.metadata.get('Name')}


def s8_dependency_plan(inventory, cuda_major=12, force_repair=False):
    """Pure planning: mutually exclusive packages share import paths, so repair all."""
    if int(cuda_major) not in (12, 13):
        raise ValueError('S8 supports an explicit CUDA package family of 12 or 13.')
    cuda_major = int(cuda_major)
    cupy_name = f'cupy-cuda{cuda_major}x'
    codec_name = f'nvidia-nvimgcodec-cu{cuda_major}'
    installed_cupy = sorted(name for name in inventory if name == 'cupy' or name.startswith('cupy-cuda'))
    installed_codec = sorted(name for name in inventory if name.startswith('nvidia-nvimgcodec'))
    remove = []
    if force_repair or installed_cupy != [cupy_name]:
        remove.extend(installed_cupy)
    if force_repair or installed_codec != [codec_name]:
        remove.extend(installed_codec)
    # Same tested module releases as the S7 lineage; one CUDA family only.
    gpu_requirements = [f'{cupy_name}[ctk]==14.2.0', f'{codec_name}[nvjpeg]==0.9.0.20']
    if cuda_major == 12:
        gpu_requirements.append('nvidia-nvjpeg-cu12==12.4.0.76')
    requirements = ['numpy>=2.0,<3', 'pandas>=2.2,<3', 'pyarrow>=16',
                    'opencv-python-headless>=4.10,<6', 'Pillow', 'tqdm',
                    'requests', 'google-auth'] + gpu_requirements
    return {'cuda_major': cuda_major, 'cupy_package': cupy_name, 'nvimgcodec_package': codec_name,
            'remove': sorted(set(remove)), 'requirements': requirements,
            's7_reference': {'cupy': '14.2.0', 'nvimgcodec': '0.9.0.20',
                             'nvjpeg_cu12': '12.4.0.76', 'cuda_runtime': '12.9'}}


def s8_restart_needed(before, after, loaded_modules, repaired_packages=()):
    package_roots = {'numpy': 'numpy', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
                     'opencv-python-headless': 'cv2', 'pillow': 'PIL'}
    changed = {name for name in set(before) | set(after) if before.get(name) != after.get(name)}
    changed.update(repaired_packages)
    affected = []
    for name in sorted(changed):
        root = package_roots.get(name)
        if name == 'cupy' or name.startswith('cupy-cuda'):
            root = 'cupy'
        elif name.startswith(('nvidia-nvimgcodec', 'nvidia-nvjpeg', 'nvidia-cuda-', 'cuda-')):
            # nvImageCodec/CuPy may have already dlopen'ed a replaced CUDA library.
            if any(m == 'cupy' or m.startswith('cupy.') or m == 'nvidia.nvimgcodec'
                   or m.startswith('nvidia.nvimgcodec.') for m in loaded_modules):
                affected.append(name)
            continue
        if root and any(m == root or m.startswith(root + '.') for m in loaded_modules):
            affected.append(name)
    return affected


def s8_install_dependencies(cuda_major=12, force_repair=False):
    """Repair duplicate GPU distributions; require a restart for loaded replacements."""
    if globals().get('S8_DEPENDENCY_RESTART_REQUIRED'):
        raise RuntimeError('A dependency repair already changed loaded binary packages in this session. '
                           'Use Runtime > Restart session in Colab, then rerun from the first cell. '
                           'Repeating this cell without a session restart is not sufficient.')
    before = s8_dependency_inventory()
    loaded = set(sys.modules)
    plan = s8_dependency_plan(before, cuda_major, force_repair)
    probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False, timeout=30)
    if probe.returncode:
        raise RuntimeError('NVIDIA driver query failed: ' + (probe.stderr or probe.stdout).strip()
                           + '. This is a driver/runtime diagnostic, not evidence about installed CuPy.')
    driver_match = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', probe.stdout)
    driver_max = int(driver_match.group(1)) if driver_match else None
    if driver_max is not None and int(cuda_major) > driver_max:
        raise RuntimeError(f'Selected CUDA {cuda_major} packages exceed driver capability CUDA {driver_max}.')
    print(probe.stdout.strip())
    print('Python:', sys.version.split()[0], '| selected CUDA package family:', cuda_major,
          '| driver CUDA maximum:', driver_max)
    print('The driver CUDA maximum is not the installed toolkit/runtime version.')
    print('Repair plan:', json.dumps(plan, indent=2))
    try:
        if plan['remove']:
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', *plan['remove']], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:',
                        *plan['requirements']], check=True)
        importlib.invalidate_caches()
        after = s8_dependency_inventory()
        installed_cupy = sorted(name for name in after if name == 'cupy' or name.startswith('cupy-cuda'))
        installed_codec = sorted(name for name in after if name.startswith('nvidia-nvimgcodec'))
        if installed_cupy != [plan['cupy_package']] or installed_codec != [plan['nvimgcodec_package']]:
            raise RuntimeError('GPU package repair did not produce one CuPy and one nvImageCodec distribution: '
                               + repr({'cupy': installed_cupy, 'nvimgcodec': installed_codec}))
    except BaseException as error:
        # pip is not transactional. Failed installs/uninstalls and interrupted
        # cells can replace binary files before package metadata is updated.
        # Preserve the restart guard even when a later rerun sees those new
        # disk versions as its starting inventory.
        requested = [re.split(r'[\[<>=!~]', value, maxsplit=1)[0].lower()
                     for value in plan['requirements']]
        potentially_changed = set(plan['remove']) | set(requested)
        inventory_error = None
        try:
            importlib.invalidate_caches()
            current = s8_dependency_inventory()
        except BaseException as inventory_failure:
            current = before
            potentially_changed.update(before)
            inventory_error = type(inventory_failure).__name__
        affected = s8_restart_needed(before, current, loaded, potentially_changed)
        report = {'status': 'restart_required' if affected else 'repair_failed',
                  'cuda_major': int(cuda_major), 'repair_error': type(error).__name__,
                  'potentially_changed_loaded_packages': affected,
                  'inventory_error': inventory_error}
        globals()['S8_DEPENDENCY_REPORT'] = report
        if affected:
            globals()['S8_DEPENDENCY_RESTART_REQUIRED'] = True
            print('Dependency repair failed or was interrupted after binary packages may have changed. '
                  'Use Runtime > Restart session in Colab, then rerun from the first cell. '
                  'Files already staged on /content are preserved by a session restart.')
        raise
    affected = s8_restart_needed(before, after, loaded, plan['remove'])
    report = {'status': 'restart_required' if affected else 'ready_for_imports',
              'cuda_major': int(cuda_major), 'changed_loaded_packages': affected,
              'gpu_packages': {name: version for name, version in after.items()
                               if name in installed_cupy + installed_codec
                               or name.startswith(('nvidia-nvjpeg', 'nvidia-cuda-runtime'))}}
    globals()['S8_DEPENDENCY_REPORT'] = report
    print(json.dumps(report, indent=2))
    if affected:
        globals()['S8_DEPENDENCY_RESTART_REQUIRED'] = True
        raise RuntimeError('Dependency repair completed, but binary packages already loaded in this session changed. '
                           'Use Runtime > Restart session in Colab (not Disconnect and delete runtime), '
                           'then rerun from the first cell. Files already staged on /content are preserved by a session restart.')
    return report

# Reproduce the CUDA 12 package family validated in S7, with no duplicate variants.
S8_DEPENDENCY_REPORT = s8_install_dependencies(cuda_major=12, force_repair=False)


Fri Oct 23 02:10:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# All numerical, rendering, progress, and optional Colab/Drive imports.
# Run only after the dependency cell has completed without a restart request.
if globals().get('S8_DEPENDENCY_RESTART_REQUIRED'):
    raise RuntimeError('Restart the Colab session, then rerun from the dependency cell before importing binaries.')

from pathlib import Path
from pathlib import Path, PurePosixPath
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import json
import math
import os
import urllib.request
import urllib.parse
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import gc
import importlib.metadata
import platform
import time
import warnings
import cv2
from contextlib import contextmanager
from datetime import datetime, timezone
from urllib.parse import urlsplit
import fcntl
import gzip
import copy
import threading
import shutil
import uuid
from PIL import Image
import re
from collections import Counter
import tempfile
from IPython.display import display

S8_GPU_IMPORT_ERRORS = {}
try:
    import cupy as cp
except Exception as exc:
    cp = None
    S8_GPU_IMPORT_ERRORS['cupy'] = type(exc).__name__ + ': ' + str(exc)

try:
    from nvidia import nvimgcodec
except Exception as exc:
    nvimgcodec = None
    S8_GPU_IMPORT_ERRORS['nvimgcodec'] = type(exc).__name__ + ': ' + str(exc)

try:
    import google.auth as google_auth
    from google.auth.transport.requests import AuthorizedSession
except ImportError:
    google_auth = None
    AuthorizedSession = None

try:
    from google.colab import auth as colab_auth
except ImportError:
    colab_auth = None

try:
    from google.colab import drive
except ImportError:
    drive = None


In [3]:

CONFIG = {
    # One immutable, reviewed main snapshot; update intentionally after a new S7 run.
    'repository': 'bogdanparvu18/msc-graduate-project',
    'github_commit': 'cb4ac8d8f94caeb83dcddfccf495cdd4188bf9f0',
    'sector7_relative_root': 'outputs/phase2/sector7',
    'sector8_relative_root': 'outputs/phase2/sector8',
    'storage_root': '/content/drive/MyDrive/MMVQA_Clinical',
    'local_root': '/content/mmvqa_sector8_v2',
    'drive_mount': '/content/drive',
    'mount_google_drive': True,
    'existing_ssd_roots': [],  # Optional existing S6/S7 SSD trees with the same relative paths.
    'io_workers': 32,
    'progress_bars': True,
    'min_free_gib': 2,

    # A logical batch contains at most 1000 UNIQUE source files, not annotation rows.
    'batch_size': 1000,
    'gpu_device_id': 0,
    'gpu_memory_fraction': 0.65,
    'require_gpu_decode': True,
    'allow_nvidia_hybrid': True,  # GPU-accelerated nvJPEG; backend is recorded explicitly.
    'require_gpu_only_backend': False,  # True retains the hardware/GPU-only restriction.
    'allow_cpu_fallback': False,
    'allow_non_jpeg_cpu': True,
    'jpeg_quality': 95,
    'panel_size': 336,
    'roi_margin_fraction': 0.20,
    'temporal_max_frames': 5,
    'progression_max_frames': 5,
    'quality_max_frames': 3,
    'quality_max_frame_gap': 30,
    'generate_roi': True,
    'generate_overlays': True,
    'generate_temporal': True,
    'generate_evidence_panels': True,
    'generate_progression': True,
    'generate_quality_panels': True,
    'generate_comparison_panels': True,
    'generate_combined_panels': True,
    'include_qc_review': True,
    'max_subjects': None,  # Set 100 for a Colab smoke run, then None for the full run.

    # One outbound payload per changed file; wait for fresh Drive API checksums.
    'drive_ack_timeout_seconds': 3600,
    'drive_ack_poll_seconds': 2,
    'drive_retry_seconds': 90,
    'drive_api_workers': 4,
    'github_publish': False,  # No commit/push implementation is executed by this notebook.
}


## 3. Verified loading from GitHub

Only the required S7 metadata is downloaded, from the pinned commit. The catalog is checked against the hash embedded in the notebook; tables, schemas, configuration, and the report are checked against the **file** SHAs in the catalog. Semantic fingerprints do not replace the file hash. A valid local cache is reused; a corrupted file is downloaded again.

Changing the commit also requires `sector7_state_sha256` for the new catalog. The upstream reference remains reproducible even as `main` advances.

In [4]:
S8_PINNED_REPOSITORY = 'bogdanparvu18/msc-graduate-project'
S8_PINNED_COMMIT = 'cb4ac8d8f94caeb83dcddfccf495cdd4188bf9f0'
S8_PINNED_STATE_SHA256 = 'a90f4048491f9aef0be931b2d4c85cf4a8ac61236534d44ca50c3331b073fd69'
S8_UPSTREAM_TABLES = ('image_qc_manifest', 'labelled_frame_manifest_with_qc', 'temporal_manifest_with_qc')
S8_UPSTREAM_CONTRACT = {
    'training_ready': False, 'split_policy': 'deferred',
    'split_assignment_status': 'pending',
    'threshold_fit_policy': 'deferred_until_validated_final_split',
}
S8_UPSTREAM_CATALOG_FIELDS = (
    'schema_version', 'complete', 'processing_signature', 'semantic_config_fingerprint',
    'input_sha256', 'core_source_sha256', 'config_snapshot', 'tables', 'summary',
    'artifacts', 'obsolete_paths', 'config_snapshot_fingerprint', 'execution_contract',
)


def s8u_json_bytes(value):
    return (json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False, allow_nan=False) + '\n').encode('utf-8')


def s8u_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()


def s8u_hash(value, name):
    if not isinstance(value, str) or len(value) != 64 or any(c not in '0123456789abcdef' for c in value):
        raise ValueError(f'{name} must be a lowercase hexadecimal SHA-256.')
    return value


def s8u_relative(value):
    if not isinstance(value, str) or '\\' in value:
        raise ValueError(f'Unsafe source relative path: {value!r}')
    path = PurePosixPath(value)
    if path.is_absolute() or '..' in path.parts or value in ('', '.') or path.as_posix() != value:
        raise ValueError(f'Unsafe or noncanonical source relative path: {value!r}')
    return value


def s8u_safe_path(root, relative):
    root = Path(root).resolve()
    path = root / s8u_relative(relative)
    if not path.resolve().is_relative_to(root):
        raise ValueError('Resolved cache path escapes its declared root.')
    return path


def s8u_fetch_cached(url, destination, expected_sha256, expected_bytes=None):
    """A valid local cache avoids downloads; a corrupt cache is replaced atomically.

    The authority is the SHA-256 in the pinned upstream catalog. Never accept a
    changed file by changing the expected digest. File hashes are bytes hashes,
    not pandas serialization fingerprints.
    """
    expected_sha256 = s8u_hash(expected_sha256, 'Expected metadata hash')
    destination = Path(destination)
    if destination.is_symlink():
        raise ValueError('Metadata cache files must not be symbolic links.')
    if destination.is_file():
        size_ok = expected_bytes is None or destination.stat().st_size == int(expected_bytes)
        if size_ok and s8u_sha256(destination) == expected_sha256:
            return {'status': 'verified_cache', 'relative_file': destination.name,
                    'sha256': expected_sha256, 'bytes': destination.stat().st_size}
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.partial-' + str(os.getpid()))
    digest, length = hashlib.sha256(), 0
    try:
        request = urllib.request.Request(url, headers={'User-Agent': 'MMVQA-Sector8/2', 'Accept-Encoding': 'identity'})
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as output:
            for block in iter(lambda: response.read(8 * 1024**2), b''):
                digest.update(block)
                length += len(block)
                if expected_bytes is not None and length > int(expected_bytes):
                    raise RuntimeError('Downloaded metadata exceeds its pinned catalog size.')
                output.write(block)
            output.flush()
            os.fsync(output.fileno())
        if (digest.hexdigest() != expected_sha256 or
                (expected_bytes is not None and length != int(expected_bytes))):
            raise RuntimeError('Pinned GitHub metadata SHA-256/size mismatch: ' + destination.name)
        if s8u_sha256(temporary) != expected_sha256:
            raise RuntimeError('SSD metadata readback SHA-256 mismatch: ' + destination.name)
        os.replace(temporary, destination)
        return {'status': 'downloaded', 'relative_file': destination.name,
                'sha256': expected_sha256, 'bytes': length}
    finally:
        temporary.unlink(missing_ok=True)


def s8u_validate_deferred(frame, name):
    required = {'final_split', 'split_assignment_status', 'training_ready',
                'image_usable_for_modelling', 'image_technically_usable', 'qc_thresholds_calibrated'}
    missing = sorted(required - set(frame))
    if missing:
        raise RuntimeError(f'{name} lacks deferred-preparation fields: {missing}')
    if frame['final_split'].notna().any() or not frame['split_assignment_status'].eq('pending').fillna(False).all():
        raise RuntimeError(f'{name} must retain missing final_split and pending assignment.')
    for field in ('training_ready', 'image_usable_for_modelling', 'qc_thresholds_calibrated'):
        if not pd.api.types.is_bool_dtype(frame[field].dtype) or frame[field].fillna(True).any():
            raise RuntimeError(f'{name}/{field} must explicitly be False until final model selection.')
    return True


def s8_load_inputs(config):
    """Load immutable GitHub metadata and only the three consumed S7 data tables.

    Images are staged separately from Drive. No Drive metadata or earlier-sector
    filename discovery is used to select the current S7 run.
    """
    repository = config.get('repository', S8_PINNED_REPOSITORY)
    commit = config.get('github_commit', S8_PINNED_COMMIT)
    if repository != S8_PINNED_REPOSITORY:
        raise ValueError('This notebook is pinned to the MMVQA project repository.')
    if not isinstance(commit, str) or len(commit) != 40 or any(c not in '0123456789abcdef' for c in commit):
        raise ValueError('github_commit must be an explicit 40-character commit SHA, never a moving branch.')
    expected_state = config.get('sector7_state_sha256')
    if expected_state is None:
        if commit != S8_PINNED_COMMIT:
            raise ValueError('A different explicit commit also requires its verified sector7_state_sha256.')
        expected_state = S8_PINNED_STATE_SHA256
    s8u_hash(expected_state, 'Pinned S7 catalog hash')
    sector_root = s8u_relative(config.get('sector7_relative_root', 'outputs/phase2/sector7'))
    local_root = Path(config['local_root']).expanduser().resolve()
    if not local_root.is_absolute() or 'drive' in [p.lower() for p in local_root.parts]:
        raise ValueError('local_root must be a local SSD directory, outside the Drive mount.')
    cache_root = local_root / 'cache' / 'upstream' / commit / sector_root
    url_root = ('https://raw.githubusercontent.com/' + repository + '/' + commit + '/'
                + urllib.parse.quote(sector_root, safe='/') + '/')
    state_relative = 'configs/sector7_state.json'
    state_path = s8u_safe_path(cache_root, state_relative)
    with tqdm(total=1, desc='Fetch / verify S7 catalog SHA-256', unit='file',
              disable=not config.get('progress_bars', True)) as progress:
        transfers = [s8u_fetch_cached(url_root + state_relative, state_path, expected_state)]
        progress.update(1)
    state = json.loads(state_path.read_bytes())
    if state.get('schema_version') != 3 or state.get('complete') is not True:
        raise RuntimeError('Sector 8 v2 requires a complete S7 schema_version=3 catalog.')
    if state.get('execution_contract') != S8_UPSTREAM_CONTRACT:
        raise RuntimeError('S7 catalog does not declare the supported deferred-split QC contract.')
    try:
        semantic_state = {key: state[key] for key in S8_UPSTREAM_CATALOG_FIELDS}
    except KeyError as exc:
        raise RuntimeError('S7 catalog lacks a semantic contract field.') from exc
    if state.get('content_id') != hashlib.sha256(s8u_json_bytes(semantic_state)).hexdigest():
        raise RuntimeError('S7 catalog content_id verification failed.')
    catalog = {}
    for record in state['artifacts']:
        relative = s8u_relative(record['relative_path'])
        if relative in catalog:
            raise RuntimeError('Duplicate S7 catalog artifact path: ' + relative)
        s8u_hash(record['sha256'], relative)
        if not isinstance(record['bytes'], int) or isinstance(record['bytes'], bool) or record['bytes'] < 0:
            raise RuntimeError('Invalid S7 catalog byte size: ' + relative)
        catalog[relative] = record
    required = {state['config_snapshot'], 'configs/sector7_lineage.json', state['summary']['summary_relative_path']}
    for name in S8_UPSTREAM_TABLES:
        if name not in state['tables']:
            raise RuntimeError('S7 lacks required data product: ' + name)
        required.update((state['tables'][name]['relative_path'], state['tables'][name]['schema_path']))
    if not required.issubset(catalog):
        raise RuntimeError('A consumed S7 artifact is absent from its catalog.')

    def fetch(relative):
        record = catalog[relative]
        result = s8u_fetch_cached(url_root + urllib.parse.quote(relative, safe='/'),
                                 s8u_safe_path(cache_root, relative), record['sha256'], record['bytes'])
        return {'relative_path': relative, **result}

    with ThreadPoolExecutor(max_workers=max(1, min(6, int(config.get('metadata_workers', 4))))) as pool:
        futures = {pool.submit(fetch, relative): relative for relative in sorted(required)}
        for future in tqdm(as_completed(futures), total=len(futures), unit='file',
                           desc='Fetch / verify S7 metadata SHA-256',
                           disable=not config.get('progress_bars', True)):
            transfers.append(future.result())
    snapshot = json.loads(s8u_safe_path(cache_root, state['config_snapshot']).read_bytes())
    lineage = json.loads(s8u_safe_path(cache_root, 'configs/sector7_lineage.json').read_bytes())
    summary = json.loads(s8u_safe_path(cache_root, state['summary']['summary_relative_path']).read_bytes())
    if (lineage.get('input_sha256') != state['input_sha256'] or
            lineage.get('core_source_sha256') != state['core_source_sha256']):
        raise RuntimeError('S7 lineage disagrees with its catalog.')
    if snapshot.get('semantic_config_fingerprint') != state['semantic_config_fingerprint']:
        raise RuntimeError('S7 effective configuration identity disagrees with its catalog.')
    if not isinstance(snapshot.get('effective_config'), dict):
        raise RuntimeError('S7 configuration snapshot lacks effective_config.')
    if summary.get('material_validation_passed') is not True or any(summary.get(k) != v for k, v in S8_UPSTREAM_CONTRACT.items()):
        raise RuntimeError('S7 summary does not certify complete material with deferred final eligibility.')
    tables, schemas = {}, {}
    for name in tqdm(S8_UPSTREAM_TABLES, desc='Validate S7 table schemas', unit='table',
                     disable=not config.get('progress_bars', True)):
        descriptor = state['tables'][name]
        schema = json.loads(s8u_safe_path(cache_root, descriptor['schema_path']).read_bytes())
        if schema.get('artifact') != descriptor['relative_path'] or schema.get('fingerprint') != descriptor['fingerprint']:
            raise RuntimeError('S7 table/schema identity mismatch: ' + name)
        frame = pd.read_parquet(s8u_safe_path(cache_root, descriptor['relative_path']))
        if list(frame.columns) != [col['name'] for col in schema['columns']] or len(frame) != schema['rows']:
            raise RuntimeError('S7 table column order or row count mismatch: ' + name)
        if frame.columns.name != schema.get('column_axis_name'):
            raise RuntimeError('S7 table column-axis metadata mismatch: ' + name)
        for column in schema['columns']:
            dtype = frame[column['name']].dtype
            if str(dtype) != column['dtype']:
                raise RuntimeError(f'S7 dtype mismatch: {name}/{column["name"]}: {dtype} versus {column["dtype"]}')
            if 'string_storage' in column and getattr(dtype, 'storage', None) != column['string_storage']:
                raise RuntimeError(f'S7 string storage mismatch: {name}/{column["name"]}')
        if summary['table_row_counts'].get(name) != len(frame):
            raise RuntimeError('S7 summary/table row count mismatch: ' + name)
        s8u_validate_deferred(frame, name)
        tables[name], schemas[name] = frame, schema
    return {'state': state, 'state_sha256': expected_state, 'commit': commit, 'repository': repository,
            'tables': tables, 'schemas': schemas, 'config': snapshot, 'effective_config': snapshot['effective_config'],
            'summary': summary, 'lineage': lineage, 'cache_root': str(cache_root),
            'consumed_sha256': {relative: catalog[relative]['sha256'] for relative in sorted(required)},
            'transfers': transfers}


def s8u_scalar(value):
    if value is None or value is pd.NA or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def s8u_ids(value, field):
    value = s8u_scalar(value)
    if value is None:
        return []
    decoded = json.loads(value) if isinstance(value, str) else value
    if not isinstance(decoded, (list, tuple)) or any(not isinstance(item, str) or not item for item in decoded):
        raise ValueError(field + ' must contain a JSON list of nonempty strings.')
    if len(decoded) != len(set(decoded)):
        raise ValueError(field + ' has duplicated identities.')
    return sorted(decoded)


def s8u_int(value, field):
    value = s8u_scalar(value)
    if value is None or isinstance(value, bool) or not math.isfinite(float(value)) or float(value) != int(value):
        raise ValueError(field + ' must be a finite integer.')
    return int(value)


def s8_build_sources(inputs, config):
    """Normalize S7's physical-file inventory without assigning final splits.

    All original tables remain in inputs. Source records retain exact row-index
    links to those tables, multiple annotation IDs and historical assignments.
    A temporal window's target class is never transferred onto its neighbours.
    """
    tables = inputs['tables']
    qc = tables['image_qc_manifest']
    labelled = tables['labelled_frame_manifest_with_qc']
    temporal = tables['temporal_manifest_with_qc']
    for name, table in tables.items():
        if name in S8_UPSTREAM_TABLES:
            s8u_validate_deferred(table, name)
    if not qc['image_relpath'].is_unique or qc['image_relpath'].isna().any():
        raise ValueError('S7 QC inventory needs unique non-null physical file paths.')
    if labelled['annotation_id'].isna().any() or not labelled['annotation_id'].is_unique:
        raise ValueError('Labelled annotation identities must remain unique and non-null.')
    old_root = Path(inputs['effective_config']['storage_root'])
    new_root = Path(config.get('storage_root', str(old_root)))
    if not old_root.is_absolute() or not new_root.is_absolute() or '..' in new_root.parts:
        raise ValueError('Storage roots must be canonical absolute paths.')
    sources = {}
    qc_fields = ('image_readable', 'image_read_error', 'image_technically_usable', 'qc_requires_review',
                 'brightness_mean', 'contrast_std', 'blur_laplacian', 'underexposed_fraction',
                 'overexposed_fraction', 'specular_fraction', 'qc_blur_outlier', 'qc_contrast_outlier',
                 'qc_low_brightness_outlier', 'qc_high_brightness_outlier', 'qc_thresholds_calibrated',
                 'qc_assessment_status', 'qc_source_matches_upstream', 'image_decode_backend', 'image_qc_backend')
    for row in tqdm(qc.to_dict('records'), desc='Validate S7 source hash identities', unit='file',
                    disable=not config.get('progress_bars', True)):
        relative = s8u_relative(row['image_relpath'])
        if str(old_root / relative) != row['image_path']:
            raise ValueError('S7 source absolute/relative path disagreement: ' + relative)
        video, frame = s8u_scalar(row['video_key']), s8u_int(row['opencv_frame_index'], 'opencv_frame_index')
        if not isinstance(video, str) or not video or frame < 0:
            raise ValueError('Invalid physical-frame source identity: ' + relative)
        physical_id = json.dumps([video, frame], separators=(',', ':'), ensure_ascii=False)
        if row['physical_frame_id'] != physical_id:
            raise ValueError('S7 physical_frame_id disagrees with video/frame identity: ' + relative)
        upstream = s8u_ids(row['upstream_splits_json'], 'upstream_splits_json')
        if (not upstream or set(upstream) - {'train', 'validation', 'test'}
                or len(upstream) != int(row['upstream_split_count'])):
            raise ValueError('Invalid historical split provenance: ' + relative)
        readable = s8u_scalar(row['image_readable']) is True
        technically_usable = s8u_scalar(row['image_technically_usable']) is True
        matched = s8u_scalar(row['qc_source_matches_upstream']) is True
        sha = s8u_scalar(row['image_sha256'])
        verified_sha = s8u_scalar(row['qc_verified_source_sha256'])
        if sha is not None:
            s8u_hash(sha, relative)
        if verified_sha is not None:
            s8u_hash(verified_sha, relative + '/upstream')
        if sha is not None and verified_sha is not None and sha != verified_sha:
            raise ValueError('S7 decoded/source-audit hashes disagree: ' + relative)
        width, height = s8u_scalar(row['image_width']), s8u_scalar(row['image_height'])
        if readable:
            width, height = s8u_int(width, 'image_width'), s8u_int(height, 'image_height')
            if min(width, height) <= 0 or sha is None or verified_sha is None:
                raise ValueError('Readable S7 source lacks dimensions or verified bytes: ' + relative)
        eligible = readable and technically_usable and matched
        reason = None if eligible else ('source_not_readable' if not readable else
                   'source_not_technically_usable' if not technically_usable else 'upstream_hash_not_verified')
        if Path(relative).suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            eligible, reason = False, 'unsupported_source_image_format'
        if not bool(config.get('include_qc_review', True)) and s8u_scalar(row['qc_requires_review']) is True:
            eligible, reason = False, 'qc_review_excluded_by_explicit_policy'
        sources[relative] = {
            'image_relpath': relative, 'image_path': str(new_root / relative), 'upstream_image_path': row['image_path'],
            'sha256': sha, 'source_sha256': sha, 'width': width, 'height': height,
            'source_width': width, 'source_height': height, 'image_size_bytes': None,
            'video_key': video, 'frame_index': frame, 'physical_frame_id': physical_id,
            'annotation_ids': set(), 'finding_segment_ids': set(), 'referenced_by_segment_ids': set(),
            'verified_labels': set(), 'upstream_splits': upstream, 'qc': {k: s8u_scalar(row[k]) for k in qc_fields},
            'labelled_row_indices': [], 'temporal_row_indices': [],
            'final_split': None, 'split_assignment_status': 'pending', 'training_ready': False,
            'image_usable_for_modelling': False, 'prepare_eligible': eligible, 'skip_reason': reason,
        }

    def validate_reference(row, relative, kind):
        if relative not in sources:
            raise ValueError(f'{kind} reference is not present in the pinned S7 QC inventory: {relative}')
        source = sources[relative]
        if row['video_key'] != source['video_key'] or s8u_int(row['opencv_frame_index'], 'reference frame') != source['frame_index']:
            raise ValueError(f'{kind} file/physical-frame identity disagreement: {relative}')
        if s8u_scalar(row['qc_reference_covered']) is not True:
            raise ValueError(f'{kind} source lacks QC coverage: {relative}')
        for field in ('qc_image_sha256', 'qc_verified_source_sha256'):
            actual = s8u_scalar(row[field])
            if actual != source['sha256']:
                raise ValueError(f'{kind} enriched/QC source hash disagreement: {relative}/{field}')
        if (s8u_scalar(row['qc_image_width']), s8u_scalar(row['qc_image_height'])) != (source['width'], source['height']):
            raise ValueError(f'{kind} enriched/QC source dimensions disagree: {relative}')
        for field, qc_field in [('qc_image_readable', 'image_readable'),
                              ('qc_source_matches_upstream', 'qc_source_matches_upstream'),
                              ('image_technically_usable', 'image_technically_usable')]:
            if s8u_scalar(row[field]) != source['qc'][qc_field]:
                raise ValueError(f'{kind} enriched/QC flag disagreement: {relative}/{field}')
        if row['upstream_split'] not in source['upstream_splits']:
            raise ValueError(f'{kind} reference historical split is absent from inventory provenance.')
        return source

    annotation_sources, annotation_labels = {}, {}
    annotation_columns = ['image_relpath', 'video_key', 'opencv_frame_index', 'annotation_id',
        'finding_segment_id', 'finding_class_normalized', 'upstream_split', 'split',
        'qc_image_sha256', 'qc_verified_source_sha256', 'qc_image_width', 'qc_image_height',
        'qc_image_readable', 'qc_source_matches_upstream', 'image_technically_usable', 'qc_reference_covered',
        'source_image_sha256', 'source_image_size_bytes']
    for index, row in enumerate(tqdm(labelled[annotation_columns].to_dict('records'),
            desc='Validate labelled source SHA references', unit='row',
            disable=not config.get('progress_bars', True))):
        relative = s8u_relative(row['image_relpath'])
        source = validate_reference(row, relative, 'Labelled')
        if row['split'] != row['upstream_split']:
            raise ValueError('Retained labelled split differs from upstream_split provenance.')
        if row['source_image_sha256'] != source['sha256']:
            raise ValueError('Labelled original bytes differ from the pinned S7 source: ' + relative)
        annotation = str(row['annotation_id'])
        label = s8u_scalar(row['finding_class_normalized'])
        if not isinstance(label, str) or not label:
            raise ValueError('Verified annotation lacks its normalized source label.')
        annotation_sources[annotation] = relative
        annotation_labels[annotation] = label
        source['annotation_ids'].add(annotation)
        source['verified_labels'].add(label)
        source['finding_segment_ids'].add(str(row['finding_segment_id']))
        source['labelled_row_indices'].append(index)
        size = s8u_int(row['source_image_size_bytes'], 'source_image_size_bytes')
        if size <= 0 or source['image_size_bytes'] not in (None, size):
            raise ValueError('Contradictory original byte size for a labelled image.')
        source['image_size_bytes'] = size
    temporal_columns = ['context_image_relpath', 'video_key', 'opencv_frame_index', 'finding_segment_id',
        'upstream_split', 'context_frame_id', 'context_image_annotation_ids_json',
        'context_image_verified_labels', 'context_annotation_status', 'qc_image_sha256',
        'qc_verified_source_sha256', 'qc_image_width', 'qc_image_height', 'qc_image_readable',
        'qc_source_matches_upstream', 'image_technically_usable', 'qc_reference_covered', 'image_sha256']
    for index, row in enumerate(tqdm(temporal[temporal_columns].to_dict('records'),
            desc='Validate temporal source SHA references', unit='row',
            disable=not config.get('progress_bars', True))):
        relative = s8u_scalar(row['context_image_relpath'])
        if relative is None:
            # Preserve the original row in inputs for the planner's skipped-reference report.
            continue
        relative = s8u_relative(relative)
        source = validate_reference(row, relative, 'Temporal')
        if row['context_frame_id'] != source['physical_frame_id'] or row['image_sha256'] != source['sha256']:
            raise ValueError('S6 temporal identity/bytes disagree with S7 verified source: ' + relative)
        selected_ids = s8u_ids(row['context_image_annotation_ids_json'], 'context_image_annotation_ids_json')
        for annotation in selected_ids:
            if annotation_sources.get(annotation) != relative:
                raise ValueError('Temporal selected-image annotation belongs to a different file: ' + annotation)
            source['annotation_ids'].add(annotation)
            source['verified_labels'].add(annotation_labels[annotation])
        if not selected_ids and s8u_scalar(row['context_image_verified_labels']) is not None:
            raise ValueError('Temporal image labels lack source annotation evidence: ' + relative)
        source['referenced_by_segment_ids'].add(str(row['finding_segment_id']))
        source['temporal_row_indices'].append(index)
    for source in sources.values():
        for field in ('annotation_ids', 'verified_labels', 'finding_segment_ids', 'referenced_by_segment_ids'):
            source[field] = sorted(source[field])
        source['annotation_status'] = 'verified_source_annotations' if source['annotation_ids'] else 'unlabelled_context'
        source['unlabelled_is_normal'] = False
    return sources


## 4. Product planning and provenance preservation

### What `skipped` means

This table records a derived product or reference that cannot be constructed from the available evidence. It is not a list of deleted images, and its row count is not an excluded-image count. Several unavailable products can refer to the same source.

For `progression_panel`, construction requires at least **two distinct verified frames in the same segment**. A single frame remains usable as an original and for any other view whose requirements are met. Unlabelled neighboring context frames do not replace verified segment anchors.

For example, in video `04a78ef00c5245e0`, angiectasia segment `00001` has frame **887** and cannot produce a progression panel; segment `00002` has frame **888** and likewise cannot; segment `00003` has frames **889 and 890** and can. This is panel construction, not a model comparison or a split exclusion.

These are segment-level records, so `image_relpath` and `subject_id` may be null. The detailed report now includes `selection_scope`, `available_verified_frames`, `required_verified_frames`, frame indices, and source paths. Reasons are displayed without truncation, with a grouped explanation table.

For the pinned complete S7 snapshot, all **66,127 source files** and **47,239 annotation subjects** are retained. The **80,475** unavailable-product records comprise **43,112** annotations without a usable bbox, **23,075** unavailable combined panels, **13,378** singleton segments without a progression panel, and **910** unavailable same-video normal comparison panels. These product requirements remain unchanged.


In [5]:
# Pure deterministic visual planning. Imports are centralized by the notebook assembler.
S8_VARIANT_NAMES = ('full', 'crop', 'fullcrop', 'overlay', 'temporal_multi',
                    'temporal_collage', 'progression', 'qc', 'comparison', 'combined', 'fullcrop_panel')

S8_SKIP_EXPLANATIONS = {
    'segment_has_fewer_than_two_available_verified_anchors':
        'Only the progression panel is unavailable: the segment has fewer than two distinct verified frames. Its original frames remain available.',
    'bbox_not_spatially_usable':
        'The annotation has no spatially usable bounding box. The original remains available; bbox-dependent views cannot be generated for this annotation.',
    'bbox_coordinates_missing':
        'Bounding-box coordinates are incomplete. The original remains available; a crop cannot be generated safely.',
    'bbox_outside_verified_source_dimensions':
        'The bounding box is invalid or outside the verified image dimensions. The original remains available.',
    'no_distinct_existing_neighbor_within_quality_window':
        'No distinct existing same-video neighbor is available within the configured quality window. Only the quality panel is unavailable.',
    'no_verified_normal_reference_in_same_video':
        'No distinct explicitly verified normal reference is available in the same video. Only the comparison panel is unavailable.',
    'no_additional_available_visual_evidence':
        'No usable bounding box or temporal neighbor is available to add another view. The original remains available; the combined panel is unavailable.',
    'missing_finding_segment_id':
        'A temporal reference has no segment identifier and cannot be assigned to a temporal panel.',
    'target_source_unavailable':
        'The temporal target is not available among technically eligible verified sources.',
    'context_source_or_actual_time_unavailable':
        'This temporal reference lacks an eligible source, frame index, or actual time offset; the remaining valid references are still considered.',
    'target_or_neighbor_context_unavailable':
        'A temporal panel requires the target and at least one available neighboring frame.',
    'labelled_source_unavailable':
        'This annotation references a source that is not eligible for visual preparation; consult the source-level reason.',
    'source_not_readable': 'S7 marked the source as unreadable.',
    'source_not_technically_usable': 'S7 marked the source as technically unusable.',
    'source_not_technically_eligible': 'The source does not meet the technical preparation requirements.',
    'upstream_hash_not_verified': 'S7 did not verify that the source bytes match the upstream hash.',
    'unsupported_source_image_format': 'The source format is outside the supported JPEG/PNG formats.',
    'qc_review_excluded_by_explicit_policy': 'The explicit configuration excludes sources requiring QC review.',
}


def s8_skip_summary(plan):
    """Summarize unavailable products/references without counting them as dropped images.

    Several records may describe one source or annotation, so their sum is not
    the number of excluded images. Use the kind and scope to interpret a reason.
    """
    columns = ['kind', 'selection_scope', 'reason', 'records', 'explanation']
    groups = {}
    for record in plan.get('skipped', []):
        key = (record['kind'], record.get('selection_scope', 'unspecified'), record['reason'])
        groups[key] = groups.get(key, 0) + 1
    return pd.DataFrame([
        {'kind': kind, 'selection_scope': scope, 'reason': reason, 'records': count,
         'explanation': S8_SKIP_EXPLANATIONS.get(reason, 'Inspect the detailed record for this selection condition.')}
        for (kind, scope, reason), count in sorted(groups.items())
    ], columns=columns)


def _s8p_scalar(value):
    if value is None:
        return None
    if isinstance(value, (list, dict, tuple)):
        return value
    try:
        if bool(pd.isna(value)):
            return None
    except (TypeError, ValueError):
        pass
    return value.item() if isinstance(value, np.generic) else value


def _s8p_text(value):
    value = _s8p_scalar(value)
    return None if value is None or not str(value).strip() else str(value)


def _s8p_number(value, integer=False):
    value = _s8p_scalar(value)
    if value is None or isinstance(value, bool):
        return None
    try:
        number = float(value)
        if not math.isfinite(number) or (integer and not number.is_integer()):
            return None
        return int(number) if integer else number
    except (TypeError, ValueError, OverflowError):
        return None


def _s8p_true(value):
    value = _s8p_scalar(value)
    return isinstance(value, (bool, int)) and value is not None and value == 1


def _s8p_strings(value):
    value = _s8p_scalar(value)
    if value is None:
        return []
    if isinstance(value, str):
        if value.startswith('['):
            value = json.loads(value)
        else:
            value = [value]
    return sorted({_s8p_text(item) for item in value if _s8p_text(item) is not None})


def _s8p_normalize_label(value):
    return ' '.join(str(value).lower().replace('-', ' ').replace('_', ' ').split())


def s8_build_plan(inputs, sources, config):
    """Plan originals and all derived kinds from verified S7 evidence only.

    Old train/validation/test values are provenance, never selection criteria.
    No derived source is called normal merely because it is unlabelled. Segment
    progression uses actual labelled memberships, not adjacent unlabelled frames.
    Clinical labels remain in private subject metadata, never renderer params.
    """
    tables = inputs.get('tables', inputs)
    labelled = tables.get('labelled_frame_manifest_with_qc', pd.DataFrame()).to_dict('records')
    temporal = tables.get('temporal_manifest_with_qc', pd.DataFrame()).to_dict('records')
    margin = _s8p_number(config.get('roi_margin_fraction', .2))
    panel_size = _s8p_number(config.get('panel_size', 336), True)
    temporal_max = _s8p_number(config.get('temporal_max_frames', 5), True)
    progression_max = _s8p_number(config.get('progression_max_frames', 5), True)
    jpeg_quality = _s8p_number(config.get('jpeg_quality', 95), True)
    if margin is None or margin < 0 or panel_size is None or not 64 <= panel_size <= 4096:
        raise ValueError('Invalid ROI margin or panel_size (64..4096).')
    if temporal_max is None or progression_max is None or min(temporal_max, progression_max) < 2:
        raise ValueError('Temporal and progression panel limits must be at least two.')
    if jpeg_quality is None or not 1 <= jpeg_quality <= 100:
        raise ValueError('jpeg_quality must be between 1 and 100.')
    flags = {name: bool(config.get(name, True)) for name in (
        'generate_roi', 'generate_overlays', 'generate_temporal', 'generate_evidence_panels',
        'generate_progression', 'generate_quality_panels', 'generate_comparison_panels', 'generate_combined_panels')}
    max_subjects = config.get('max_subjects')
    if max_subjects is not None and (_s8p_number(max_subjects, True) is None or int(max_subjects) < 1):
        raise ValueError('max_subjects must be None or a positive integer.')
    artifacts, artifact_map, subjects, skipped = [], {}, [], []
    prepared_sources = {}

    def omit(kind, reason, subject_id=None, image_relpath=None, segment_id=None, **details):
        skipped.append({'kind': kind, 'status': 'skipped', 'reason': reason,
                        'subject_id': subject_id, 'image_relpath': image_relpath,
                        'finding_segment_id': segment_id,
                        'selection_scope': ('annotation' if subject_id is not None else
                            'segment' if segment_id is not None else
                            'source' if image_relpath is not None else 'reference'), **details})

    for path, raw in sorted(sources.items()):
        path = str(s8_safe_relative(path))
        source = dict(raw)
        if not _s8p_true(source.get('prepare_eligible')):
            omit('source', _s8p_text(source.get('skip_reason')) or 'source_not_technically_eligible', image_relpath=path)
            continue
        sha = source.get('sha256')
        width, height = _s8p_number(source.get('width'), True), _s8p_number(source.get('height'), True)
        frame = _s8p_number(source.get('frame_index'), True)
        video = _s8p_text(source.get('video_key'))
        if not isinstance(sha, str) or len(sha) != 64 or any(c not in '0123456789abcdef' for c in sha):
            raise ValueError('Missing verified S7 source SHA-256: ' + path)
        if width is None or height is None or min(width, height) < 1 or frame is None or video is None:
            raise ValueError('Missing verified source dimensions/frame identity: ' + path)
        source.update({'image_relpath': path, 'width': width, 'height': height, 'frame_index': frame,
                       'video_key': video, 'sha256': sha, 'training_ready': False, 'final_split': None,
                       'split_assignment_status': 'pending'})
        source['physical_frame_id'] = _s8p_text(source.get('physical_frame_id')) or s8_json_hash({'video_key': video, 'frame_index': frame})
        for name in ('annotation_ids', 'finding_segment_ids', 'verified_labels', 'upstream_splits'):
            source[name] = _s8p_strings(source.get(name))
        prepared_sources[path] = source

    def make_ref(path, role='target', index=0, is_target=True, offset=None, annotation_ids=None, segment_id=None):
        source = prepared_sources[path]
        return {'image_relpath': path, 'source_sha256': source['sha256'],
                'source_width': source['width'], 'source_height': source['height'],
                'video_key': source['video_key'], 'frame_index': source['frame_index'],
                'physical_frame_id': source['physical_frame_id'],
                'annotation_ids': source['annotation_ids'] if annotation_ids is None else sorted(set(annotation_ids)),
                'finding_segment_id': segment_id,
                'upstream_splits': source['upstream_splits'], 'role': role,
                'panel_index': int(index), 'is_target': bool(is_target),
                'time_offset_seconds': offset, 'final_split': None, 'training_ready': False}

    def add(kind, refs, params, materialized=True, assisted=False):
        # Logical IDs survive source-byte updates; signatures invalidate old pixels.
        logical = [{key: ref.get(key) for key in ('image_relpath', 'video_key', 'frame_index',
                   'annotation_ids', 'finding_segment_id', 'role', 'panel_index', 'is_target', 'time_offset_seconds')} for ref in refs]
        identity = {'kind': kind, 'sources': logical, 'params': params,
                    'version': S8_RECIPE_VERSION, 'jpeg_quality': jpeg_quality if materialized else None}
        aid = s8_json_hash(identity)
        signature = s8_json_hash({'recipe': identity, 'source_hashes': [r['source_sha256'] for r in refs],
                                 'source_sizes': [[r['source_width'], r['source_height']] for r in refs]})
        spec = {'artifact_id': aid, 'kind': kind, 'materialized': bool(materialized),
                'relative_path': f'images/{kind}/{aid}.jpg' if materialized else refs[0]['image_relpath'],
                'recipe_signature': signature, 'source_refs': refs, 'params': params,
                'annotation_assisted': bool(assisted), 'final_split': None, 'training_ready': False,
                'split_assignment_status': 'pending',
                'upstream_splits': sorted({v for r in refs for v in r['upstream_splits']})}
        existing = artifact_map.get(aid)
        if existing is not None:
            if existing != spec:
                raise ValueError('Conflicting visual artifact identity: ' + aid)
        else:
            artifact_map[aid] = spec
            artifacts.append(spec)
        return aid

    # Originals are registered, never rewritten or copied back to Drive.
    full_ids = {path: add('full_frame', [make_ref(path)], {}, False) for path in prepared_sources}
    annotations, subjects_by_annotation, subjects_by_path, segment_members = {}, {}, {}, {}
    all_segment_members, all_labels_by_path = {}, {}
    ordered_labelled = sorted(labelled, key=lambda r: (_s8p_text(r.get('annotation_id')) or '', _s8p_text(r.get('image_relpath')) or ''))
    eligible_annotation_ids = [_s8p_text(r.get('annotation_id')) for r in ordered_labelled if _s8p_text(r.get('image_relpath')) in prepared_sources]
    selected_annotation_ids = set(eligible_annotation_ids if max_subjects is None else eligible_annotation_ids[:int(max_subjects)])
    valid_boxes_by_path = {}
    for row in ordered_labelled:
        annotation_id, path = _s8p_text(row.get('annotation_id')), _s8p_text(row.get('image_relpath'))
        if annotation_id is None or path is None:
            raise ValueError('S7 labelled row has no annotation/source identity.')
        if annotation_id in annotations:
            raise ValueError('Duplicate S7 annotation_id: ' + annotation_id)
        annotations[annotation_id] = row
        subject_id = s8_json_hash({'annotation_id': annotation_id, 'image_relpath': path})
        if path not in prepared_sources:
            omit('subject', 'labelled_source_unavailable', subject_id, path)
            continue
        source = prepared_sources[path]
        if (_s8p_text(row.get('video_key')), _s8p_number(row.get('opencv_frame_index'), True)) != (source['video_key'], source['frame_index']):
            raise ValueError('Labelled/source frame identity disagreement: ' + annotation_id)
        segment_id = _s8p_text(row.get('finding_segment_id'))
        label = _s8p_text(row.get('finding_class_normalized')) or _s8p_text(row.get('finding_class'))
        group = _s8p_text(row.get('clinical_group'))
        if label:
            all_labels_by_path.setdefault(path, set()).add(_s8p_normalize_label(label))
        if segment_id:
            all_segment_members.setdefault(segment_id, []).append({'annotation_id': annotation_id, 'source_image_relpath': path,
                'video_key': source['video_key'], 'frame_index': source['frame_index']})
        box = None
        if _s8p_true(row.get('bbox_spatially_usable')):
            values = [_s8p_number(row.get(k)) for k in ('bbox_xmin', 'bbox_ymin', 'bbox_xmax', 'bbox_ymax')]
            if all(v is not None for v in values):
                x1, y1, x2, y2 = values
                if 0 <= x1 < x2 <= source['width'] and 0 <= y1 < y2 <= source['height']:
                    box = values
                else:
                    if annotation_id in selected_annotation_ids:
                        omit('roi_crop', 'bbox_outside_verified_source_dimensions', subject_id, path, segment_id)
            else:
                if annotation_id in selected_annotation_ids:
                    omit('roi_crop', 'bbox_coordinates_missing', subject_id, path, segment_id)
        else:
            if annotation_id in selected_annotation_ids:
                omit('roi_crop', 'bbox_not_spatially_usable', subject_id, path, segment_id)
        if box is not None:
            valid_boxes_by_path.setdefault(path, []).append({'bbox_xyxy': box, 'annotation_ids': [annotation_id]})
        if annotation_id not in selected_annotation_ids:
            continue
        subject = {'subject_id': subject_id, 'annotation_id': annotation_id, 'source_image_relpath': path,
                   'finding_segment_id': segment_id, 'video_key': source['video_key'], 'frame_index': source['frame_index'],
                   'upstream_splits': source['upstream_splits'], 'final_split': None, 'training_ready': False,
                   'variants': {'full': [full_ids[path]]},
                   'private_label_metadata': {'label': label, 'clinical_group': group, 'bbox_xyxy': box,
                       'all_verified_source_labels': source['verified_labels'], 'annotation_id': annotation_id,
                       'source_qc': source.get('qc', {}), 'ground_truth_scope': 'Original dataset annotation; no generated diagnosis.'}}
        subjects.append(subject)
        subjects_by_annotation[annotation_id] = subject
        subjects_by_path.setdefault(path, []).append(subject)
        if segment_id:
            segment_members.setdefault(segment_id, []).append(subject)
        if box is not None:
            ref = make_ref(path, annotation_ids=[annotation_id], segment_id=segment_id)
            params = {'bbox_xyxy': box, 'margin_fraction': margin}
            if flags['generate_roi']:
                crop = add('roi_crop', [ref], params, assisted=True)
                subject['variants']['crop'] = [crop]
                subject['variants']['fullcrop'] = [full_ids[path], crop]
            if flags['generate_evidence_panels']:
                # A paired view exists as a separate artifact; fullcrop benchmark
                # uses two original-resolution inputs, not this resized collage.
                subject['evidence_panel_id'] = add('evidence_panel', [ref], dict(params, panel_size=panel_size), assisted=True)
                subject['variants']['fullcrop_panel'] = [subject['evidence_panel_id']]

    for path, members in sorted(subjects_by_path.items()):
        boxes = sorted(valid_boxes_by_path.get(path, []), key=lambda item: (item['bbox_xyxy'], item['annotation_ids']))
        if flags['generate_overlays'] and boxes:
            overlay = add('bbox_overlay', [make_ref(path)], {'boxes': boxes}, assisted=True)
            for subject in members:
                if subject['private_label_metadata']['bbox_xyxy'] is not None:
                    subject['variants']['overlay'] = [overlay]

    # Quality comparison uses only existing same-video neighbours and raw S7
    # metrics. Selection is not a clinical cleanliness/evidence-sufficiency label.
    if flags['generate_quality_panels']:
        quality_max = _s8p_number(config.get('quality_max_frames', 3), True)
        quality_gap = _s8p_number(config.get('quality_max_frame_gap', 30), True)
        if quality_max is None or quality_max < 2 or quality_gap is None or quality_gap < 1:
            raise ValueError('Quality panels require >=2 frames and a positive same-video frame gap.')
        by_video = {}
        for path, source in prepared_sources.items():
            by_video.setdefault(source['video_key'], []).append(path)
        for video in by_video:
            by_video[video].sort(key=lambda p: (prepared_sources[p]['frame_index'], p))
        video_frame_arrays = {video: np.asarray([prepared_sources[p]['frame_index'] for p in paths], dtype=np.int64) for video, paths in by_video.items()}
        for path, members in sorted(subjects_by_path.items()):
            target = prepared_sources[path]
            numbers = video_frame_arrays[target['video_key']]
            lo = int(np.searchsorted(numbers, target['frame_index'] - quality_gap, side='left'))
            hi = int(np.searchsorted(numbers, target['frame_index'] + quality_gap, side='right'))
            candidates = [p for p in by_video[target['video_key']][lo:hi]
                if p != path and prepared_sources[p]['physical_frame_id'] != target['physical_frame_id']
                and prepared_sources[p]['sha256'] != target['sha256']
                and abs(prepared_sources[p]['frame_index'] - target['frame_index']) <= quality_gap]
            def variation(p):
                other, total, available = prepared_sources[p], 0.0, 0
                for metric in ('blur_laplacian', 'contrast_std', 'brightness_mean'):
                    a = _s8p_number(target.get('qc', {}).get(metric))
                    b = _s8p_number(other.get('qc', {}).get(metric))
                    if a is not None and b is not None:
                        total += abs(a - b) / max(abs(a), abs(b), 1.0)
                        available += 1
                return total if available else -1.0
            ordered_candidates = sorted(candidates, key=lambda p: (-variation(p),
                abs(prepared_sources[p]['frame_index'] - target['frame_index']), prepared_sources[p]['frame_index'], p))
            chosen, seen_frames, seen_sha = [], {target['physical_frame_id']}, {target['sha256']}
            for other in ordered_candidates:
                source = prepared_sources[other]
                if source['physical_frame_id'] in seen_frames or source['sha256'] in seen_sha:
                    continue
                chosen.append(other)
                seen_frames.add(source['physical_frame_id'])
                seen_sha.add(source['sha256'])
                if len(chosen) >= quality_max - 1:
                    break
            if not chosen:
                for subject in members:
                    omit('quality_panel', 'no_distinct_existing_neighbor_within_quality_window', subject['subject_id'], path)
                continue
            selected_paths = [path] + chosen
            refs = [make_ref(p, 'target' if p == path else 'quality_context', i, p == path) for i, p in enumerate(selected_paths)]
            quality = add('quality_panel', refs, {'panel_size': panel_size,
                'qc_by_image': {p: prepared_sources[p].get('qc', {}) for p in selected_paths},
                'selection_policy': 'Same video bounded frame gap; prioritize variation of available raw S7 blur/contrast/brightness.',
                'quality_max_frame_gap': quality_gap,
                'quality_semantics': 'Technical proxies only; no good/bad clinical visibility labels.'})
            for subject in members:
                subject['variants']['qc'] = [quality]

    # Build temporal views from existing S7 windows; all inherited split values
    # remain attached, even when a physical context frame has several origins.
    temporal_groups, temporal_for_subject = {}, {}
    for row in temporal:
        segment_id = _s8p_text(row.get('finding_segment_id'))
        if segment_id is None:
            omit('temporal_collage', 'missing_finding_segment_id')
            continue
        temporal_groups.setdefault(segment_id, []).append(row)
    for segment_id, rows in sorted(temporal_groups.items()):
        if not any(_s8p_text(r.get('target_image_annotation_id')) in subjects_by_annotation for r in rows) and not any(
                s['finding_segment_id'] == segment_id for r in rows for s in subjects_by_path.get(_s8p_text(r.get('target_image_relpath')), [])):
            continue
        targets = {(_s8p_text(r.get('target_image_relpath')), _s8p_text(r.get('video_key')),
                    _s8p_number(r.get('target_opencv_frame_index'), True)) for r in rows}
        if len(targets) != 1:
            raise ValueError('Temporal group has conflicting target identity: ' + segment_id)
        target_path, video, target_frame = next(iter(targets))
        if target_path not in prepared_sources or video is None or target_frame is None:
            omit('temporal_collage', 'target_source_unavailable', image_relpath=target_path, segment_id=segment_id)
            continue
        candidates = {}
        for row in rows:
            path = _s8p_text(row.get('context_image_relpath'))
            frame = _s8p_number(row.get('opencv_frame_index'), True)
            offset = _s8p_number(row.get('effective_temporal_offset_seconds'))
            if path not in prepared_sources or frame is None or offset is None:
                omit('temporal_collage', 'context_source_or_actual_time_unavailable', image_relpath=path, segment_id=segment_id)
                continue
            source = prepared_sources[path]
            if source['video_key'] != video or source['frame_index'] != frame:
                raise ValueError('Temporal/source frame identity mismatch: ' + segment_id)
            target = frame == target_frame
            if _s8p_true(row.get('is_target')) != target or (target and path != target_path):
                raise ValueError('Temporal target flag/path mismatch: ' + segment_id)
            if (target and offset != 0) or (frame < target_frame and offset >= 0) or (frame > target_frame and offset <= 0):
                raise ValueError('Temporal effective offset disagrees with frame ordering: ' + segment_id)
            candidate = (path, frame, offset, target)
            old = candidates.get(frame)
            if old is not None and old != candidate:
                raise ValueError('Ambiguous physical temporal frame source: ' + segment_id)
            candidates[frame] = candidate
        if target_frame not in candidates or len(candidates) < 2:
            omit('temporal_collage', 'target_or_neighbor_context_unavailable', image_relpath=target_path, segment_id=segment_id)
            continue
        chosen = sorted(sorted(candidates.values(), key=lambda v: (abs(v[2]), abs(v[1] - target_frame), v[1]))[:temporal_max], key=lambda v: v[1])
        refs = [make_ref(p, 'target' if t else 'temporal_context', i, t, off, segment_id=segment_id)
                for i, (p, frame, off, t) in enumerate(chosen)]
        target_annotations = {_s8p_text(r.get('target_image_annotation_id')) for r in rows}
        matching = [subjects_by_annotation[a] for a in sorted(a for a in target_annotations if a in subjects_by_annotation)]
        if not matching:
            matching = [s for s in subjects_by_path.get(target_path, []) if s['finding_segment_id'] == segment_id]
        if flags['generate_temporal']:
            collage = add('temporal_collage', refs, {'panel_size': panel_size, 'temporal_max_frames': temporal_max,
                'selection_policy': 'Nearest available actual offsets; chronological panels.',
                'temporal_semantics': 'Container-FPS offsets; no verified clinical boundaries.'})
            for subject in matching:
                subject['variants']['temporal_collage'] = [collage]
                subject['variants']['temporal_multi'] = [full_ids[ref['image_relpath']] for ref in refs]
                subject.setdefault('variant_input_metadata', {})['temporal_multi'] = [
                    {'frame_index': ref['frame_index'], 'time_offset_seconds': ref['time_offset_seconds'], 'is_target': ref['is_target']} for ref in refs]
        for subject in matching:
            temporal_for_subject[subject['subject_id']] = refs

    # Representative endpoints/interior anchors span the existing segment's
    # observed evidence. This does not infer onset, offset, tracking or continuity.
    if flags['generate_progression']:
        for segment_id, selected_members in sorted(segment_members.items()):
            members = all_segment_members.get(segment_id, [])
            if len({s['video_key'] for s in members}) != 1:
                raise ValueError('Segment annotation memberships cross videos: ' + segment_id)
            frames = {}
            for subject in members:
                path, frame = subject['source_image_relpath'], subject['frame_index']
                frames.setdefault(frame, []).append(path)
            ordered = [(frame, sorted(set(paths))[0]) for frame, paths in sorted(frames.items())]
            if len(ordered) < 2:
                omit('progression_panel', 'segment_has_fewer_than_two_available_verified_anchors',
                     segment_id=segment_id, available_verified_frames=len(ordered), required_verified_frames=2,
                     available_verified_frame_indices=[frame for frame, path in ordered],
                     available_verified_source_paths=[path for frame, path in ordered])
                continue
            count = min(len(ordered), progression_max)
            indexes = sorted({int(round(i * (len(ordered) - 1) / (count - 1))) for i in range(count)})
            chosen = [ordered[i] for i in indexes]
            refs = [make_ref(path, 'verified_segment_anchor', i, False, None,
                            annotation_ids=sorted({s['annotation_id'] for s in members if s['frame_index'] == frame and s['source_image_relpath'] == path}),
                            segment_id=segment_id) for i, (frame, path) in enumerate(chosen)]
            progression = add('progression_panel', refs, {'panel_size': panel_size,
                'selection_policy': 'Evenly spaced observed verified anchors including segment endpoints.',
                'available_verified_anchor_count': len(ordered), 'selected_verified_anchor_count': len(chosen),
                'observed_frame_min': ordered[0][0], 'observed_frame_max': ordered[-1][0],
                'clinical_boundary_verified': False, 'continuous_visibility_verified': False,
                'all_verified_anchor_annotation_ids': sorted({s['annotation_id'] for s in members}),
                'all_verified_anchor_frame_indices': [frame for frame, path in ordered]})
            for subject in selected_members:
                subject['variants']['progression'] = [progression]

    # A normal reference must have an explicit normal annotation on that file,
    # and no contradictory non-normal annotation. Unlabelled never means normal.
    normal_labels = {_s8p_normalize_label(v) for v in config.get('normal_reference_labels', ['normal clean mucosa'])}
    abnormal_groups = set(config.get('comparison_target_clinical_groups', [
        'bleeding', 'foreign_body', 'lymphatic_lesion', 'mucosal_lesion', 'protruding_lesion', 'vascular_lesion']))
    normals_by_video = {}
    for path, verified in all_labels_by_path.items():
        # Require the normalized source label union to agree as well, so an
        # alias/context annotation cannot hide a non-normal source label.
        source_verified = {_s8p_normalize_label(label) for label in prepared_sources[path]['verified_labels']}
        if verified and verified <= normal_labels and source_verified and source_verified <= normal_labels:
            normals_by_video.setdefault(prepared_sources[path]['video_key'], []).append(path)
    mappings = config.get('normal_comparison_pairs', {})
    if not isinstance(mappings, dict):
        raise ValueError('normal_comparison_pairs must map subject/annotation IDs to verified normal image_relpath.')
    if flags['generate_comparison_panels']:
        for subject in subjects:
            path = subject['source_image_relpath']
            if subject['private_label_metadata']['clinical_group'] not in abnormal_groups:
                continue
            candidates = [p for p in normals_by_video.get(subject['video_key'], [])
                          if prepared_sources[p]['physical_frame_id'] != prepared_sources[path]['physical_frame_id']]
            requested = mappings.get(subject['subject_id'], mappings.get(subject['annotation_id']))
            if requested is not None:
                if requested not in candidates:
                    raise ValueError('Configured normal comparison must be a distinct verified-normal frame in the same video: ' + str(requested))
                pair = requested
                policy = 'Explicit configured verified-normal pair in same video.'
            elif candidates:
                target_splits = set(subject['upstream_splits'])
                pair = min(candidates, key=lambda p: (-len(target_splits.intersection(prepared_sources[p]['upstream_splits'])),
                    abs(prepared_sources[p]['frame_index'] - subject['frame_index']), prepared_sources[p]['frame_index'], p))
                policy = 'Same video; prefer shared inherited provenance, then nearest verified-normal frame.'
            else:
                omit('comparison_panel', 'no_verified_normal_reference_in_same_video', subject['subject_id'], path, subject['finding_segment_id'])
                continue
            refs = [make_ref(path, 'target', 0, True, annotation_ids=[subject['annotation_id']], segment_id=subject['finding_segment_id']),
                    make_ref(pair, 'comparison_reference', 1, False)]
            if int(s8_json_hash({'comparison_subject': subject['subject_id'], 'reference': pair})[:8], 16) % 2:
                refs.reverse()
            refs = [dict(ref, panel_index=i) for i, ref in enumerate(refs)]
            comparison = add('comparison_panel', refs, {'panel_size': panel_size, 'pair_selection_policy': policy,
                'reference_annotation_verified': True, 'same_video': True, 'position_policy': 'Deterministic opaque permutation; no fixed normal position.',
                'clinical_comparability_validated': False, 'final_split_assignment_required': True})
            subject['variants']['comparison'] = [comparison]
            subject['comparison_reference_image_relpath'] = pair

    if flags['generate_combined_panels']:
        for subject in subjects:
            path = subject['source_image_relpath']
            box = subject['private_label_metadata']['bbox_xyxy']
            boxes = valid_boxes_by_path.get(path, [])
            temporal_refs = temporal_for_subject.get(subject['subject_id'], [])
            neighbors = [ref for ref in temporal_refs if ref['image_relpath'] != path]
            # Keep an explicit combined view even when one component is absent;
            # mark availability so no downstream task assumes invented evidence.
            refs = [make_ref(path, 'target', 0, True, annotation_ids=[subject['annotation_id']], segment_id=subject['finding_segment_id'])]
            for index, ref in enumerate(neighbors, 1):
                refs.append(dict(ref, panel_index=index, role='temporal_context'))
            if box is None and not boxes and not neighbors:
                omit('combined_panel', 'no_additional_available_visual_evidence', subject['subject_id'], path, subject['finding_segment_id'])
                continue
            params = {'panel_size': panel_size, 'margin_fraction': margin, 'boxes': boxes,
                'has_roi': box is not None, 'has_overlay': bool(boxes), 'has_temporal_context': bool(neighbors)}
            if box is not None:
                params['bbox_xyxy'] = box
            combined = add('combined_panel', refs, params, assisted=bool(boxes or box is not None))
            subject['variants']['combined'] = [combined]

    for subject in subjects:
        subject['variants'] = {name: subject['variants'][name] for name in S8_VARIANT_NAMES if name in subject['variants']}
        subject['available_variants'] = list(subject['variants'])
        subject['unavailable_variants'] = [name for name in S8_VARIANT_NAMES if name not in subject['variants']]
    artifacts.sort(key=lambda a: (a['kind'], a['artifact_id']))
    subjects.sort(key=lambda s: s['subject_id'])
    skipped.sort(key=lambda r: json.dumps(r, sort_keys=True, default=str))
    return {'artifacts': artifacts, 'subjects': subjects, 'sources': prepared_sources, 'skipped': skipped,
            'training_ready': False, 'final_split': None, 'split_assignment_status': 'pending',
            'summary': {'source_images': len(prepared_sources), 'subjects': len(subjects),
                        'materialized_artifacts': sum(a['materialized'] for a in artifacts),
                        'reference_artifacts': sum(not a['materialized'] for a in artifacts),
                        'skipped_records': len(skipped), 'max_subjects': max_subjects,
                        'eligible_annotations_before_scope_limit': len(eligible_annotation_ids)}}


## 5. GPU decoding, crops, overlays, and panels

In [6]:
"""Sector 8 decoding and deterministic visual rendering.

JPEGs use verified NVIDIA device output by default. A logical source batch is at
most 1000 unique files; adaptive subbatches and host spill bound GPU residency.
Source checksums are verified before decode. CPU composition is recorded; it is
not reported as GPU decoding or GPU resizing.
"""



def s8_warn_fallback_once(runtime, reason):
    runtime['fallback_reasons'].add(str(reason))
    if not runtime['warning_emitted']:
        runtime['warning_emitted'] = True
        warnings.warn('Sector 8: an explicitly permitted alternate decode route is active: ' + str(reason)
                      + '. Every backend/reason is recorded; this is the only decode-route warning for this runtime.',
                      UserWarning, stacklevel=2)


def s8_runtime_fingerprint(runtime):
    """Record pixel/encoding dependencies and selected policy, not changing free VRAM."""
    result = {
        'numpy_version': str(np.__version__), 'opencv_version': str(cv2.__version__),
        'cupy_version': None if cp is None else str(cp.__version__),
        'nvimagecodec_version': None if nvimgcodec is None else str(getattr(nvimgcodec, '__version__', 'unknown')),
        'primary_decoder': runtime['decoder_mode'], 'device_name': runtime.get('device_name'),
        'cuda_runtime_version': runtime.get('cuda_runtime_version'),
        'orientation': 'ignore_exif', 'colour': 'RGB_uint8',
        'cpu_decode_fallback_allowed': runtime['allow_cpu_fallback'],
        'require_gpu_decode': runtime['require_gpu_decode'],
        'non_jpeg_cpu_allowed': runtime['allow_non_jpeg_cpu'],
        'nvidia_hybrid_allowed': runtime['allow_nvidia_hybrid'],
        'require_gpu_only_backend': runtime.get('require_gpu_only_backend', False),
        'selected_backend_kinds': runtime.get('selected_backend_kinds', []),
        'gpu_backend_order': ['HW_GPU_ONLY', 'GPU_ONLY'],
        'crop_algorithm': 'xyxy_half_open_floor_ceil_margin_v1',
        'composition_algorithm': 'opencv_letterbox_area_down_linear_up_v1',
    }
    return result


def s8_cuda_pixels(owner, runtime):
    if owner is None:
        raise RuntimeError('NVIDIA decoder returned no image')
    if owner.buffer_kind != nvimgcodec.ImageBufferKind.STRIDED_DEVICE:
        raise RuntimeError('NVIDIA decoder returned host pixels')
    if int(owner.device_id) != runtime['device_id']:
        raise RuntimeError('NVIDIA decoder returned the wrong CUDA device')
    interface = getattr(owner, '__cuda_array_interface__', None)
    if not isinstance(interface, dict) or not interface.get('data', (0,))[0]:
        raise RuntimeError('NVIDIA image has no valid CUDA array interface')
    pixels = cp.asarray(owner)
    if (not isinstance(pixels, cp.ndarray) or int(pixels.device.id) != runtime['device_id']
            or pixels.ndim != 3 or pixels.shape[2] != 3 or pixels.dtype != cp.uint8):
        raise RuntimeError('NVIDIA image must be RGB uint8 on the selected device')
    return pixels


def s8_gpu_error_text(exc):
    return type(exc).__name__ + ': ' + str(exc).strip()


def s8_gpu_environment():
    packages = {}
    for package in ('cupy', 'cupy-cuda11x', 'cupy-cuda12x', 'cupy-cuda13x',
                    'nvidia-nvimgcodec-cu12', 'nvidia-nvimgcodec-cu13',
                    'nvidia-nvjpeg-cu12', 'nvidia-nvjpeg-cu13'):
        try:
            packages[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            pass
    return {'python': platform.python_version(), 'packages': packages,
            'import_errors': dict(globals().get('S8_GPU_IMPORT_ERRORS', {}))}


def s8_probe_decoder(runtime, names):
    available = [getattr(nvimgcodec.BackendKind, name) for name in names
                 if hasattr(nvimgcodec.BackendKind, name)]
    if not available:
        raise RuntimeError('Requested NVIDIA backend kinds are unavailable')
    with cp.cuda.Device(runtime['device_id']), runtime['stream'], warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        decoder = nvimgcodec.Decoder(device_id=runtime['device_id'],
                                    max_num_cpu_threads=runtime['workers'],
                                    backends=[nvimgcodec.Backend(kind) for kind in available])
        # A reproducible local RGB image tests real baseline-JPEG decoding, not just CUDA presence.
        probe_bgr = np.zeros((336, 336, 3), dtype=np.uint8)
        probe_bgr[:, :, 2] = 180
        ok, encoded = cv2.imencode('.jpg', probe_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
        if not ok:
            raise RuntimeError('Could not construct the JPEG capability probe')
        payload = encoded.tobytes()
        stream = nvimgcodec.CodeStream(payload)
        owners = None
        try:
            owners = decoder.decode([stream], params=runtime['params'], cuda_stream=runtime['stream'].ptr)
            if not isinstance(owners, (list, tuple)) or len(owners) != 1:
                raise RuntimeError('Unexpected NVIDIA capability probe result')
            pixels = s8_cuda_pixels(owners[0], runtime)
            if tuple(pixels.shape) != (336, 336, 3):
                raise RuntimeError('Unexpected NVIDIA capability probe dimensions')
            runtime['stream'].synchronize()
            sample = cp.asnumpy(pixels[0, 0])
            if int(sample[0]) < 150 or int(sample[2]) > 25:
                raise RuntimeError('NVIDIA capability probe RGB channel order failed')
        finally:
            runtime['stream'].synchronize()
            if captured:
                runtime.setdefault('probe_warnings', []).append({
                    'backends': list(names), 'warnings': [str(w.message) for w in captured]})
    return decoder


def s8_get_hybrid_decoder(runtime):
    if runtime.get('require_gpu_only_backend', False) or not runtime['allow_nvidia_hybrid']:
        return None
    if runtime['hybrid_attempted']:
        return runtime['hybrid_decoder']
    runtime['hybrid_attempted'] = True
    if not runtime['cuda_available'] or nvimgcodec is None or runtime.get('params') is None:
        return None
    try:
        runtime['hybrid_decoder'] = s8_probe_decoder(runtime, ('HYBRID_CPU_GPU',))
    except Exception as exc:
        reason = 'nvidia_hybrid_unavailable:' + s8_gpu_error_text(exc)
        runtime['fallback_reasons'].add(reason)
        runtime.setdefault('gpu_diagnostics', []).append({'stage': 'HYBRID_CPU_GPU probe', 'error': reason})
    return runtime['hybrid_decoder']


def s8_make_runtime(config):
    runtime = {'device_id': int(config.get('gpu_device_id', 0)), 'workers': int(config.get('io_workers', 16)),
               'require_gpu_decode': bool(config.get('require_gpu_decode', True)),
               'allow_cpu_fallback': bool(config.get('allow_cpu_fallback', False)),
               'allow_non_jpeg_cpu': bool(config.get('allow_non_jpeg_cpu', True)),
               'allow_nvidia_hybrid': bool(config.get('allow_nvidia_hybrid', False)),
               'require_gpu_only_backend': bool(config.get('require_gpu_only_backend', False)),
               'decoder': None, 'decoder_mode': 'unavailable', 'params': None, 'stream': None,
               'cuda_available': False, 'hybrid_attempted': False, 'hybrid_decoder': None,
               'warning_emitted': False, 'fallback_reasons': set(), 'batch_history': [],
               'gpu_environment': s8_gpu_environment(), 'gpu_diagnostics': [], 'probe_warnings': []}
    if runtime['workers'] < 1:
        raise ValueError('io_workers must be at least one')
    reason = 'cupy_unavailable: ' + str(runtime['gpu_environment']['import_errors'].get('cupy', 'not imported'))
    cupy_packages = [p for p in runtime['gpu_environment']['packages'] if p == 'cupy' or p.startswith('cupy-cuda')]
    if len(cupy_packages) > 1:
        runtime['gpu_unavailable_reason'] = ('conflicting_cupy_distributions: ' + ', '.join(cupy_packages)
            + '. Rerun the dependency repair cell, restart the Colab session when requested, and rerun imports.')
        runtime['gpu_diagnostics'].append({'stage': 'package inventory', 'error': runtime['gpu_unavailable_reason']})
        return runtime
    codec_packages = [p for p in runtime['gpu_environment']['packages'] if p.startswith('nvidia-nvimgcodec')]
    if len(codec_packages) > 1:
        runtime['gpu_unavailable_reason'] = ('conflicting_nvimagecodec_distributions: ' + ', '.join(codec_packages)
            + '. Both distributions share nvidia.nvimgcodec. Run dependency repair, then restart if requested.')
        runtime['gpu_diagnostics'].append({'stage': 'package inventory', 'error': runtime['gpu_unavailable_reason']})
        return runtime
    if cp is not None:
        try:
            if runtime['device_id'] < 0 or runtime['device_id'] >= cp.cuda.runtime.getDeviceCount():
                raise RuntimeError('Selected CUDA device unavailable')
            with cp.cuda.Device(runtime['device_id']):
                runtime['stream'] = cp.cuda.Stream(non_blocking=True)
                with runtime['stream']:
                    probe = cp.asarray(np.array([1], dtype=np.uint8)) + 1
                    if int(cp.asnumpy(probe)[0]) != 2:
                        raise RuntimeError('CUDA computation probe failed')
                runtime['stream'].synchronize()
                props = cp.cuda.runtime.getDeviceProperties(runtime['device_id'])
                name = props['name']
                runtime['device_name'] = name.decode(errors='replace') if isinstance(name, bytes) else str(name)
                runtime['cuda_runtime_version'] = int(cp.cuda.runtime.runtimeGetVersion())
                runtime['cuda_available'] = True
        except Exception as exc:
            reason = 'cuda_unavailable:' + s8_gpu_error_text(exc)
            runtime['gpu_diagnostics'].append({'stage': 'CUDA compute probe', 'error': reason})
    if runtime['cuda_available'] and nvimgcodec is not None:
        try:
            runtime['params'] = nvimgcodec.DecodeParams(color_spec=nvimgcodec.ColorSpec.SRGB,
                                                       sample_format=nvimgcodec.SampleFormat.I_RGB,
                                                       apply_exif_orientation=False)
            # Try dedicated JPEG hardware first, then GPU-only implementations.
            # A CUDA device alone does not guarantee dedicated JPEG hardware.
            for names in (('HW_GPU_ONLY', 'GPU_ONLY'), ('GPU_ONLY',)):
                try:
                    runtime['decoder'] = s8_probe_decoder(runtime, names)
                    runtime['selected_backend_kinds'] = list(names)
                    runtime['decoder_mode'] = 'nvidia_gpu'
                    return runtime
                except Exception as exc:
                    runtime['gpu_diagnostics'].append({'stage': '+'.join(names) + ' JPEG probe',
                                                       'error': s8_gpu_error_text(exc)})
            raise RuntimeError('; '.join(item['stage'] + ': ' + item['error']
                                         for item in runtime['gpu_diagnostics']))
        except Exception as exc:
            reason = 'nvidia_gpu_unavailable:' + s8_gpu_error_text(exc)
            runtime['fallback_reasons'].add(reason)
        runtime['decoder'] = s8_get_hybrid_decoder(runtime)
        if runtime['decoder'] is not None:
            runtime['decoder_mode'] = 'nvidia_hybrid'
            runtime['selected_backend_kinds'] = ['HYBRID_CPU_GPU']
            runtime['decode_policy_note'] = ('NVIDIA nvJPEG CUDA-accelerated decode: CPU header/entropy work may be used; '
                                             'decoded pixels and cropping are on the selected GPU. No CPU-only decoder.')
            s8_warn_fallback_once(runtime, 'nvidia_hybrid_cpu_gpu_decode')
            return runtime
    elif runtime['cuda_available']:
        reason = 'nvimagecodec_unavailable: ' + str(runtime['gpu_environment']['import_errors'].get('nvimgcodec', 'not imported'))
    # Eligibility is decided per actual encoded format in s8_decode_batch. PNG
    # sources may use their authorized CPU route even on a runtime without CUDA;
    # strict-policy JPEGs fail explicitly instead of silently decoding on CPU.
    if runtime.get('gpu_diagnostics'):
        reason += ' | ' + '; '.join(item['stage'] + ': ' + item['error'] for item in runtime['gpu_diagnostics'])
    runtime['gpu_unavailable_reason'] = reason
    if runtime['allow_cpu_fallback'] and not runtime['require_gpu_decode']:
        runtime['decoder_mode'] = 'opencv_cpu'
    return runtime


def s8_assert_gpu_runtime(runtime, config, require_jpeg=True):
    """Fail early with actual package/CUDA/JPEG probe reasons, not a generic GPU message."""
    if not require_jpeg or not bool(config.get('require_gpu_decode', True)):
        return runtime
    accepted = {'nvidia_gpu'}
    if config.get('allow_nvidia_hybrid', False) and not config.get('require_gpu_only_backend', False):
        accepted.add('nvidia_hybrid')
    if runtime.get('decoder_mode') in accepted and runtime.get('cuda_available'):
        return runtime
    reason = runtime.get('gpu_unavailable_reason', 'selected decoder mode: ' + str(runtime.get('decoder_mode')))
    environment = runtime.get('gpu_environment', {})
    diagnostics = runtime.get('gpu_diagnostics', [])
    raise RuntimeError('Required NVIDIA JPEG decode capability probe failed. ' + str(reason)
                       + ' | Environment: ' + repr(environment)
                       + ' | Backend probes: ' + repr(diagnostics)
                       + '. A GPU VM can still have conflicting packages, a missing nvJPEG library, '
                         'or an unsupported backend. Run the dependency repair cell and restart the '
                         'session if requested; then rerun imports and the early GPU preflight. '
                         'CPU-only JPEG fallback remains disabled.')


def s8_gpu_preflight(config):
    """A small baseline-JPEG decode probe before dataset staging or resume verification."""
    runtime = s8_make_runtime(config)
    print('GPU environment:', runtime.get('gpu_environment'))
    print('CUDA device:', runtime.get('device_name'), '| CUDA runtime:', runtime.get('cuda_runtime_version'))
    print('JPEG decoder:', runtime.get('decoder_mode'), '| backends:', runtime.get('selected_backend_kinds'))
    for diagnostic in runtime.get('gpu_diagnostics', []):
        print('Backend diagnostic:', diagnostic)
    for item in runtime.get('probe_warnings', []):
        print('Probe warnings:', item)
    if runtime.get('decode_policy_note'):
        print(runtime['decode_policy_note'])
    return s8_assert_gpu_runtime(runtime, config)


def s8_read_encoded_local(item, config):
    path = Path(str(item['local_path']))
    # Drive-backed paths must never enter the decoding loop.
    if path.is_symlink() or 'drive' in [p.lower() for p in path.resolve().parts]:
        raise ValueError('Decode requires an ordinary SSD staged file, not a Drive path or symlink')
    local_root = config.get('local_root')
    if local_root is not None and not path.resolve().is_relative_to(Path(local_root).resolve()):
        raise ValueError('Decode source is outside configured local_root')
    expected = str(item.get('sha256', '')).lower()
    if len(expected) != 64 or any(c not in '0123456789abcdef' for c in expected):
        raise ValueError('A valid verified source SHA256 is required before decoding')
    payload = path.read_bytes()
    actual = hashlib.sha256(payload).hexdigest()
    if actual != expected:
        raise ValueError('source_sha256_mismatch: expected=' + expected + ' actual=' + actual)
    return {'payload': payload, 'sha256': actual, 'is_jpeg': payload.startswith(b'\xff\xd8\xff')}


def s8_decode_cpu(snapshot, runtime, reason):
    result = {'pixels': None, 'owner': None, 'decode_backend': 'opencv_cpu',
              'fallback_reason': str(reason), 'error': None, 'source_sha256': snapshot['sha256'],
              'pixel_residency': 'host', 'gpu_device_id': None}
    authorized = (runtime['allow_non_jpeg_cpu'] if not snapshot['is_jpeg'] else
                  runtime['allow_cpu_fallback'] and not runtime['require_gpu_decode'])
    if not authorized:
        result['error'] = 'cpu_fallback_disabled:' + str(reason)
        result['decode_backend'] = 'not_decoded'
        result['pixel_residency'] = 'none'
        return result
    result['decode_policy'] = ('authorized_non_jpeg_cpu' if not snapshot['is_jpeg']
                               else 'explicit_jpeg_cpu_fallback')
    try:
        pixels = cv2.imdecode(np.frombuffer(snapshot['payload'], dtype=np.uint8),
                              cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
        if pixels is None or pixels.size == 0:
            raise ValueError('opencv_decode_failed')
        result['pixels'] = cv2.cvtColor(pixels, cv2.COLOR_BGR2RGB)
    except Exception as exc:
        result['error'] = type(exc).__name__ + ': ' + str(exc)
    return result


def s8_spill_device_pixels(output, runtime, required_bytes, fraction):
    """Bound retained device pixels across microbatches within one logical batch.

    This is a memory transfer, not a switch to CPU decoding. Spilled frames retain
    their NVIDIA decoder provenance and are uploaded one at a time for GPU crops.
    """
    with cp.cuda.Device(runtime['device_id']):
        free_bytes, _ = cp.cuda.runtime.memGetInfo()
        if required_bytes <= int(free_bytes * fraction):
            return int(free_bytes)
        candidates = sorted(
            ((key, int(record['pixels'].nbytes)) for key, record in output.items()
             if isinstance(record.get('pixels'), cp.ndarray)), key=lambda pair: (-pair[1], pair[0]))
        spilled_bytes, spilled_count = 0, 0
        for key, size in candidates:
            record = output[key]
            runtime['stream'].synchronize()
            host_pixels = cp.asnumpy(record['pixels'])
            record['pixels'] = host_pixels
            record['owner'] = None
            record['pixel_residency'] = 'host_after_gpu_decode'
            record['gpu_device_id'] = runtime['device_id']
            spilled_bytes += size
            spilled_count += 1
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()
            free_bytes, _ = cp.cuda.runtime.memGetInfo()
            if required_bytes <= int(free_bytes * fraction):
                break
        if spilled_count:
            runtime['batch_history'].append({'backend': 'device_to_host_transfer',
                'batch_size': spilled_count, 'status': 'spilled_for_vram', 'rgb_bytes': spilled_bytes,
                'crop_policy': 'upload_one_source_for_gpu_crop'})
        return int(free_bytes)


def s8_decode_nvidia_group(keys, snapshots, runtime, decoder, backend, config, output, failures):
    """Decode a true list batch; adaptively bisect failed calls before any CPU fallback."""
    if not keys:
        return
    codestreams, eligible = [], []
    for key in keys:
        try:
            cs = nvimgcodec.CodeStream(snapshots[key]['payload'])
            if str(cs.codec_name).lower() != 'jpeg':
                raise ValueError('Expected JPEG codestream')
            # Header dimensions estimate output+temporary memory before GPU allocation.
            estimated = max(1, int(cs.width)) * max(1, int(cs.height)) * 3
            snapshots[key]['decoded_byte_estimate'] = estimated
            codestreams.append(cs)
            eligible.append(key)
        except Exception as exc:
            failures[key] = 'nvidia_header_failed:' + s8_gpu_error_text(exc)
    if not eligible:
        return
    needed = sum(snapshots[key]['decoded_byte_estimate'] for key in eligible)
    fraction = float(config.get('gpu_memory_fraction', 0.65))
    if not 0 < fraction <= 0.90:
        raise ValueError('gpu_memory_fraction must be in (0, 0.90]')
    free_bytes = s8_spill_device_pixels(output, runtime, needed * 3, fraction)
    # Retained pixels are already reflected in free_bytes. Reserve 2x RGB output for SDK workspace.
    if len(eligible) > 1 and needed * 3 > int(free_bytes * fraction):
        middle = len(eligible) // 2
        codestreams.clear()
        runtime['batch_history'].append({'backend': backend, 'batch_size': len(eligible),
                                         'status': 'split_for_vram', 'estimated_rgb_bytes': needed})
        s8_decode_nvidia_group(eligible[:middle], snapshots, runtime, decoder, backend, config, output, failures)
        s8_decode_nvidia_group(eligible[middle:], snapshots, runtime, decoder, backend, config, output, failures)
        return
    owners, temporary = None, {}
    start = time.perf_counter()
    error = None
    try:
        with cp.cuda.Device(runtime['device_id']), runtime['stream'], warnings.catch_warnings():
            warnings.simplefilter('ignore')
            try:
                owners = decoder.decode(codestreams, params=runtime['params'], cuda_stream=runtime['stream'].ptr)
                if not isinstance(owners, (list, tuple)) or len(owners) != len(eligible):
                    raise RuntimeError('NVIDIA decode result count mismatch')
                for key, owner in zip(eligible, owners):
                    try:
                        pixels = s8_cuda_pixels(owner, runtime)
                        temporary[key] = {'pixels': pixels, 'owner': owner, 'decode_backend': backend,
                                          'fallback_reason': 'nvidia_hybrid_cpu_gpu_decode' if backend == 'nvidia_hybrid' else None,
                                          'error': None, 'source_sha256': snapshots[key]['sha256'],
                                          'pixel_residency': 'device', 'gpu_device_id': runtime['device_id']}
                    except Exception as exc:
                        failures[key] = backend + '_image_failed:' + s8_gpu_error_text(exc)
            finally:
                runtime['stream'].synchronize()
    except Exception as exc:
        error = backend + '_batch_failed:' + type(exc).__name__ + ':' + str(exc)[:200]
    runtime['batch_history'].append({'backend': backend, 'batch_size': len(eligible),
                                     'status': 'failed' if error else 'decoded', 'error': error,
                                     'decoded_count': 0 if error else len(temporary),
                                     'elapsed_seconds': round(time.perf_counter() - start, 6)})
    if error:
        temporary.clear()
        pixels = None
        owner = None
        owners = None
        codestreams.clear()
        gc.collect()
        with cp.cuda.Device(runtime['device_id']):
            cp.get_default_memory_pool().free_all_blocks()
        if len(eligible) > 1:
            middle = len(eligible) // 2
            s8_decode_nvidia_group(eligible[:middle], snapshots, runtime, decoder, backend, config, output, failures)
            s8_decode_nvidia_group(eligible[middle:], snapshots, runtime, decoder, backend, config, output, failures)
        else:
            failures[eligible[0]] = error
        return
    output.update(temporary)
    for key in temporary:
        failures.pop(key, None)
    if backend == 'nvidia_hybrid' and temporary:
        s8_warn_fallback_once(runtime, 'nvidia_hybrid_cpu_gpu_decode')
    # nvImageCodec also signals some allocation failures by returning None.
    # Retry an all-failed multi-image call as smaller batches before fallback.
    pending = [key for key in eligible if key not in output]
    if pending and len(eligible) > 1:
        owners = None
        owner = None
        pixels = None
        codestreams.clear()
        gc.collect()
        with cp.cuda.Device(runtime['device_id']):
            cp.get_default_memory_pool().free_all_blocks()
        middle = max(1, len(pending) // 2)
        s8_decode_nvidia_group(pending[:middle], snapshots, runtime, decoder, backend, config, output, failures)
        s8_decode_nvidia_group(pending[middle:], snapshots, runtime, decoder, backend, config, output, failures)


def s8_decode_batch(items, runtime, config):
    """One logical batch <=1000 frames; return pixels keyed by unique source relpath."""
    if not 1 <= int(config.get('batch_size', 1000)) <= 1000:
        raise ValueError('batch_size must be in [1,1000]')
    if len(items) > int(config.get('batch_size', 1000)) or len(items) > 1000:
        raise ValueError('A logical decode batch may contain at most 1000 source frames')
    keys = [str(item['image_relpath']) for item in items]
    if len(keys) != len(set(keys)):
        raise ValueError('Duplicate source image_relpath in logical decode batch')
    snapshots, output, failures = {}, {}, {}
    started = time.perf_counter()
    workers = max(1, min(int(config.get('io_workers', 16)), len(items) or 1))
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(s8_read_encoded_local, item, config): str(item['image_relpath']) for item in items}
        for future in tqdm(as_completed(futures), total=len(futures),
                           desc='Read SSD sources / SHA-256', unit='file',
                           disable=not config.get('progress_bars', True), mininterval=0.25):
            key = futures[future]
            try:
                snapshots[key] = future.result()
            except Exception as exc:
                # Source-integrity errors cannot be bypassed by another decoder.
                output[key] = {'pixels': None, 'owner': None, 'decode_backend': 'not_decoded',
                               'fallback_reason': None, 'error': type(exc).__name__ + ': ' + str(exc)}
    jpeg_keys = [key for key in keys if key in snapshots and snapshots[key]['is_jpeg']]
    if runtime['decoder'] is not None and jpeg_keys:
        s8_decode_nvidia_group(jpeg_keys, snapshots, runtime, runtime['decoder'],
                              runtime['decoder_mode'], config, output, failures)
        pending = [key for key in jpeg_keys if key not in output]
        if pending and runtime['decoder_mode'] == 'nvidia_gpu':
            hybrid = s8_get_hybrid_decoder(runtime)
            if hybrid is not None:
                s8_decode_nvidia_group(pending, snapshots, runtime, hybrid, 'nvidia_hybrid', config, output, failures)
    cpu_keys = [key for key in keys if key in snapshots and key not in output]
    # CPU image decode can run in parallel; only the main thread emits the one warning.
    allowed_cpu_keys = [key for key in cpu_keys if
                       (runtime['allow_non_jpeg_cpu'] if not snapshots[key]['is_jpeg'] else
                        runtime['allow_cpu_fallback'] and not runtime['require_gpu_decode'])]
    for key in allowed_cpu_keys:
        s8_warn_fallback_once(runtime, failures.get(key,
                              'non_jpeg_cpu_decode' if not snapshots[key]['is_jpeg'] else
                              runtime.get('gpu_unavailable_reason', 'gpu_decoder_unavailable')))
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {}
        for key in cpu_keys:
            reason = failures.get(key, 'non_jpeg_cpu_decode' if not snapshots[key]['is_jpeg'] else
                                  runtime.get('gpu_unavailable_reason', 'gpu_decoder_unavailable'))
            futures[pool.submit(s8_decode_cpu, snapshots[key], runtime, reason)] = key
        for future in as_completed(futures):
            output[futures[future]] = future.result()
    runtime['batch_history'].append({'backend': 'logical_batch', 'batch_size': len(items),
                                    'status': 'complete', 'decoded_count': sum(v['pixels'] is not None for v in output.values()),
                                    'failed_count': sum(v.get('error') is not None for v in output.values()),
                                    'jpeg_count': len(jpeg_keys), 'unique_source_count': len(keys),
                                    'elapsed_seconds': round(time.perf_counter() - started, 6)})
    return output


def s8_release_decoded(decoded, runtime):
    if runtime.get('stream') is not None:
        runtime['stream'].synchronize()
    for record in decoded.values():
        record['pixels'] = None
        record['owner'] = None
    decoded.clear()
    gc.collect()
    if runtime.get('cuda_available'):
        with cp.cuda.Device(runtime['device_id']):
            cp.get_default_memory_pool().free_all_blocks()


def s8_ref_pixels(ref, decoded):
    key = str(ref['image_relpath'])
    record = decoded.get(key)
    if record is None or record.get('error') or record.get('pixels') is None:
        raise ValueError('Source cannot be rendered: ' + key + ': ' + str(None if record is None else record.get('error')))
    pixels = record['pixels']
    expected = (int(ref['source_height']), int(ref['source_width']), 3)
    if min(expected) <= 0 or tuple(pixels.shape) != expected:
        raise ValueError('Decoded source dimensions disagree with verified manifest: ' + key)
    if pixels.dtype != np.uint8:
        raise ValueError('Decoded pixels must be RGB uint8: ' + key)
    expected_sha = ref.get('source_sha256')
    if expected_sha and str(expected_sha).lower() != str(record.get('source_sha256', '')).lower():
        raise ValueError('Decoded source SHA256 disagrees with artifact reference: ' + key)
    return pixels, record


def s8_pixels_host(pixels):
    if cp is not None and isinstance(pixels, cp.ndarray):
        with cp.cuda.Device(int(pixels.device.id)):
            return cp.asnumpy(pixels)
    return np.asarray(pixels)


def s8_crop_pixels(pixels, params, gpu_device_id=None):
    bbox = [float(value) for value in params['bbox_xyxy']]
    if len(bbox) != 4 or not all(math.isfinite(value) for value in bbox):
        raise ValueError('ROI bbox must contain four finite coordinates')
    x1, y1, x2, y2 = bbox
    height, width = pixels.shape[:2]
    if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
        raise ValueError('Original bbox is outside the source or has zero/negative area; it must not be silently repaired')
    margin = float(params.get('margin_fraction', 0.20))
    if not math.isfinite(margin) or margin < 0:
        raise ValueError('ROI margin_fraction must be finite and nonnegative')
    left = max(0, int(math.floor(x1 - margin * (x2 - x1))))
    top = max(0, int(math.floor(y1 - margin * (y2 - y1))))
    right = min(width, int(math.ceil(x2 + margin * (x2 - x1))))
    bottom = min(height, int(math.ceil(y2 + margin * (y2 - y1))))
    is_gpu = cp is not None and isinstance(pixels, cp.ndarray)
    uploaded_for_crop = False
    if not is_gpu and gpu_device_id is not None:
        if cp is None:
            raise RuntimeError('Cannot restore NVIDIA-decoded frame to GPU for cropping')
        with cp.cuda.Device(int(gpu_device_id)):
            pixels = cp.asarray(pixels)
        is_gpu = True
        uploaded_for_crop = True
    if is_gpu:
        with cp.cuda.Device(int(pixels.device.id)):
            cropped = cp.ascontiguousarray(pixels[top:bottom, left:right, :])
            host = cp.asnumpy(cropped)
    else:
        host = np.ascontiguousarray(pixels[top:bottom, left:right, :])
    return host, {'bbox_xyxy': bbox, 'crop_xyxy': [left, top, right, bottom],
                  'margin_fraction': margin, 'crop_backend': 'cupy_gpu' if is_gpu else 'numpy_cpu',
                  'gpu_source_restored_from_host': uploaded_for_crop,
                  'resized': False, 'source_to_crop_translation': [-left, -top]}


def s8_letterbox_panel(rgb, panel_size, label):
    """Aspect-preserving panel with a label strip outside all source pixels."""
    size = int(panel_size)
    if not 64 <= size <= 4096:
        raise ValueError('panel_size must be between 64 and 4096')
    height, width = rgb.shape[:2]
    if rgb.ndim != 3 or rgb.shape[2] != 3 or min(height, width) < 1 or rgb.dtype != np.uint8:
        raise ValueError('Panel input must be nonempty RGB uint8')
    scale = min(size / width, size / height)
    out_width = max(1, min(size, round(width * scale)))
    out_height = max(1, min(size, round(height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(rgb, (out_width, out_height), interpolation=interpolation)
    band = max(24, int(round(size * 0.09)))
    panel = np.full((size + band, size, 3), 16, dtype=np.uint8)
    left, top = (size - out_width) // 2, (size - out_height) // 2
    panel[top:top + out_height, left:left + out_width] = resized
    font_scale = max(0.32, min(0.55, size / 672))
    while cv2.getTextSize(str(label), cv2.FONT_HERSHEY_SIMPLEX, font_scale, 1)[0][0] > size - 8 and font_scale > 0.15:
        font_scale *= 0.90
    cv2.putText(panel, str(label), (4, size + band - 7), cv2.FONT_HERSHEY_SIMPLEX,
                font_scale, (230, 230, 230), 1, cv2.LINE_AA)
    return panel, {'input_to_panel_scale_xy': [out_width / width, out_height / height],
                   'input_to_panel_translation_xy': [left, top],
                   'source_to_panel_scale_xy': [out_width / width, out_height / height],
                   'source_to_panel_translation_xy': [left, top],
                   'image_region_xyxy': [left, top, left + out_width, top + out_height],
                   'label_strip_height': band, 'label': str(label),
                   'resize_interpolation': 'INTER_AREA' if scale < 1 else 'INTER_LINEAR'}


def s8_overlay_pixels(rgb, boxes, line_width=2):
    """Overlay every validated xyxy annotation without resizing or clinical text."""
    if not boxes:
        raise ValueError('bbox_overlay requires at least one box')
    thickness = int(line_width)
    if not 1 <= thickness <= 32:
        raise ValueError('overlay_line_width must be between 1 and 32')
    output = np.array(rgb, dtype=np.uint8, copy=True, order='C')
    height, width = output.shape[:2]
    geometries = []
    for box in boxes:
        coords = [float(value) for value in box['bbox_xyxy']]
        if len(coords) != 4 or not all(math.isfinite(value) for value in coords):
            raise ValueError('Overlay bbox must contain four finite coordinates')
        x1, y1, x2, y2 = coords
        if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
            raise ValueError('Overlay bbox lies outside source or has nonpositive area')
        # xyxy uses exclusive right/bottom edges; OpenCV rectangle endpoints are
        # inclusive. Drawing inside these integer bounds avoids a one-pixel shift.
        left, top = int(math.floor(x1)), int(math.floor(y1))
        right, bottom = int(math.ceil(x2)) - 1, int(math.ceil(y2)) - 1
        color = (255, 200, 0)  # RGB array: same color for every box, no class code.
        # Draw inward strips rather than cv2's center-aligned thick stroke, which
        # would modify pixels outside the supplied annotation rectangle.
        stroke = min(thickness, right - left + 1, bottom - top + 1)
        output[top:top + stroke, left:right + 1] = color
        output[bottom - stroke + 1:bottom + 1, left:right + 1] = color
        output[top:bottom + 1, left:left + stroke] = color
        output[top:bottom + 1, right - stroke + 1:right + 1] = color
        geometries.append({'bbox_xyxy': coords, 'raster_xyxy': [left, top, right + 1, bottom + 1],
                           'annotation_ids': list(box.get('annotation_ids', [])),
                           'line_width': stroke, 'color_rgb': list(color)})
    return output, {'boxes': geometries, 'overlay_backend': 'numpy_cpu',
                    'overlay_text': None, 'resized': False,
                    'source_to_output_scale_xy': [1.0, 1.0],
                    'source_to_output_translation_xy': [0, 0]}


def _s8_unique_panel_refs(refs):
    """Deduplicate annotation lineage rows, never silently merge distinct frames."""
    by_index = {}
    fields = ('image_relpath', 'source_sha256', 'source_width', 'source_height',
              'video_key', 'frame_index', 'is_target', 'time_offset_seconds')
    for ref in refs:
        index = int(ref.get('panel_index', len(by_index)))
        if index < 0:
            raise ValueError('panel_index must be nonnegative')
        if index in by_index:
            if any(by_index[index].get(field) != ref.get(field) for field in fields):
                raise ValueError('Conflicting source geometry for panel index ' + str(index))
        else:
            by_index[index] = ref
    indexes = sorted(by_index)
    if indexes != list(range(len(indexes))):
        raise ValueError('Panel indexes must be contiguous and zero based')
    return [by_index[index] for index in indexes]


def _s8_validate_temporal_refs(refs, require_target):
    videos = {str(ref.get('video_key', ref.get('video_id'))) for ref in refs
              if ref.get('video_key', ref.get('video_id')) is not None}
    if len(videos) > 1:
        raise ValueError('Temporal panels must all come from the same video')
    frames = [int(ref['frame_index']) for ref in refs]
    if frames != sorted(set(frames)):
        raise ValueError('Temporal panels require unique frames in chronological order')
    targets = sum(bool(ref.get('is_target', False)) for ref in refs)
    if require_target and targets != 1:
        raise ValueError('Temporal collage must identify exactly one target frame')
    if not require_target and targets > 1:
        raise ValueError('Progression panel may identify at most one target frame')
    offsets = [ref.get('time_offset_seconds') for ref in refs]
    known_offsets = [float(offset) for offset in offsets if offset is not None]
    if not all(math.isfinite(offset) for offset in known_offsets):
        raise ValueError('Temporal offsets must be finite when present')
    if known_offsets != sorted(known_offsets):
        raise ValueError('Temporal offsets disagree with chronological order')


def _s8_frame_label(ref):
    label = 'Frame ' + str(int(ref['frame_index']))
    offset = ref.get('time_offset_seconds')
    if offset is not None:
        if not math.isfinite(float(offset)):
            raise ValueError('Temporal label offset must be finite')
        label += ' ' + format(float(offset), '+.3f') + 's'
    if ref.get('is_target', False):
        label += ' target'
    return label


def _s8_panel_letter(index):
    # Spreadsheet-style A..Z, AA.. avoids any class-dependent visible label.
    label = ''
    number = index + 1
    while number:
        number, remainder = divmod(number - 1, 26)
        label = chr(65 + remainder) + label
    return label


def s8_render_artifact(spec, decoded, config):
    """Render all Sector 8 variants using only planner-provided source geometry.

    Clinical labels, QC scores/flags and arbitrary caller-provided display labels
    never enter pixel text. Semantic roles remain in the external provenance.
    """
    kind = str(spec['kind'])
    supported = {'roi_crop', 'bbox_overlay', 'evidence_panel', 'temporal_collage',
                 'progression_panel', 'quality_panel', 'comparison_panel', 'combined_panel'}
    if kind not in supported:
        raise ValueError('Unsupported visual artifact kind: ' + kind)
    refs = list(spec['source_refs'])
    params = dict(spec.get('params', {}))
    if not refs:
        raise ValueError('Artifact needs at least one source reference')
    metadata = {'kind': kind, 'source_decode_backends': {}, 'source_fallback_reasons': {},
                'source_pixel_residency': {}, 'source_sha256': {},
                'colour': 'RGB_uint8', 'apply_exif_orientation': False,
                'annotation_assisted': spec.get('annotation_assisted', params.get('annotation_assisted')),
                'clinical_labels_rendered': False, 'qc_scores_rendered': False}
    for ref in refs:
        _, record = s8_ref_pixels(ref, decoded)
        key = str(ref['image_relpath'])
        metadata['source_decode_backends'][key] = record['decode_backend']
        metadata['source_fallback_reasons'][key] = record.get('fallback_reason')
        metadata['source_pixel_residency'][key] = record.get('pixel_residency', 'unknown')
        metadata['source_sha256'][key] = record.get('source_sha256')
        if ref.get('split') is not None and spec.get('split') is not None and ref['split'] != spec['split']:
            raise ValueError('Artifact source split differs from artifact split')

    if kind == 'roi_crop':
        if len(refs) != 1:
            raise ValueError('ROI crop must have exactly one source reference')
        pixels, record = s8_ref_pixels(refs[0], decoded)
        host, geometry = s8_crop_pixels(pixels, params, record.get('gpu_device_id'))
        metadata.update(geometry)
        metadata['output_dimensions_wh'] = [int(host.shape[1]), int(host.shape[0])]
        return host, metadata

    if kind == 'bbox_overlay':
        if len(refs) != 1:
            raise ValueError('BBox overlay must have exactly one source reference')
        pixels, _ = s8_ref_pixels(refs[0], decoded)
        host, geometry = s8_overlay_pixels(s8_pixels_host(pixels), params.get('boxes', []),
                                          params.get('overlay_line_width', 2))
        metadata.update(geometry)
        metadata['output_dimensions_wh'] = [int(host.shape[1]), int(host.shape[0])]
        return host, metadata

    panel_size = int(params.get('panel_size', config.get('panel_size', 336)))
    panels, panel_geometries = [], []

    def add_panel(ref, operation='full', temporal_label=False):
        pixels, record = s8_ref_pixels(ref, decoded)
        source_geometry = {}
        if operation == 'crop':
            rgb, source_geometry = s8_crop_pixels(pixels, params, record.get('gpu_device_id'))
        elif operation == 'overlay':
            rgb, source_geometry = s8_overlay_pixels(s8_pixels_host(pixels), params.get('boxes', []),
                                                      params.get('overlay_line_width', 2))
        elif operation == 'full':
            rgb = s8_pixels_host(pixels)
        else:
            raise ValueError('Unknown panel operation: ' + str(operation))
        label = _s8_frame_label(ref) if temporal_label else _s8_panel_letter(len(panels))
        panel, geometry = s8_letterbox_panel(rgb, panel_size, label)
        geometry.update({'image_relpath': str(ref['image_relpath']), 'source_sha256': ref.get('source_sha256'),
                         'frame_index': ref.get('frame_index'), 'video_key': ref.get('video_key'),
                         'role': ref.get('role'), 'operation': operation,
                         'is_target': bool(ref.get('is_target', False)),
                         'time_offset_seconds': ref.get('time_offset_seconds'),
                         'source_geometry': source_geometry})
        if operation == 'crop':
            sx, sy = geometry['input_to_panel_scale_xy']
            tx, ty = geometry['input_to_panel_translation_xy']
            left, top = source_geometry['crop_xyxy'][:2]
            # Coordinates refer to the original source, not the cropped array.
            geometry['source_to_panel_translation_xy'] = [tx - left * sx, ty - top * sy]
        panels.append(panel)
        panel_geometries.append(geometry)

    if kind == 'evidence_panel':
        if len(refs) != 1:
            raise ValueError('Evidence panel needs one original source')
        add_panel(refs[0])
        add_panel(refs[0], 'crop')
    elif kind in ('temporal_collage', 'progression_panel'):
        ordered = _s8_unique_panel_refs(refs)
        _s8_validate_temporal_refs(ordered, require_target=kind == 'temporal_collage')
        if len(ordered) < 2:
            raise ValueError('Temporal/progression panels require at least two different frames')
        for ref in ordered:
            add_panel(ref, temporal_label=True)
    elif kind == 'comparison_panel':
        if len(refs) != 2:
            raise ValueError('Comparison panel requires exactly two source references')
        if refs[0]['image_relpath'] == refs[1]['image_relpath']:
            raise ValueError('Comparison panel requires different source images')
        for ref in refs:
            add_panel(ref)
    elif kind == 'quality_panel':
        # The planner determines eligibility and selection. QC scores stay in
        # its manifest; this panel never visualizes the answer in text or colors.
        for ref in _s8_unique_panel_refs(refs):
            add_panel(ref)
    elif kind == 'combined_panel':
        target, neighbors = refs[0], refs[1:]
        if neighbors:
            if any(ref['image_relpath'] == target['image_relpath'] for ref in neighbors):
                raise ValueError('Combined temporal neighbors must exclude target source')
            _s8_validate_temporal_refs(neighbors, require_target=False)
            videos = {str(ref.get('video_key')) for ref in refs if ref.get('video_key') is not None}
            if len(videos) > 1:
                raise ValueError('Combined temporal context must use target video')
        add_panel(target)
        if params.get('bbox_xyxy') is not None:
            add_panel(target, 'crop')
        if params.get('boxes'):
            add_panel(target, 'overlay')
        for ref in neighbors:
            add_panel(ref, temporal_label=True)
        if len(panels) < 2:
            raise ValueError('Combined panel requires at least one ROI, overlay, or neighbor')

    columns = min(len(panels), int(params.get('columns', config.get('panel_columns', 5))))
    if columns < 1 or columns > 5:
        raise ValueError('Panel columns must be between one and five')
    rows = (len(panels) + columns - 1) // columns
    ph, pw = panels[0].shape[:2]
    canvas = np.full((rows * ph, columns * pw, 3), 16, dtype=np.uint8)
    for index, panel in enumerate(panels):
        x, y = (index % columns) * pw, (index // columns) * ph
        canvas[y:y + ph, x:x + pw] = panel
        geometry = panel_geometries[index]
        geometry['panel_origin_xy'] = [x, y]
        geometry['source_to_output_scale_xy'] = geometry['source_to_panel_scale_xy']
        tx, ty = geometry['source_to_panel_translation_xy']
        geometry['source_to_output_translation_xy'] = [tx + x, ty + y]
    metadata.update({'composition_backend': 'opencv_cpu', 'panels': panel_geometries,
                     'panel_count': len(panels), 'layout_rows': rows, 'layout_columns': columns,
                     'output_dimensions_wh': [int(canvas.shape[1]), int(canvas.shape[0])]})
    return canvas, metadata


### Early GPU capability check

Run this before loading the S7 dataset. It decodes a synthetic baseline JPEG on the selected GPU and checks RGB layout and dimensions. If it fails, the output distinguishes package conflicts, CUDA initialization, and decoder/backend probe errors. This cell accesses no dataset or Drive files.

In [7]:
S8_GPU_PREFLIGHT = s8_gpu_preflight(CONFIG)
print('GPU preflight passed. Continue with storage definitions and S7 data loading.')


GPU environment: {'python': '3.13.15', 'packages': {'cupy-cuda12x': '14.2.0', 'nvidia-nvimgcodec-cu12': '0.9.0.20', 'nvidia-nvjpeg-cu12': '12.4.0.76'}, 'import_errors': {}}
CUDA device: NVIDIA L4 | CUDA runtime: 12090
JPEG decoder: nvidia_gpu | backends: ['HW_GPU_ONLY', 'GPU_ONLY']
GPU preflight passed. Continue with storage definitions and S7 data loading.


## 6. Staging, checkpoints, and Drive confirmation

Inputs are copied once from the mount to SSD, deduplicated by SHA. Files already present in `existing_ssd_roots` are verified before reuse. All computation reads from SSD. Originals used only as manifest references are loaded onto SSD on demand when preparing model input.

Each product is written to a temporary file, verified, and then checkpointed. A missing or mismatched hash triggers product regeneration. In a new runtime, previous products can be restored using the S8 manifest and verified; corrupted products are regenerated.

Export uses the Drive API directly to avoid a second copy through the DriveFS queue: **products → metadata → final catalog**. Server-side size + SHA-256 are compared; when Drive exposes only MD5, it is compared against the MD5 computed from the same SSD bytes validated by SHA-256. This method is recorded explicitly. A simple `exists()` or `fsync()` on the mount is not treated as cloud confirmation.

An unchanged rerun preserves manifest hashes. Operational logs and execution counters are kept separate from semantic identity. Upstream content is not modified.

Progress is reported for copying/reusing sources, verifying source SHA identities, restoring previous products, and checking output hashes. Drive export uses the **Google Drive API**, while DriveFS is the mounted filesystem used for incoming data. API inventory, directory preparation, uploads, and server checksum confirmation have separate bars; no extra image download is introduced for progress reporting.


In [8]:
# Imports are moved into the notebook's single import cell by the builder.



def s8_storage_progress(iterable=None, *, config=None, **kwargs):
    """One aggregate bar per operation; worker threads never create their own bars."""
    options = {'disable': not (config or {}).get('progress_bars', True),
               'dynamic_ncols': True, 'mininterval': 0.5, 'unit': 'file'}
    options.update(kwargs)
    return tqdm(iterable, **options)


def s8_storage_json_bytes(value):
    return (json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False, allow_nan=False) + '\n').encode('utf-8')


def s8_storage_hashes(path):
    sha, md5 = hashlib.sha256(), hashlib.md5()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            sha.update(block)
            md5.update(block)
    return {'sha256': sha.hexdigest(), 'md5': md5.hexdigest(), 'bytes': Path(path).stat().st_size}


def s8_storage_valid_sha(value):
    return isinstance(value, str) and len(value) == 64 and all(c in '0123456789abcdef' for c in value)


def s8_storage_path(root, relative):
    if not isinstance(relative, str) or not relative or '\\' in relative or '\x00' in relative:
        raise ValueError('Expected a nonempty safe POSIX relative path.')
    parts = relative.split('/')
    if any(p in ('', '.', '..') for p in parts) or relative.startswith('/'):
        raise ValueError(f'Unsafe relative path: {relative!r}')
    root = Path(root).resolve()
    path = root
    for part in parts:
        path = path / part
        if path.is_symlink():
            raise ValueError(f'Symbolic links are not allowed in artifact paths: {relative!r}')
    if not path.resolve().is_relative_to(root):
        raise ValueError('Artifact path escapes its declared root.')
    return path


def s8_storage_stat(path):
    stat = Path(path).stat()
    return (stat.st_dev, stat.st_ino, stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns)


def s8_storage_atomic_bytes(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial-' + uuid.uuid4().hex)
    try:
        with temporary.open('wb') as stream:
            stream.write(payload)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def s8_storage_atomic_json(path, value):
    s8_storage_atomic_bytes(path, s8_storage_json_bytes(value))


@contextmanager
def s8_workspace_lock(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a+') as handle:
        try:
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError('Another S8 cell is using this workspace. Wait for it to finish.') from None
        try:
            yield
        finally:
            fcntl.flock(handle.fileno(), fcntl.LOCK_UN)


def s8_storage_assert_ssd(local_root, config):
    local = Path(local_root).expanduser().resolve()
    drive = Path(config.get('drive_mount', '/content/drive')).expanduser().resolve()
    storage = Path(config['storage_root']).expanduser().resolve()
    if not local.is_absolute() or local.is_relative_to(drive) or local.is_relative_to(storage):
        raise ValueError('The processing/cache/output workspace must be SSD storage outside Google Drive.')
    return local


def s8_storage_copy_verified(source, destination, expected_sha256, expected_bytes=None):
    """Read an incoming source once; hash during copy, then verify SSD bytes.

    fsync only flushes the local filesystem; it is never evidence of remote Drive
    persistence. Existing SSD files are fully hashed before reuse.
    """
    if not s8_storage_valid_sha(expected_sha256):
        raise ValueError('A trusted upstream SHA-256 is required for source staging.')
    source, destination = Path(source), Path(destination)
    if expected_bytes is not None:
        expected_bytes = int(expected_bytes)
        if expected_bytes < 0:
            raise ValueError('Invalid expected file size.')
    if destination.is_file():
        identity = s8_storage_hashes(destination)
        if identity['sha256'] == expected_sha256 and (expected_bytes is None or identity['bytes'] == expected_bytes):
            return {'status': 'reused', **identity, 'local_path': str(destination)}
    if not source.is_file():
        raise FileNotFoundError(f'Source file is unavailable: {source}')
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.partial-' + uuid.uuid4().hex)
    before = s8_storage_stat(source)
    digest, total = hashlib.sha256(), 0
    try:
        with source.open('rb') as incoming, temporary.open('wb') as outgoing:
            for block in iter(lambda: incoming.read(8 * 1024 * 1024), b''):
                digest.update(block)
                total += len(block)
                outgoing.write(block)
            outgoing.flush()
            os.fsync(outgoing.fileno())
        if (digest.hexdigest() != expected_sha256 or (expected_bytes is not None and total != expected_bytes)
                or before != s8_storage_stat(source)):
            raise ValueError(f'Source-to-SSD byte identity does not match S7: {source.name}')
        actual = s8_storage_hashes(temporary)
        if actual['sha256'] != expected_sha256 or actual['bytes'] != total:
            raise RuntimeError('SSD copy failed destination SHA-256/size verification.')
        os.replace(temporary, destination)
        return {'status': 'copied', **actual, 'local_path': str(destination)}
    finally:
        temporary.unlink(missing_ok=True)


def s8_stage_sources(sources, config):
    """Stage each eligible source SHA once, using existing S7 SSD bytes first."""
    local = s8_storage_assert_ssd(config['local_root'], config)
    local.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(local).free < float(config.get('min_free_gib', 2)) * 1024 ** 3:
        raise RuntimeError('Not enough free SSD space for source staging.')
    items = sources.items() if isinstance(sources, dict) else ((s['image_relpath'], s) for s in sources)
    aliases, unique = {}, {}
    for relative, original in items:
        source = {**original, 'image_relpath': relative}
        s8_storage_path(config['storage_root'], relative)
        if not bool(source.get('prepare_eligible', True)):
            continue
        digest = source.get('sha256', source.get('source_sha256'))
        if not s8_storage_valid_sha(digest):
            raise ValueError(f'S7 supplies no valid source-image SHA-256: {relative}')
        source['sha256'] = digest
        aliases[relative] = source
        expected_bytes = source.get('bytes', source.get('image_size_bytes'))
        if digest in unique:
            prior_size = unique[digest].get('bytes', unique[digest].get('image_size_bytes'))
            if expected_bytes is not None and prior_size is not None and int(expected_bytes) != int(prior_size):
                raise ValueError('Identical source hashes have conflicting recorded byte sizes.')
        unique.setdefault(digest, source)
    cache = local / 'cache' / 'sources'
    copied, errors, results = [], [], {}

    def stage(source):
        digest = source['sha256']
        destination = cache / digest[:2] / (digest + '.image')
        size = source.get('bytes', source.get('image_size_bytes'))
        candidates = [s8_storage_path(root, source['image_relpath']) for root in config.get('existing_ssd_roots', [])]
        candidates = [candidate for candidate in candidates
                      if not candidate.resolve().is_relative_to(Path(config['storage_root']).resolve())]
        candidates.append(s8_storage_path(config['storage_root'], source['image_relpath']))
        # A corrupt optional SSD candidate does not conceal a valid Drive source.
        last_error = None
        for candidate in candidates:
            try:
                return s8_storage_copy_verified(candidate, destination, digest, size)
            except (OSError, ValueError, RuntimeError) as error:
                last_error = error
        raise last_error

    with s8_workspace_lock(local / '.source-stage.lock'):
        with ThreadPoolExecutor(max_workers=int(config.get('io_workers', 16))) as pool:
            futures = {pool.submit(stage, source): digest for digest, source in unique.items()}
            with s8_storage_progress(total=len(futures), config=config,
                                     desc='Copy/reuse sources to SSD + SHA-256') as progress:
                counts = {'copied': 0, 'reused': 0, 'failed': 0}
                for future in as_completed(futures):
                    digest = futures[future]
                    try:
                        result = future.result()
                        results[digest] = result
                        copied.append({'sha256': digest, **result})
                        counts[result['status']] += 1
                    except Exception as error:
                        errors.append({'sha256': digest, 'error': f'{type(error).__name__}: {error}'})
                        counts['failed'] += 1
                    progress.set_postfix(counts, refresh=False)
                    progress.update(1)
        s8_storage_atomic_json(local / 'checkpoints' / 'source_staging.json',
                               {'complete': not errors, 'sources': copied, 'errors': errors})
    if errors:
        raise RuntimeError(f'{len(errors)} source staging failures; verified SSD copies remain reusable. First: {errors[0]}')
    return {relative: {**source, **results[source['sha256']], 'image_relpath': relative}
            for relative, source in aliases.items()}


def s8_artifact_journal_path(spec, run_dir):
    artifact_id = str(spec['artifact_id'])
    if not artifact_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in artifact_id):
        raise ValueError('Artifact identifiers must be safe filename components.')
    return s8_storage_path(run_dir, 'checkpoints/artifacts/' + artifact_id + '.json')


def s8_artifact_valid(spec, run_dir):
    """Hash every candidate on reuse. Missing journal/hash means regeneration."""
    if spec.get('materialized', True) is False:
        return None
    journal = s8_artifact_journal_path(spec, run_dir)
    try:
        record = json.loads(journal.read_text(encoding='utf-8'))
        for name in ('artifact_id', 'relative_path', 'recipe_signature', 'render_signature'):
            if record.get(name) != spec.get(name):
                return None
        if record.get('complete') is not True or not s8_storage_valid_sha(record.get('sha256')):
            return None
        path = s8_storage_path(run_dir, record['relative_path'])
        identity = s8_storage_hashes(path)
        if identity['sha256'] != record['sha256'] or identity['bytes'] != record.get('bytes'):
            return None
        return {**record, 'reuse_status': 'verified_local'}
    except (OSError, ValueError, TypeError, KeyError):
        return None


def s8_commit_artifact(spec, encoded_bytes, run_dir, extra=None):
    """Install bytes, verify them, then atomically mark that artifact complete."""
    if spec.get('materialized', True) is False:
        raise ValueError('External full-frame references must not be materialized by S8.')
    for key in ('artifact_id', 'relative_path', 'recipe_signature', 'render_signature'):
        if not isinstance(spec.get(key), str) or not spec[key]:
            raise ValueError(f'Artifact specification requires {key}.')
    if not isinstance(encoded_bytes, (bytes, bytearray)) or not encoded_bytes:
        raise ValueError('Renderer must return nonempty encoded image bytes.')
    path = s8_storage_path(run_dir, spec['relative_path'])
    journal = s8_artifact_journal_path(spec, run_dir)
    expected = hashlib.sha256(encoded_bytes).hexdigest()
    s8_storage_atomic_bytes(path, encoded_bytes)
    identity = s8_storage_hashes(path)
    if identity['sha256'] != expected or identity['bytes'] != len(encoded_bytes):
        raise RuntimeError('Derived artifact failed local write verification; rerun this cell.')
    record = {**(extra or {}), **spec, **identity, 'complete': True}
    s8_storage_atomic_json(journal, record)
    return record


def s8_load_previous(config, run_dir):
    """Read prior metadata once per run; import only outputs with trusted hashes."""
    with s8_storage_progress(total=4, config=config, unit='step',
                             desc='Load/copy previous S8 metadata + SHA-256') as progress:
        return s8_load_previous_impl(config, run_dir, progress)


def s8_load_previous_impl(config, run_dir, progress):
    run_dir = Path(run_dir)
    checkpoint = run_dir / 'checkpoints' / 'previous_catalog.json'
    if checkpoint.is_file():
        try:
            previous = json.loads(checkpoint.read_text(encoding='utf-8'))
            if previous.get('schema_version') == 2:
                progress.set_postfix_str('SSD checkpoint reused', refresh=False)
                progress.update(4)
                return previous
        except (OSError, ValueError):
            pass
    previous = {'schema_version': 2, 'records': {}, 'state': None, 'state_sha256': None, 'warning': None}
    drive_root = s8_storage_path(config['storage_root'], config.get('sector8_relative_root', 'outputs/phase2/sector8'))
    catalog_path = drive_root / 'configs' / 'sector8_state.json'
    if not catalog_path.is_file():
        s8_storage_atomic_json(checkpoint, previous)
        progress.set_postfix_str('No previous Drive catalog; fresh run', refresh=False)
        progress.update(4)
        return previous
    try:
        raw = catalog_path.read_bytes()  # One small metadata read from Drive.
        state = json.loads(raw)
        progress.update(1)
        progress.set_postfix_str('Validate catalog descriptors', refresh=False)
        if state.get('complete') is not True:
            raise ValueError('Previous catalog is incomplete.')
        artifacts = {row['relative_path']: row for row in state['artifacts']}
        if len(artifacts) != len(state['artifacts']):
            raise ValueError('Previous catalog contains duplicate paths.')
        for relative, row in artifacts.items():
            s8_storage_path(drive_root, relative)
            if not s8_storage_valid_sha(row.get('sha256')):
                raise ValueError('Previous catalog has a missing/invalid artifact hash.')
        previous['state_sha256'] = hashlib.sha256(raw).hexdigest()
        previous['state'] = state
        progress.update(1)
        progress.set_postfix_str('Copy/reuse record snapshot and verify SHA-256', refresh=False)
        records = state.get('visual_artifacts', [])
        if state.get('record_snapshot'):
            relative = state['record_snapshot']
            declared = artifacts[relative]
            cached = s8_storage_path(Path(config['local_root']) / 'cache' / 'previous' / previous['state_sha256'], relative)
            s8_storage_copy_verified(s8_storage_path(drive_root, relative), cached, declared['sha256'], declared['bytes'])
            snapshot_bytes = cached.read_bytes()
            if cached.name.endswith('.json.gz'):
                snapshot_bytes = gzip.decompress(snapshot_bytes)
            value = json.loads(snapshot_bytes)
            records = value if isinstance(value, list) else value.get('artifacts', value.get('records', []))
        if not records:
            records = [row for row in state['artifacts'] if 'artifact_id' in row]
        if not records:
            descriptor = state.get('tables', {}).get('visual_manifest', state.get('tables', {}).get('visual_artifact_manifest'))
            if descriptor:
                relative = descriptor['relative_path']
                declared = artifacts[relative]
                cached = s8_storage_path(Path(config['local_root']) / 'cache' / 'previous' / previous['state_sha256'], relative)
                s8_storage_copy_verified(s8_storage_path(drive_root, relative), cached, declared['sha256'], declared['bytes'])
                if cached.suffix == '.parquet':
                    records = pd.read_parquet(cached).to_dict('records')
                elif cached.suffix == '.json':
                    value = json.loads(cached.read_text(encoding='utf-8'))
                    records = value if isinstance(value, list) else value.get('artifacts', value.get('records', []))
        progress.update(1)
        progress.set_postfix_str(f'Index {len(records):,} previous records', refresh=False)
        for record in records:
            if record.get('materialized', True) is False:
                continue
            relative = record.get('relative_path')
            declared = artifacts.get(relative)
            if not declared or declared.get('sha256') != record.get('sha256') or int(declared['bytes']) != int(record.get('bytes', -1)):
                continue  # No recorded hash: regenerate; never infer trust from existing bytes.
            if record.get('artifact_id') in previous['records']:
                raise ValueError('Duplicate artifact ID in previous manifest.')
            previous['records'][record['artifact_id']] = record
        progress.update(1)
        progress.set_postfix_str(f"{len(previous['records']):,} reusable records", refresh=False)
    except Exception as error:
        previous['records'] = {}
        previous['warning'] = f'Prior S8 metadata cannot be reused: {type(error).__name__}: {error}'
    s8_storage_atomic_json(checkpoint, previous)
    return previous


def s8_import_previous_artifact(spec, previous, run_dir, config):
    """One inbound copy for a matching prior derivative; corruption regenerates."""
    current = s8_artifact_valid(spec, run_dir)
    if current is not None:
        return current
    prior = previous.get('records', {}).get(spec['artifact_id'])
    if not prior or not s8_storage_valid_sha(prior.get('sha256')):
        return None
    if any(prior.get(key) != spec.get(key) for key in ('relative_path', 'recipe_signature', 'render_signature')):
        return None
    drive_root = s8_storage_path(config['storage_root'], config.get('sector8_relative_root', 'outputs/phase2/sector8'))
    try:
        identity = s8_storage_copy_verified(s8_storage_path(drive_root, prior['relative_path']),
                    s8_storage_path(run_dir, spec['relative_path']), prior['sha256'], prior['bytes'])
    except (OSError, ValueError, RuntimeError):
        return None
    record = {**prior, **spec, 'sha256': identity['sha256'], 'md5': identity['md5'],
              'bytes': identity['bytes'], 'complete': True, 'reuse_status': 'verified_drive_import'}
    s8_storage_atomic_json(s8_artifact_journal_path(spec, run_dir), record)
    return record


def s8_build_publication_delta(previous, state):
    """Metadata for a later GitHub decision; this function performs no Git writes."""
    previous_state = previous.get('state') or {}
    old = {row['relative_path']: row for row in previous_state.get('artifacts', [])}
    changes = []
    for row in state.get('artifacts', []):
        prior = old.get(row['relative_path'])
        if prior is None or prior.get('sha256') != row['sha256']:
            changes.append({'relative_path': row['relative_path'], 'old_sha256': (prior or {}).get('sha256'),
                            'sha256': row['sha256'], 'bytes': row['bytes'],
                            'action': 'add' if prior is None else 'replace',
                            'github_eligible': Path(row['relative_path']).suffix.lower() not in ('.jpg', '.jpeg', '.png')})
    return {'schema_version': 1, 'status': 'prepared_only', 'github_written': False,
            'basis': 'Previous saved S8 catalog; not a claim about current GitHub contents.',
            'changes': changes}


class S8CloudIdentityError(RuntimeError):
    pass


class S8CloudSourceChanged(RuntimeError):
    pass


def s8_cloud_match(record, metadata):
    """Prefer SHA256; MD5 fallback is bound to this locally verified SHA256."""
    if not isinstance(metadata, dict) or metadata.get('mimeType') == 'application/vnd.google-apps.folder':
        return None
    try:
        if int(metadata.get('size', -1)) != record['bytes']:
            return None
    except (ValueError, TypeError):
        return None
    cloud_sha = str(metadata.get('sha256Checksum') or '').lower()
    if cloud_sha:
        if cloud_sha != record['sha256']:
            return None
        return {'algorithm': 'sha256', 'cloud_checksum': cloud_sha, 'local_sha256': record['sha256']}
    cloud_md5 = str(metadata.get('md5Checksum') or '').lower()
    if cloud_md5 and cloud_md5 == record.get('md5'):
        return {'algorithm': 'md5_bound_to_local_sha256', 'cloud_checksum': cloud_md5,
                'local_md5': record['md5'], 'local_sha256': record['sha256'],
                'note': 'Drive supplies MD5 only; both hashes were calculated from the same verified SSD bytes.'}
    return None


def s8_cloud_relative(relative):
    if (not isinstance(relative, str) or not relative or "\\" in relative
            or "\x00" in relative or relative.startswith("/")
            or any(part in ("", ".", "..") for part in relative.split("/"))):
        raise ValueError("Expected a safe cloud-relative POSIX path.")
    return PurePosixPath(relative)


class S8DriveAPI:
    """Synchronous Drive REST uploads; no DriveFS reads or writes."""
    _base = 'https://www.googleapis.com/drive/v3'
    _upload = 'https://www.googleapis.com/upload/drive/v3'
    _folder_type = 'application/vnd.google-apps.folder'
    _fields = 'id,name,mimeType,size,sha256Checksum,md5Checksum,modifiedTime'

    def __init__(self, session, worker_factory=None):
        self.session = session
        self._worker_factory = worker_factory
        self._progress_config = {}
        self._active_inventory_progress = None

    def new_worker(self):
        if self._worker_factory is None:
            raise RuntimeError('A separate authorized session is required for parallel Drive uploads.')
        client = self._worker_factory()
        client._upload_directories = dict(getattr(self, '_upload_directories', {}))
        client._progress_config = dict(self._progress_config)
        return client

    def prepare_uploads(self, root_id, relatives):
        # Create directories serially before parallel file uploads: concurrent
        # folder creation can otherwise create duplicate Google Drive names.
        self._upload_directories = {}
        directories = set()
        for relative in relatives:
            parts = s8_cloud_relative(relative).parts[:-1]
            directories.update(parts[:count] for count in range(1, len(parts) + 1))
        with s8_storage_progress(total=len(directories), config=self._progress_config, unit='folder',
                                 desc='Drive API: prepare upload folders') as progress:
            for parts in sorted(directories, key=lambda value: (len(value), value)):
                parent = root_id if len(parts) == 1 else self._upload_directories[(root_id, '/'.join(parts[:-1]))]
                self._upload_directories[(root_id, '/'.join(parts))] = self._child(
                    parent, parts[-1], folder=True, create=True)['id']
                progress.update(1)

    def _response(self, method, url, **kwargs):
        try:
            response = self.session.request(method, url, timeout=(30, 120), **kwargs)
        except Exception as exc:
            # Upload session URLs can appear in transport exception messages.
            raise RuntimeError(f'Drive API {method} transport failed: {type(exc).__name__}') from None
        if not response.ok:
            # Do not print authenticated URLs, request headers or tokens.
            raise RuntimeError(f'Drive API {method} failed: HTTP {response.status_code}')
        return response

    def _request(self, method, url, **kwargs):
        return self._response(method, url, **kwargs).json()

    @staticmethod
    def _quote(value):
        return "'" + str(value).replace('\\', '\\\\').replace("'", "\\'") + "'"

    def _list(self, parent, name=None):
        query = f'{self._quote(parent)} in parents and trashed = false'
        if name is not None:
            query += f' and name = {self._quote(name)}'
        params = {'q': query, 'pageSize': 1000, 'spaces': 'drive',
                  'fields': f'nextPageToken,incompleteSearch,files({self._fields})'}
        result = []
        while True:
            page = self._request('GET', self._base + '/files', params=params)
            progress = self._active_inventory_progress
            if progress is not None:
                self._inventory_pages += 1
                self._inventory_rows += len(page.get('files', []))
                progress.set_postfix(api_pages=self._inventory_pages, listed=self._inventory_rows)
            if page.get('incompleteSearch'):
                raise RuntimeError('Drive API inventory is incomplete; no missing-file decision is safe.')
            result.extend(page.get('files', []))
            token = page.get('nextPageToken')
            if not token:
                return result
            params['pageToken'] = token

    def _child(self, parent, name, *, folder, create=False):
        matches = self._list(parent, name)
        if len(matches) > 1:
            raise S8CloudIdentityError(f'Duplicate Drive name; delivery blocked: {name}')
        if matches:
            item = matches[0]
            if (item.get('mimeType') == self._folder_type) != folder:
                raise S8CloudIdentityError(f'Drive file/folder conflict: {name}')
            return item
        if not create:
            return None
        body = {'name': name, 'parents': [parent]}
        if folder:
            body['mimeType'] = self._folder_type
        # A failed/unknown create is never repeated here. The next outer pass
        # lists this exact path first, so an accepted create can be recovered.
        return self._request('POST', self._base + '/files',
                             params={'fields': self._fields}, json=body)

    def resolve_root(self, parts):
        parent = 'root'
        required = getattr(self, '_required_prefix_length', 1)
        with s8_storage_progress(total=len(parts), config=self._progress_config, unit='folder',
                                 desc='Drive API: resolve project path', leave=False) as progress:
            for index, name in enumerate(parts):
                item = self._child(parent, name, folder=True, create=index >= required)
                if item is None:
                    raise S8CloudIdentityError(
                        f'Configured Drive project is absent: {name}; verify the authenticated account and storage_root.')
                parent = item['id']
                progress.update(1)
        return parent

    def inventory(self, root_id, relatives):
        # Directory/name caches last for this scan only. Every new pass re-lists.
        directories = {'': root_id}
        listings = {}

        def listing(folder_id):
            if folder_id not in listings:
                index = {}
                for item in self._list(folder_id):
                    index.setdefault(item['name'], []).append(item)
                listings[folder_id] = index
            return listings[folder_id]

        def select(folder_id, name):
            matches = listing(folder_id).get(name, [])
            if len(matches) > 1:
                raise S8CloudIdentityError(f'Duplicate Drive name; delivery blocked: {name}')
            return matches[0] if matches else None

        def directory(parts):
            key = '/'.join(parts)
            if key in directories:
                return directories[key]
            parent = directory(parts[:-1])
            item = None if parent is None else select(parent, parts[-1])
            if item is not None and item.get('mimeType') != self._folder_type:
                raise S8CloudIdentityError(f'Drive folder conflict: {key}')
            directories[key] = None if item is None else item['id']
            return directories[key]

        result = {}
        with s8_storage_progress(total=len(relatives), config=self._progress_config,
                                 desc='Drive API: list files/checksum metadata') as progress:
            self._active_inventory_progress = progress
            self._inventory_pages, self._inventory_rows = 0, 0
            try:
                for relative in relatives:
                    parts = s8_cloud_relative(relative).parts
                    folder_id = directory(parts[:-1])
                    item = None if folder_id is None else select(folder_id, parts[-1])
                    if item is not None and item.get('mimeType') == self._folder_type:
                        raise S8CloudIdentityError(f'Drive file conflict: {relative}')
                    result[relative] = item
                    progress.update(1)
            finally:
                self._active_inventory_progress = None
        return result

    def upload(self, root_id, relative, local_path, existing_metadata):
        parts = s8_cloud_relative(relative).parts
        parent = root_id
        directories = getattr(self, '_upload_directories', {})
        directory_parts = []
        for name in parts[:-1]:
            directory_parts.append(name)
            cache_key = (root_id, '/'.join(directory_parts))
            if cache_key not in directories:
                directories[cache_key] = self._child(parent, name, folder=True, create=True)['id']
            parent = directories[cache_key]
        self._upload_directories = directories
        # Resolve immediately before writing; never trust an ID from an old scan.
        target = self._child(parent, parts[-1], folder=False, create=True)
        if existing_metadata is not None and target['id'] != existing_metadata['id']:
            raise S8CloudIdentityError(f'Drive identity changed before upload: {relative}')
        size = local_path.stat().st_size
        # Resumable protocol is appropriate for the larger Parquet artifacts.
        # A failed/unknown transfer is reconciled through fresh API metadata before
        # another attempt; the next attempt always opens the SSD file at byte 0.
        initiation = self._response(
            'PATCH', self._upload + '/files/' + target['id'],
            params={'uploadType': 'resumable', 'fields': self._fields},
            headers={'X-Upload-Content-Type': 'application/octet-stream',
                     'X-Upload-Content-Length': str(size)}, json={})
        upload_url = initiation.headers.get('Location')
        parsed = urlsplit(upload_url or '')
        if parsed.scheme != 'https' or parsed.hostname != 'www.googleapis.com':
            raise RuntimeError('Drive API did not return a trusted resumable upload URL.')
        headers = {'Content-Type': 'application/octet-stream', 'Content-Length': str(size)}
        # Each retry opens the file afresh at byte 0. AuthorizedSession is created
        # with transparent 401 retries disabled for these streaming requests.
        with local_path.open('rb') as stream:
            response = self._response('PUT', upload_url, headers=headers, data=stream)
            if response.status_code not in (200, 201):
                raise RuntimeError(f'Drive resumable upload is not complete: HTTP {response.status_code}')
        return target['id']



def s8_make_drive_api():
    """Optional Colab imports are declared in the notebook's import cell."""
    if globals().get('google_auth') is None or globals().get('colab_auth') is None or globals().get('AuthorizedSession') is None:
        raise RuntimeError('Google Colab auth and google-auth are required for confirmed cloud delivery.')
    colab_auth.authenticate_user()
    credentials, _ = google_auth.default(scopes=['https://www.googleapis.com/auth/drive'])
    def worker_factory():
        # Clone credentials and create an independent HTTP session for each worker.
        return S8DriveAPI(AuthorizedSession(copy.copy(credentials), max_refresh_attempts=0))
    return S8DriveAPI(AuthorizedSession(credentials, max_refresh_attempts=0), worker_factory=worker_factory)


def s8_cloud_parts(config):
    mount = PurePosixPath(str(config.get('drive_mount', '/content/drive')))
    storage = PurePosixPath(str(config['storage_root']))
    for anchor in (mount / 'MyDrive', mount / 'My Drive'):
        try:
            project = storage.relative_to(anchor)
            break
        except ValueError:
            pass
    else:
        raise ValueError('Default cloud delivery requires storage_root under the configured MyDrive mount.')
    if any(part in ('.', '..') for part in project.parts):
        raise ValueError('Unsafe Drive storage root.')
    output = s8_cloud_relative(config.get('sector8_relative_root', 'outputs/phase2/sector8'))
    return list(project.parts + output.parts), len(project.parts)


def s8_export_drive(state, run_dir, config, api=None):
    """Export a sealed SSD bundle; complete only after fresh Drive API checks.

    Source media are never copied back. Existing remote outputs are compared by
    server checksum and size; only missing/changed outputs receive a payload.
    Direct API delivery replaces the DriveFS outbound write queue used by v1.
    An interrupted upload is reconciled through remote metadata on rerun.
    The final catalog is delivered after every listed artifact is confirmed.
    """
    if state.get('complete') is not True:
        raise ValueError('Seal a complete local Sector 8 catalog before cloud delivery.')
    run_dir = s8_storage_assert_ssd(run_dir, config)
    catalog_relative = 'configs/sector8_state.json'
    catalog_path = s8_storage_path(run_dir, catalog_relative)
    if not catalog_path.is_file() or json.loads(catalog_path.read_text(encoding='utf-8')) != state:
        raise ValueError('The final SSD catalog differs from the state selected for export.')
    records = [dict(row) for row in state.get('artifacts', [])]
    if not records:
        raise ValueError('An empty artifact catalog cannot be delivered.')
    if catalog_relative in {row.get('relative_path') for row in records}:
        raise ValueError('The catalog must not hash itself.')
    records.append({'relative_path': catalog_relative, **s8_storage_hashes(catalog_path)})
    expected, seals = {}, {}
    with s8_storage_progress(total=len(records), config=config,
                             desc='Verify SSD export SHA-256/MD5') as progress:
        for declared in records:
            relative = declared['relative_path']
            if relative in expected:
                raise ValueError('Duplicate path in the SSD artifact catalog.')
            path = s8_storage_path(run_dir, relative)
            before = s8_storage_stat(path)
            identity = s8_storage_hashes(path)
            if (not s8_storage_valid_sha(declared.get('sha256')) or identity['sha256'] != declared['sha256']
                    or identity['bytes'] != declared.get('bytes') or before != s8_storage_stat(path)):
                raise S8CloudSourceChanged(f'SSD artifact is corrupt/changed: {relative}. Rerun rendering/finalization.')
            expected[relative] = {'relative_path': relative, **identity}
            seals[relative] = before

            progress.update(1)

    def check_local(relative=None):
        keys = [relative] if relative is not None else list(seals)
        for key in keys:
            try:
                if s8_storage_stat(run_dir / key) != seals[key]:
                    raise S8CloudSourceChanged(f'SSD artifact changed during delivery: {key}')
            except OSError:
                raise S8CloudSourceChanged(f'SSD artifact disappeared during delivery: {key}') from None

    def compare_remote(observed):
        result = {}
        with s8_storage_progress(total=len(expected), config=config, leave=False,
                                 desc='Compare remote size + SHA-256/MD5') as progress:
            for key, record in expected.items():
                result[key] = s8_cloud_match(record, observed.get(key))
                progress.update(1)
        return result

    poll_seconds = float(config.get('drive_ack_poll_seconds', 5))
    retry_seconds = float(config.get('drive_retry_seconds', 90))
    timeout_seconds = float(config.get('drive_ack_timeout_seconds', 3600))
    if min(poll_seconds, retry_seconds, timeout_seconds) <= 0:
        raise ValueError('Cloud acknowledgement intervals must be positive.')
    cloud_parts, project_length = s8_cloud_parts(config)
    journal = run_dir / 'checkpoints' / 'drive_delivery.json'
    prior_changes_path = run_dir / 'checkpoints' / 'drive_delivery_changes.json'
    changes = {}
    last_attempt, errors = {}, {}
    report = {'status': 'pending', 'complete': False, 'total': len(expected),
              'verified': 0, 'pending': len(expected), 'source_catalog_sha256': expected[catalog_relative]['sha256'],
              'cloud_path': 'MyDrive/' + '/'.join(cloud_parts), 'proofs': [], 'errors': {}}

    def checkpoint():
        report['checked_at_utc'] = datetime.now(timezone.utc).isoformat()
        s8_storage_atomic_json(journal, report)
        s8_storage_atomic_json(prior_changes_path, {'schema_version': 1, 'github_written': False,
                                 'changes': [changes[key] for key in sorted(changes)]})

    checkpoint()  # Validate every local byte before authentication/network activity.
    client = api if api is not None else s8_make_drive_api()
    if isinstance(client, S8DriveAPI):
        client._required_prefix_length = project_length
        client._progress_config = {'progress_bars': config.get('progress_bars', True)}
    deadline = time.monotonic() + timeout_seconds
    with s8_workspace_lock(run_dir / '.cloud-delivery.lock'), s8_storage_progress(
            total=len(expected), config=config, desc='Drive API: confirmed remote checksums') as confirmation:
        scan_count = 0
        while True:
            check_local()
            scan_count += 1
            confirmation.set_postfix(scan=scan_count, pending=report['pending'], refresh=False)
            root_id = client.resolve_root(cloud_parts)
            observed = client.inventory(root_id, list(expected))
            proofs = compare_remote(observed)
            pending = [key for key in expected if proofs[key] is None]
            if not pending:
                # A second independent API inventory is the final durability check.
                observed = client.inventory(client.resolve_root(cloud_parts), list(expected))
                proofs = compare_remote(observed)
                pending = [key for key in expected if proofs[key] is None]
            report.update(verified=len(expected) - len(pending), pending=len(pending),
                          pending_paths=pending, errors={key: value for key, value in errors.items() if key in pending})
            confirmation.n = len(expected) - len(pending)
            confirmation.set_postfix(scan=scan_count, pending=len(pending), errors=len(report['errors']))
            confirmation.refresh()
            if not pending:
                check_local()
                report.update(status='complete', complete=True,
                    proofs=[{'relative_path': key, 'bytes': record['bytes'], 'sha256': record['sha256'],
                             'file_id': observed[key]['id'], **proofs[key]} for key, record in expected.items()])
                checkpoint()
                print(f"S8 Drive delivery confirmed: {len(expected):,} files; {len(changes):,} payloads added/replaced.")
                return report
            checkpoint()
            if time.monotonic() >= deadline:
                raise TimeoutError(f'{len(pending)} Drive files remain unconfirmed. Rerun the export cell; matching remote files are reused.')
            data_pending = [key for key in pending if key != catalog_relative]
            image_pending = [key for key in data_pending if Path(key).suffix.lower() in ('.jpg', '.jpeg', '.png')]
            eligible = image_pending if image_pending else data_pending if data_pending else [catalog_relative]
            if isinstance(client, S8DriveAPI):
                client._upload_directories = {}
            now = time.monotonic()
            eligible = [key for key in eligible if now - last_attempt.get(key, float('-inf')) >= retry_seconds]
            uploaded = False
            workers = int(config.get('drive_api_workers', 4))
            if workers < 1:
                raise ValueError('drive_api_workers must be positive.')
            # Test/injected adapters without worker factories are deliberately
            # serial; production clients each own a separate authorized session.
            if (not callable(getattr(client, 'new_worker', None))
                    or (isinstance(client, S8DriveAPI) and client._worker_factory is None)):
                workers = 1
            if isinstance(client, S8DriveAPI):
                client.prepare_uploads(root_id, eligible)
            thread_clients = threading.local()

            def upload_one(relative):
                check_local(relative)
                worker_client = client
                if workers > 1:
                    if not hasattr(thread_clients, 'client'):
                        thread_clients.client = client.new_worker()
                    worker_client = thread_clients.client
                worker_client.upload(root_id, relative, run_dir / relative, observed.get(relative))
                check_local(relative)
                return relative

            for relative in eligible:
                fresh = observed.get(relative)
                last_attempt[relative] = time.monotonic()
                changes.setdefault(relative, {
                    'relative_path': relative, 'action': 'add' if fresh is None else 'replace',
                    'previous_cloud_sha256': (fresh or {}).get('sha256Checksum'),
                    'previous_cloud_md5': (fresh or {}).get('md5Checksum'),
                    'sha256': expected[relative]['sha256'], 'bytes': expected[relative]['bytes'],
                })
            phase = 'images' if image_pending else 'metadata' if data_pending else 'final catalog'
            with ThreadPoolExecutor(max_workers=workers) as pool, s8_storage_progress(
                    total=len(eligible), config=config,
                    desc=f'Drive API: upload changed {phase}') as upload_progress:
                futures = {pool.submit(upload_one, relative): relative for relative in eligible}
                completed_bytes, failed_uploads = 0, 0
                for position, future in enumerate(as_completed(futures), 1):
                    relative = futures[future]
                    try:
                        future.result()
                        errors.pop(relative, None)
                        uploaded = True
                        completed_bytes += expected[relative]['bytes']
                    except (S8CloudIdentityError, S8CloudSourceChanged):
                        for pending_future in futures:
                            pending_future.cancel()
                        raise
                    except Exception as error:
                        # Transport errors can contain authenticated URLs; retain type only.
                        errors[relative] = type(error).__name__
                        failed_uploads += 1
                    upload_progress.set_postfix(
                        uploaded_MiB=round(completed_bytes / 1024 ** 2, 1), failed=failed_uploads,
                        refresh=False)
                    upload_progress.update(1)
                    if position % 1000 == 0:
                        report['errors'] = dict(errors)
                        checkpoint()
            report['errors'] = dict(errors)
            checkpoint()
            if not uploaded:
                confirmation.set_postfix(scan=scan_count, pending=len(pending),
                                         status=f'waiting {poll_seconds:g}s before recheck')
                time.sleep(poll_seconds)


## 7. Benchmark recipes and local orchestration

In [9]:
# Benchmark recipes remain unassigned, without generated questions or answers.

def s8_build_benchmark_recipes(plan, artifact_records, config):
    """Create ordered visual variants, private annotation evidence and leakage groups.

    Recipes are previewable preparation outputs, not a training-ready QA dataset.
    Every component linked by physical frame identity OR source SHA must receive
    one final split later. The same-cohort flag only concerns input availability;
    a later question/answer validation must establish task comparability.
    """
    if artifact_records is None:
        records = {}
    elif isinstance(artifact_records, dict):
        records = artifact_records
    else:
        records = {row['artifact_id']: row for row in artifact_records}
        if len(records) != len(artifact_records):
            raise ValueError('Duplicate artifact records cannot enter benchmark recipes.')
    artifacts = {row['artifact_id']: row for row in plan['artifacts']}
    if len(artifacts) != len(plan['artifacts']):
        raise ValueError('Duplicate artifact identities in visual plan.')
    sources = plan['sources']
    comparison_variants = list(config.get('same_cohort_variants', [
        'full', 'crop', 'fullcrop', 'overlay', 'temporal_multi', 'temporal_collage']))
    if not comparison_variants or any(name not in S8_VARIANT_NAMES for name in comparison_variants):
        raise ValueError('same_cohort_variants must name supported variants.')

    # Union-find operates on source identities, not path text. Equal byte hashes
    # and equivalent physical video/frame identities join aliases consistently.
    parents = {}
    def find(token):
        parents.setdefault(token, token)
        root = token
        while parents[root] != root:
            root = parents[root]
        while token != root:
            token, parents[token] = parents[token], root
        return root
    def unite(tokens):
        roots = [find(token) for token in tokens]
        if roots:
            leader = min(roots)
            for root in roots:
                parents[root] = leader
    def tokens_for_path(path):
        source = sources[path]
        physical = source.get('physical_frame_id') or s8_json_hash({'video_key': source['video_key'], 'frame_index': source['frame_index']})
        return ['frame:' + str(physical), 'sha256:' + str(source['sha256'])]
    for path in sorted(sources):
        unite(tokens_for_path(path))
    for artifact in plan['artifacts']:
        unite([token for ref in artifact['source_refs'] for token in tokens_for_path(ref['image_relpath'])])
    # Multi-image variants can join artifacts even if neither was materialized as
    # a collage. This keeps their source-sharing requirements explicit.
    for subject in plan['subjects']:
        for artifact_ids in subject['variants'].values():
            unite([token for aid in artifact_ids for ref in artifacts[aid]['source_refs']
                   for token in tokens_for_path(ref['image_relpath'])])
    components = {}
    for path in sorted(sources):
        root = find(tokens_for_path(path)[0])
        components.setdefault(root, []).append(path)
    group_for_path, shared_groups = {}, []
    for paths in components.values():
        tokens = sorted({token for path in paths for token in tokens_for_path(path)})
        group_id = s8_json_hash({'physical_or_byte_shared_source_component': tokens})
        for path in paths:
            group_for_path[path] = group_id
        shared_groups.append({'shared_source_group_id': group_id, 'source_image_relpaths': sorted(paths),
            'physical_frame_ids': sorted({str(sources[path]['physical_frame_id']) for path in paths}),
            'source_sha256_values': sorted({sources[path]['sha256'] for path in paths}),
            'upstream_splits': sorted({s for path in paths for s in sources[path].get('upstream_splits', [])}),
            'final_split': None, 'training_ready': False, 'split_assignment_status': 'pending',
            'assignment_constraint': 'All physical-frame or SHA-sharing members and their variants must remain in one final split.'})
    shared_groups.sort(key=lambda row: row['shared_source_group_id'])

    families = {
        'full': ['finding_identification', 'visual_quality_assessment'],
        'crop': ['indicated_region_characterization'],
        'fullcrop': ['indicated_region_characterization', 'global_local_comparison'],
        'fullcrop_panel': ['indicated_region_characterization', 'global_local_comparison'],
        'overlay': ['indicated_region_interpretation'],
        'temporal_multi': ['neighbor_frame_comparison'],
        'temporal_collage': ['neighbor_frame_comparison'],
        'progression': ['observed_verified_anchor_comparison'],
        'qc': ['visual_quality_assessment'],
        'comparison': ['verified_reference_image_comparison'],
        'combined': ['multi_view_evidence_interpretation'],
    }
    examples, ground_truth, counts = [], [], {name: 0 for name in S8_VARIANT_NAMES}
    same_cohort_subjects = 0
    for subject in plan['subjects']:
        sid = subject['subject_id']
        available = list(subject['variants'])
        missing = [name for name in comparison_variants if name not in subject['variants']]
        same_cohort = not missing
        same_cohort_subjects += int(same_cohort)
        truth_id = s8_json_hash({'source_annotation_evidence': sid})
        metadata = subject.get('private_label_metadata', {})
        ground_truth.append({'ground_truth_ref': truth_id, 'subject_id': sid,
            'annotation_id': subject['annotation_id'], 'image_relpath': subject['source_image_relpath'],
            'video_key': subject['video_key'], 'frame_index': subject['frame_index'],
            'finding_segment_id': subject.get('finding_segment_id'),
            'source_label': metadata.get('label'), 'clinical_group': metadata.get('clinical_group'),
            'bbox_xyxy': metadata.get('bbox_xyxy'), 'verified_source_labels': metadata.get('all_verified_source_labels', []),
            'source_qc': metadata.get('source_qc', {}),
            'comparison_reference_image_relpath': subject.get('comparison_reference_image_relpath'),
            'comparison_reference_verified_labels': (sources[subject['comparison_reference_image_relpath']]['verified_labels']
                if subject.get('comparison_reference_image_relpath') else []),
            'question': None, 'answer': None, 'qa_validation_status': 'not_authored',
            'ground_truth_scope': 'Source annotations only; not automatic answers to arbitrary future questions.'})
        for variant, artifact_ids in subject['variants'].items():
            ordered_inputs, component_ids, complete = [], set(), True
            assisted = False
            for order, aid in enumerate(artifact_ids):
                artifact = artifacts[aid]
                assisted = assisted or bool(artifact.get('annotation_assisted'))
                for ref in artifact['source_refs']:
                    component_ids.add(group_for_path[ref['image_relpath']])
                if artifact['materialized']:
                    record = records.get(aid)
                    if record is not None:
                        if record.get('recipe_signature') not in (None, artifact['recipe_signature']):
                            raise ValueError('Derivative record belongs to a different recipe: ' + aid)
                        if not isinstance(record.get('sha256'), str) or len(record['sha256']) != 64 or any(c not in '0123456789abcdef' for c in record['sha256']):
                            raise ValueError('Derivative record lacks valid SHA-256: ' + aid)
                    relative = record['relative_path'] if record else artifact['relative_path']
                    sha = record['sha256'] if record else None
                    width, height = (record.get('width'), record.get('height')) if record else (None, None)
                    storage = 'derived'
                    complete = complete and record is not None
                else:
                    ref = artifact['source_refs'][0]
                    relative, sha = ref['image_relpath'], ref['source_sha256']
                    width, height = ref['source_width'], ref['source_height']
                    storage = 'original'
                role = ('target' if order == 0 else 'context')
                if variant == 'fullcrop':
                    role = 'full_frame' if order == 0 else 'region_of_interest'
                elif variant == 'temporal_multi':
                    physical = artifact['source_refs'][0]
                    role = 'target' if physical['image_relpath'] == subject['source_image_relpath'] else 'temporal_context'
                elif variant != 'full':
                    role = variant
                ordered_inputs.append({'input_id': s8_json_hash({'example_subject': sid, 'variant': variant, 'order': order, 'artifact_id': aid}),
                    'artifact_id': aid, 'input_order': order, 'storage': storage, 'relative_path': relative,
                    'sha256': sha, 'width': width, 'height': height, 'image_role': role})
                temporal_metadata = subject.get('variant_input_metadata', {}).get(variant)
                if temporal_metadata is not None:
                    ordered_inputs[-1].update(temporal_metadata[order])
            if len(component_ids) != 1:
                raise ValueError('A benchmark example has not been assigned one shared-source component.')
            eid = s8_json_hash({'subject_id': sid, 'variant': variant, 'ordered_artifact_ids': artifact_ids})
            examples.append({'example_id': eid, 'subject_id': sid, 'variant': variant,
                'ordered_inputs': ordered_inputs, 'image_count': len(ordered_inputs),
                'supported_task_families': families[variant], 'ground_truth_ref': truth_id,
                'annotation_assisted': assisted, 'pixels_available': bool(complete),
                'shared_source_group_id': next(iter(component_ids)),
                'same_cohort_subject_id': sid, 'same_cohort_eligible': same_cohort,
                'same_cohort_variants': comparison_variants, 'same_cohort_missing_variants': missing,
                'available_subject_variants': available,
                'same_cohort_scope': 'Same source annotation and available inputs; question/task equivalence remains unvalidated.',
                'final_split': None, 'training_ready': False, 'split_assignment_status': 'pending',
                'question_status': 'not_authored', 'qa_validation_status': 'not_authored'})
            counts[variant] += 1
    examples.sort(key=lambda row: (row['subject_id'], row['variant']))
    ground_truth.sort(key=lambda row: row['subject_id'])
    return {'examples': examples, 'private_ground_truth': ground_truth,
            'shared_source_groups': shared_groups,
            'coverage': {'subjects': len(plan['subjects']), 'examples': len(examples),
                'examples_by_variant': counts, 'same_cohort_variants': comparison_variants,
                'same_cohort_subjects': same_cohort_subjects, 'shared_source_groups': len(shared_groups),
                'pixels_available_examples': sum(row['pixels_available'] for row in examples),
                'training_ready': False, 'final_split_assignment': 'pending', 'questions_and_answers': 'not_authored'}}


def s8_assert_benchmark_ready(examples):
    """Explicit gate for training/evaluation; visual previews do not pass this gate."""
    rows = examples.get('examples', []) if isinstance(examples, dict) else list(examples)
    if not rows:
        raise RuntimeError('No benchmark examples are available.')
    bad = [r['example_id'] for r in rows if r.get('training_ready') is not True
           or r.get('split_assignment_status') != 'assigned'
           or r.get('final_split') not in ('train', 'validation', 'val', 'test')
           or r.get('qa_validation_status') != 'validated' or r.get('pixels_available') is not True]
    if bad:
        raise RuntimeError(f'{len(bad)} examples are not ready: verified final split assignment, validated QA, and available pixels are required. S8 previews remain allowed.')
    groups = {}
    for row in rows:
        split = 'validation' if row['final_split'] == 'val' else row['final_split']
        old = groups.setdefault(row['shared_source_group_id'], split)
        if old != split:
            raise RuntimeError('A shared-source component crosses final splits.')
    return True


def s8_load_model_input(example, question, config, run_dir):
    """Load an ordered, hash-verified PIL projection with no labels or filenames.

    Original frames are staged to the SSD on demand. Derivatives are loaded from
    the verified local run. This function is a preview/inference adapter; calling
    it does not certify a train/test split or create clinical ground truth.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError('Supply a non-empty question; S8 does not fabricate QA.')
    inputs = example.get('ordered_inputs', [])
    if not inputs or [r['input_order'] for r in inputs] != list(range(len(inputs))):
        raise ValueError('Benchmark input order must be contiguous and zero based.')
    originals = {}
    for item in inputs:
        sha = item.get('sha256')
        if not isinstance(sha, str) or len(sha) != 64 or any(c not in '0123456789abcdef' for c in sha):
            raise RuntimeError('Model input has no verified pixel hash; materialize and verify it first.')
        if item.get('storage') == 'original':
            path = str(s8_safe_relative(item['relative_path']))
            originals[path] = {'image_relpath': path, 'sha256': sha,
                'width': item.get('width'), 'height': item.get('height'), 'prepare_eligible': True}
        elif item.get('storage') != 'derived':
            raise ValueError('Unknown model-input storage scope.')
    staged = s8_stage_sources(originals, config) if originals else {}
    images, roles = [], []
    for item in tqdm(inputs, desc='Verify model-input SHA / load images', unit='image',
                     disable=not config.get('progress_bars', True)):
        if item['storage'] == 'original':
            staged_item = staged[item['relative_path']]
            path = Path(staged_item['local_path'] if isinstance(staged_item, dict) else staged_item)
        else:
            path = s8_safe_path(run_dir, item['relative_path'])
        if s8_sha256(path) != item['sha256']:
            raise RuntimeError('Model-input hash changed; regenerate/restage the affected artifact.')
        with Image.open(path) as opened:
            rgb = opened.convert('RGB').copy()
        expected = (item.get('width'), item.get('height'))
        if None not in expected and rgb.size != expected:
            raise RuntimeError('Model-input dimensions disagree with its verified manifest.')
        rgb.info.clear()
        if hasattr(rgb, 'filename'):
            rgb.filename = ''
        images.append(rgb)
        role = item['image_role']
        if item.get('frame_index') is not None:
            role += '; frame=' + str(int(item['frame_index']))
        if item.get('time_offset_seconds') is not None:
            role += '; offset_seconds=' + format(float(item['time_offset_seconds']), '+.6g')
        roles.append(role)
    # Do not include example paths, source labels, annotation metadata, ground
    # truth, QC values, or dataset class filenames in this model-facing object.
    return {'question': question.strip(), 'images': images, 'image_roles': roles}


In [10]:
S8_IMPLEMENTATION_SHA256 = '7d6fd0c54d7204b2ed62065f049a28fb48a288cfd12906479f020311e5aa7a42'

"""Local orchestration. The notebook never executes code downloaded from GitHub."""


S8_RECIPE_VERSION = 's8-visual-preparation-v2'


def s8_safe_relative(value):
    raw = str(value)
    path = PurePosixPath(raw)
    if not raw or '\\' in raw or path.is_absolute() or '..' in path.parts or ':' in raw or path.as_posix() != raw or raw == '.':
        raise ValueError(f'Unsafe relative path: {raw!r}')
    return raw


def s8_safe_path(root, relative):
    root = Path(root).resolve()
    target = root.joinpath(*PurePosixPath(s8_safe_relative(relative)).parts).resolve()
    if root not in target.parents:
        raise ValueError('Path escapes the selected root.')
    return target


def s8_clean(value):
    if isinstance(value, dict):
        return {str(k): s8_clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [s8_clean(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return s8_clean(value.item())
    if value is None or value is pd.NA:
        return None
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def s8_json_text(value):
    return json.dumps(s8_clean(value), sort_keys=True, ensure_ascii=False, allow_nan=False, indent=2) + '\n'


def s8_json_hash(value):
    return hashlib.sha256(s8_json_text(value).encode()).hexdigest()


def s8_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024**2), b''):
            h.update(block)
    return h.hexdigest()


def s8_atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f'.{os.getpid()}.tmp')
    try:
        with temporary.open('w', encoding='utf-8') as stream:
            stream.write(s8_json_text(value))
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


@contextmanager
def s8_run_lock(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a+') as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError('Another S8 cell/process is using this SSD workspace.') from None
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


def s8_config_digest(config):
    return s8_json_hash(config)


def s8_render_environment():
    versions = {}
    for package in ('numpy', 'opencv-python-headless', 'Pillow',
                    'nvidia-nvimgcodec-cu12', 'nvidia-nvimgcodec-cu13',
                    'nvidia-nvjpeg-cu12', 'nvidia-nvjpeg-cu13', 'cupy-cuda12x', 'cupy-cuda13x'):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    versions['opencv_runtime'] = cv2.__version__
    versions['python'] = platform.python_version()
    return versions


def s8_prepare_run(inputs, sources, config):
    if not re.fullmatch(r'[0-9a-f]{40}', str(inputs['commit'])):
        raise ValueError('Upstream must identify one immutable Git commit.')
    if int(config['batch_size']) < 1 or int(config['batch_size']) > 1000:
        raise ValueError('batch_size must be between 1 and 1000 unique source files.')
    if not 1 <= int(config['jpeg_quality']) <= 100:
        raise ValueError('jpeg_quality must be between 1 and 100.')
    local = Path(config['local_root']).expanduser().resolve()
    drive = Path(config['storage_root']).expanduser().resolve()
    if local == drive or drive in local.parents:
        raise ValueError('local_root must be on local SSD, outside the Drive mount.')
    local.mkdir(parents=True, exist_ok=True)
    plan = s8_build_plan(inputs, sources, config)
    if not plan['artifacts'] or not plan['subjects']:
        raise ValueError('No eligible visual subjects were selected; review S7 evidence and configuration.')
    environment = s8_render_environment()
    core_hash = globals().get('S8_IMPLEMENTATION_SHA256', 'development')
    runtime_signature = s8_json_hash({'implementation': core_hash, 'environment': environment,
        'gpu_policy': {key: config.get(key) for key in
            ('require_gpu_decode', 'require_gpu_only_backend', 'allow_nvidia_hybrid',
             'allow_cpu_fallback', 'allow_non_jpeg_cpu', 'gpu_device_id')},
        'jpeg_quality': config['jpeg_quality']})
    for spec in plan['artifacts']:
        spec['recipe_signature'] = spec.get('recipe_signature') or s8_json_hash({
            'kind': spec['kind'], 'params': spec.get('params', {}), 'source_refs': spec['source_refs']})
        spec['render_signature'] = s8_json_hash({'recipe': spec['recipe_signature'], 'runtime': runtime_signature})
        if spec.get('materialized', True):
            spec['relative_path'] = f"products/{spec['render_signature'][:20]}/{spec['kind']}/{spec['artifact_id']}.jpg"
    processing = s8_json_hash({'upstream': inputs['state_sha256'], 'commit': inputs['commit'],
        'runtime': runtime_signature, 'artifacts': [(s['artifact_id'], s['render_signature']) for s in plan['artifacts']],
        'subjects': [s['subject_id'] for s in plan['subjects']]})
    run_dir = local / 'runs' / processing
    run_dir.mkdir(parents=True, exist_ok=True)
    plan.update(run_dir=str(run_dir), processing_signature=processing,
        config_digest=s8_config_digest(config), s7_state_sha256=inputs['state_sha256'],
        upstream_commit=inputs['commit'], environment=environment, core_sha256=core_hash,
        consumed_sha256=inputs.get('consumed_sha256', {}))
    # A compact durable plan checkpoint; full source evidence is finalized separately.
    s8_atomic_json(run_dir / 'checkpoints/plan_identity.json', {
        k: plan[k] for k in ('processing_signature', 'config_digest', 's7_state_sha256', 'upstream_commit', 'core_sha256')})
    counts = Counter(s['kind'] for s in plan['artifacts'])
    print('Planned views:', dict(counts), '| subjects:', len(plan['subjects']), '| skipped:', len(plan['skipped']))
    print('Final splits remain pending; preparation is allowed, training/evaluation is not yet ready.')
    return plan


def s8_artifact_batches(specs, batch_size):
    batch, paths = [], set()
    for spec in specs:
        required = {r['image_relpath'] for r in spec['source_refs']}
        if len(required) > batch_size:
            raise ValueError(f"Artifact {spec['artifact_id']} requires more than batch_size source files.")
        if batch and len(paths | required) > batch_size:
            yield batch, sorted(paths)
            batch, paths = [], set()
        batch.append(spec)
        paths |= required
    if batch:
        yield batch, sorted(paths)


def s8_reference_record(spec):
    ref = spec['source_refs'][0]
    return {'artifact_id': spec['artifact_id'], 'kind': spec['kind'], 'materialized': False,
        'relative_path': ref['image_relpath'], 'sha256': ref.get('sha256', ref.get('source_sha256')),
        'width': ref.get('width', ref.get('source_width')), 'height': ref.get('height', ref.get('source_height')),
        'bytes': ref.get('image_size_bytes'), 'recipe_signature': spec['recipe_signature'],
        'render_signature': spec['render_signature'], 'verification_scope': 'inherited_verified_S7_snapshot'}


def s8_process(plan, config):
    if plan['config_digest'] != s8_config_digest(config):
        raise RuntimeError('CONFIG changed. Rerun the source/plan cells before processing.')
    run_dir = Path(plan['run_dir'])
    if shutil.disk_usage(run_dir).free < float(config.get('min_free_gib', 2)) * 1024**3:
        raise RuntimeError('Insufficient free SSD space.')
    with s8_run_lock(Path(config['local_root']) / '.s8_processing.lock'):
        previous = s8_load_previous(config, run_dir)
        records, pending = [], []
        execution = {'reused': 0, 'restored': 0, 'generated': 0, 'failures': [], 'batches': []}
        def reconcile(spec):
            if not spec.get('materialized', True):
                return 'reference', s8_reference_record(spec), spec
            record = s8_artifact_valid(spec, run_dir)
            if record is not None:
                return 'reused', record, spec
            record = s8_import_previous_artifact(spec, previous, run_dir, config)
            if record is not None:
                return 'restored', record, spec
            return 'pending', None, spec
        originals = [spec for spec in plan['artifacts'] if not spec.get('materialized', True)]
        products = [spec for spec in plan['artifacts'] if spec.get('materialized', True)]
        records.extend(s8_reference_record(spec) for spec in originals)
        print(f'Retained {len(originals):,} original references; checking {len(products):,} derived products.')
        product_order = {spec['artifact_id']: index for index, spec in enumerate(products)}
        with ThreadPoolExecutor(max_workers=int(config.get('io_workers', 32))) as pool:
            futures = [pool.submit(reconcile, spec) for spec in products]
            for future in tqdm(as_completed(futures), total=len(futures), unit='product',
                    desc='Verify SSD SHA / restore previous products',
                    disable=not config.get('progress_bars', True)):
                status, record, spec = future.result()
                if status == 'pending':
                    pending.append(spec)
                else:
                    records.append(record)
                    if status in ('reused', 'restored'):
                        execution[status] += 1
        pending.sort(key=lambda spec: product_order[spec['artifact_id']])
        # Originals referenced only by the full-frame catalog are copied on demand by the loader.
        required = {r['image_relpath'] for spec in pending for r in spec['source_refs']}
        # Fail early on GPU configuration before transferring image payloads.
        runtime = s8_make_runtime(config) if pending else None
        if runtime is not None:
            s8_assert_gpu_runtime(runtime, config,
                require_jpeg=any(PurePosixPath(p).suffix.lower() in ('.jpg', '.jpeg') for p in required))
        staged = s8_stage_sources({p: plan['sources'][p] for p in sorted(required)}, config) if required else {}
        for number, (batch, paths) in enumerate(s8_artifact_batches(pending, int(config['batch_size'])), 1):
            decoded = {}
            print(f'Batch {number}: {len(paths)} unique files -> {len(batch)} products')
            try:
                decoded = s8_decode_batch([staged[p] for p in paths], runtime, config)
                for spec in tqdm(batch, desc=f'Render / encode / checkpoint batch {number}', unit='product',
                                 disable=not config.get('progress_bars', True)):
                    try:
                        rgb, metadata = s8_render_artifact(spec, decoded, config)
                        if rgb.dtype != np.uint8 or rgb.ndim != 3 or rgb.shape[2] != 3:
                            raise ValueError('Renderer must return RGB uint8 HxWx3.')
                        ok, encoded = cv2.imencode('.jpg', cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR),
                            [cv2.IMWRITE_JPEG_QUALITY, int(config['jpeg_quality'])])
                        if not ok:
                            raise RuntimeError('JPEG encoding failed.')
                        record = s8_commit_artifact(spec, encoded.tobytes(), run_dir, extra={
                            'width': int(rgb.shape[1]), 'height': int(rgb.shape[0]),
                            'kind': spec['kind'], 'materialized': True, 'metadata': metadata,
                            'jpeg_quality': int(config['jpeg_quality'])})
                        records.append(record)
                        execution['generated'] += 1
                    except Exception as error:
                        execution['failures'].append({'artifact_id': spec['artifact_id'], 'error': str(error)})
            except Exception as error:
                execution['failures'].append({'batch': number, 'error': str(error)})
            finally:
                if runtime is not None:
                    s8_release_decoded(decoded, runtime)
                execution['batches'].append({'batch': number, 'unique_files': len(paths), 'products': len(batch)})
                execution['gpu_batch_history'] = runtime.get('batch_history', []) if runtime else []
                s8_atomic_json(run_dir / 'checkpoints/execution.json', execution)
        if execution['failures']:
            raise RuntimeError(f"{len(execution['failures'])} failures. Rerun this cell; verified products are retained. First: {execution['failures'][0]}")
        if len(records) != len(plan['artifacts']):
            raise RuntimeError('Record coverage mismatch; no complete catalog written.')
        return s8_finalize(plan, records, execution, config)


def s8_flat_table(rows, empty_columns=('record_id',)):
    normalized = []
    for row in rows:
        item = {}
        for key, value in s8_clean(row).items():
            if isinstance(value, (dict, list)):
                item[key if key.endswith('_json') else key + '_json'] = json.dumps(value, ensure_ascii=False, sort_keys=True)
            else:
                item[key] = value
        normalized.append(item)
    if normalized:
        return pd.DataFrame(normalized, columns=sorted({key for row in normalized for key in row}))
    return pd.DataFrame({key: pd.Series(dtype='string') for key in empty_columns})


def s8_finalize(plan, records, execution, config):
    run_dir = Path(plan['run_dir'])
    specs = {s['artifact_id']: s for s in plan['artifacts']}
    by_id = {r['artifact_id']: r for r in records}
    # Runtime counters and repair/import history are checkpoints, not semantic content.
    # An unchanged rerun therefore keeps the same manifests and catalog hashes.
    records = [{k: v for k, v in record.items() if k not in ('reuse_status', 'created_at', 'created_at_utc', 'updated_at')}
               for record in sorted(records, key=lambda r: r['artifact_id'])]
    by_id = {r['artifact_id']: r for r in records}
    if len(by_id) != len(specs) or set(by_id) != set(specs):
        raise ValueError('Every planned view requires exactly one record.')
    views, links = [], []
    for aid, spec in tqdm(specs.items(), total=len(specs), desc='Final SSD product SHA verification',
                          unit='view', disable=not config.get('progress_bars', True)):
        record = by_id[aid]
        if spec.get('materialized', True) and s8_artifact_valid(spec, run_dir) is None:
            raise RuntimeError(f'Product changed before finalization: {aid}')
        views.append({**record, 'kind': spec['kind'], 'storage': 'derived' if spec.get('materialized', True) else 'source',
            'annotation_assisted': bool(spec.get('annotation_assisted', False)),
            'params': spec.get('params', {}), 'final_split': None, 'split_assignment_status': 'pending',
            'training_ready': False, 's7_state_sha256': plan['s7_state_sha256']})
        for index, ref in enumerate(spec['source_refs']):
            links.append({'artifact_id': aid, 'source_index': index, **ref})
    benchmark = s8_build_benchmark_recipes(plan, records, config)
    frames = {
        'visual_manifest': s8_flat_table(views, ('artifact_id',)),
        'visual_source_manifest': s8_flat_table(links, ('artifact_id', 'source_index')),
        'source_evidence_manifest': s8_flat_table(list(plan['sources'].values()), ('image_relpath',)),
        'benchmark_input_manifest': s8_flat_table(benchmark['examples'], ('example_id',)),
        'benchmark_ground_truth_manifest': s8_flat_table(benchmark['private_ground_truth'], ('example_id',)),
        'shared_source_groups': s8_flat_table(benchmark['shared_source_groups'], ('example_id',)),
        'skipped_products': s8_flat_table(plan['skipped'], ('reason',)),
    }
    prefix = f"metadata/{plan['processing_signature']}"
    artifacts, descriptors = [], {}
    def register(relative):
        path = run_dir / relative
        rec = {'relative_path': relative, 'sha256': s8_sha256(path), 'bytes': path.stat().st_size}
        artifacts.append(rec)
        return rec
    for name, frame in tqdm(frames.items(), desc='Write / hash manifest tables', unit='table',
                            disable=not config.get('progress_bars', True)):
        relative = f'{prefix}/results/{name}.parquet'
        path = run_dir / relative
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix('.parquet.tmp')
        frame.to_parquet(temporary, index=False)
        os.replace(temporary, path)
        descriptor = register(relative)
        schema_relative = f'{prefix}/results/{name}_schema.json'
        s8_atomic_json(run_dir / schema_relative, {'rows': len(frame), 'columns': list(frame.columns),
            'dtypes': {str(k): str(v) for k, v in frame.dtypes.items()}})
        register(schema_relative)
        descriptors[name] = {**descriptor, 'schema_path': schema_relative, 'rows': len(frame)}
    # JSONL preserves ordered model input lists without forcing consumers to decode Parquet JSON columns.
    relative = f'{prefix}/benchmark/model_inputs.jsonl.gz'
    path = run_dir / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.gz.tmp')
    with temporary.open('wb') as raw, gzip.GzipFile(filename='', mode='wb', fileobj=raw, mtime=0) as stream:
        for row in tqdm(benchmark['examples'], desc='Write ordered benchmark inputs', unit='recipe',
                        disable=not config.get('progress_bars', True)):
            stream.write((json.dumps(s8_clean(row), ensure_ascii=False, sort_keys=True, allow_nan=False) + '\n').encode())
    os.replace(temporary, path)
    register(relative)
    for name, value in [('sector8_config', config), ('upstream_snapshot', {
            'repository': config['repository'], 'commit': plan['upstream_commit'],
            'sector7_state_sha256': plan['s7_state_sha256'], 'consumed_sha256': plan['consumed_sha256']})]:
        relative = f'{prefix}/configs/{name}.json'
        s8_atomic_json(run_dir / relative, value)
        register(relative)
    relative = f'{prefix}/configs/artifact_records.json.gz'
    path = run_dir / relative
    temporary = path.with_suffix('.gz.tmp')
    with temporary.open('wb') as raw, gzip.GzipFile(filename='', mode='wb', fileobj=raw, mtime=0) as stream:
        stream.write(b'[')
        for index, record in enumerate(tqdm(records, desc='Write recovery record snapshot', unit='view',
                                            disable=not config.get('progress_bars', True))):
            if index:
                stream.write(b',')
            stream.write(json.dumps(s8_clean(record), sort_keys=True, ensure_ascii=False, allow_nan=False).encode())
        stream.write(b']\n')
    os.replace(temporary, path)
    register(relative)
    summary = {'phase': 2, 'sector': 8, 'complete': True, 'training_ready': False,
        'split_assignment_status': 'pending', 'upstream_commit': plan['upstream_commit'],
        'original_reference_views': sum(not s.get('materialized', True) for s in specs.values()),
        'derived_jpegs': sum(s.get('materialized', True) for s in specs.values()),
        'kind_counts': dict(Counter(s['kind'] for s in specs.values())),
        'source_files_in_catalog': len(plan['sources']), 'subjects': len(plan['subjects']),
        'benchmark_examples': len(benchmark['examples']), 'skipped': len(plan['skipped']),
        'benchmark_coverage': benchmark['coverage'], 'selection_max_subjects': config.get('max_subjects'),
        'limitations': ['No generated clinical ground truth or QA pairs.',
            'Observed anchor appearance is not clinical disease progression.',
            'QC selection uses uncalibrated metrics; no clinical quality ground truth.',
            'Final benchmark splits and question eligibility require downstream validation.']}
    relative = f'{prefix}/reports/summary.json'
    s8_atomic_json(run_dir / relative, summary)
    register(relative)
    for record in records:
        if specs[record['artifact_id']].get('materialized', True):
            artifacts.append({key: record[key] for key in ('relative_path', 'sha256', 'bytes')})
    state = {'schema_version': 1, 'phase': 2, 'sector': 8, 'complete': True,
        'processing_signature': plan['processing_signature'], 'core_source_sha256': plan['core_sha256'],
        'upstream_commit': plan['upstream_commit'], 'input_sha256': {'sector7_state': plan['s7_state_sha256']},
        'training_ready': False, 'split_assignment_status': 'pending',
        'config_snapshot': f'{prefix}/configs/sector8_config.json',
        'record_snapshot': f'{prefix}/configs/artifact_records.json.gz',
        'tables': descriptors, 'summary': f'{prefix}/reports/summary.json',
        'artifacts': sorted(artifacts, key=lambda r: r['relative_path'])}
    state['content_id'] = s8_json_hash(state)
    s8_atomic_json(run_dir / 'configs/sector8_state.json', state)
    print(f"SSD complete: {summary['derived_jpegs']:,} derived JPEGs, {summary['original_reference_views']:,} original references.")
    return {'state': state, 'run_dir': str(run_dir), 'tables': frames, 'summary': summary,
        'benchmark': benchmark, 'execution': execution, 'records': records}


def s8_prepare_github_review(run, config):
    """Local metadata-only review bundle. Does not clone, commit, push or contact GitHub."""
    root = Path(run['run_dir'])
    out = root / 'github_review' / config['sector8_relative_root']
    prior_path = Path(config['local_root']) / 'last_reviewed_metadata.json'
    prior = json.loads(prior_path.read_text()) if prior_path.is_file() else {}
    current, delta = {}, []
    review_artifacts = [item for item in run['state']['artifacts'] if not item['relative_path'].startswith('products/')]
    for descriptor in tqdm(review_artifacts, desc='Copy / verify local review metadata', unit='file',
                            disable=not config.get('progress_bars', True)):
        relative = descriptor['relative_path']
        if relative.startswith('products/'):
            continue
        current[relative] = descriptor['sha256']
        target = out / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        s8_storage_copy_verified(root / relative, target, descriptor['sha256'], descriptor.get('bytes'))
        if prior.get(relative) != descriptor['sha256']:
            delta.append({'relative_path': relative, 'old_sha256': prior.get(relative),
                'new_sha256': descriptor['sha256'], 'action': 'add' if relative not in prior else 'update'})
    target = out / 'configs/sector8_state.json'
    target.parent.mkdir(parents=True, exist_ok=True)
    catalog_source = root / 'configs/sector8_state.json'
    with tqdm(total=1, desc='Copy / verify review catalog', unit='file',
              disable=not config.get('progress_bars', True)) as progress:
        s8_storage_copy_verified(catalog_source, target, s8_sha256(catalog_source), catalog_source.stat().st_size)
        progress.update(1)
    s8_atomic_json(out / 'reports/metadata_hash_diff.json', {'baseline': 'previous_local_review_bundle',
        'changes': delta, 'images_included': False, 'github_publication_performed': False})
    s8_atomic_json(out / 'reports/product_hash_diff.json',
        s8_build_publication_delta(s8_load_previous(config, root), run['state']))
    s8_atomic_json(prior_path, current)
    print('Metadata-only review bundle:', out)
    return str(out)


## 8. Local self-test, then data loading

The self-test below uses a synthetic geometric pattern in RAM and a temporary directory. It checks pixel operations and product recovery; it is not a clinical example and does not test CUDA. Actual GPU testing takes place during the processing stage in Colab.

In [11]:
def s8_notebook_selftest():
    rgb = np.zeros((64, 64, 3), dtype=np.uint8)
    rgb[16:40, 12:36] = (200, 80, 30)
    source_hash = hashlib.sha256(rgb.tobytes()).hexdigest()
    ref = {'image_relpath': 'synthetic.jpg', 'source_sha256': source_hash,
           'source_width': 64, 'source_height': 64, 'frame_index': 1,
           'video_key': 'synthetic', 'panel_index': 0, 'is_target': True}
    decoded = {'synthetic.jpg': {'pixels': rgb, 'source_sha256': source_hash,
               'decode_backend': 'synthetic_array', 'pixel_residency': 'host'}}
    spec = {'artifact_id': 'synthetic_crop', 'kind': 'roi_crop', 'materialized': True,
            'relative_path': 'products/synthetic/crop.jpg', 'recipe_signature': 'r',
            'render_signature': 's', 'source_refs': [ref],
            'params': {'bbox_xyxy': [12, 16, 36, 40], 'margin_fraction': 0}}
    crop, _ = s8_render_artifact(spec, decoded, CONFIG)
    assert crop.shape == (24, 24, 3) and np.array_equal(crop, rgb[16:40, 12:36])
    overlay_spec = dict(spec, kind='bbox_overlay', params={'boxes': [{'bbox_xyxy': [12,16,36,40]}]})
    overlay, _ = s8_render_artifact(overlay_spec, decoded, CONFIG)
    assert overlay.shape == rgb.shape and not np.array_equal(overlay, rgb)
    assert np.array_equal(overlay[0:8], rgb[0:8])
    ok, encoded = cv2.imencode('.jpg', cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
    assert ok
    with tempfile.TemporaryDirectory(prefix='s8-selftest-') as folder:
        record = s8_commit_artifact(spec, encoded.tobytes(), folder)
        assert s8_artifact_valid(spec, folder)['sha256'] == record['sha256']
        (Path(folder) / spec['relative_path']).write_bytes(b'incomplete')
        assert s8_artifact_valid(spec, folder) is None
        repaired = s8_commit_artifact(spec, encoded.tobytes(), folder)
        assert repaired['sha256'] == record['sha256']
    print('PASS: crop, overlay, unchanged outside region, checkpoint, corruption and repair.')

s8_notebook_selftest()


PASS: crop, overlay, unchanged outside region, checkpoint, corruption and repair.


In [13]:
if CONFIG.get('mount_google_drive', True):
    if drive is None:
        raise RuntimeError('Google Colab Drive integration is unavailable in this Python environment.')
    drive.mount(CONFIG['drive_mount'])

INPUTS = s8_load_inputs(CONFIG)
SOURCES = s8_build_sources(INPUTS, CONFIG)
display(pd.DataFrame([{'table': name, 'rows': len(frame)} for name, frame in INPUTS['tables'].items()]))
print('Pinned upstream commit:', INPUTS['commit'])
print('Verified S7 catalog SHA:', INPUTS['state_sha256'])
print('Source files:', len(SOURCES), '| technically eligible:', sum(s['prepare_eligible'] for s in SOURCES.values()))


Mounted at /content/drive


Fetch / verify S7 catalog SHA-256:   0%|          | 0/1 [00:00<?, ?file/s]

Fetch / verify S7 metadata SHA-256:   0%|          | 0/9 [00:00<?, ?file/s]

Validate S7 table schemas:   0%|          | 0/3 [00:00<?, ?table/s]

Validate S7 source hash identities:   0%|          | 0/66127 [00:00<?, ?file/s]

Validate labelled source SHA references:   0%|          | 0/47239 [00:00<?, ?row/s]

Validate temporal source SHA references:   0%|          | 0/110666 [00:00<?, ?row/s]

,table,rows
0,image_qc_manifest,66127
1,labelled_frame_manifest_with_qc,47239
2,temporal_manifest_with_qc,110666


Pinned upstream commit: cb4ac8d8f94caeb83dcddfccf495cdd4188bf9f0
Verified S7 catalog SHA: a90f4048491f9aef0be931b2d4c85cf4a8ac61236534d44ca50c3331b073fd69
Source files: 66127 | technically eligible: 66127


In [14]:
PLAN = s8_prepare_run(INPUTS, SOURCES, CONFIG)
display(pd.DataFrame(sorted(Counter(s['kind'] for s in PLAN['artifacts']).items()), columns=['kind', 'planned_views']))
with pd.option_context('display.max_colwidth', None, 'display.max_columns', None):
    display(s8_skip_summary(PLAN))
    SKIPPED_DETAILS = s8_flat_table(PLAN['skipped'])
    detail_columns = [name for name in ('kind', 'selection_scope', 'finding_segment_id',
        'available_verified_frames', 'required_verified_frames',
        'available_verified_frame_indices_json', 'reason', 'status') if name in SKIPPED_DETAILS.columns]
    display(SKIPPED_DETAILS[detail_columns].head(12))
print('SSD run directory:', PLAN['run_dir'])


Planned views: {'bbox_overlay': 4126, 'combined_panel': 24164, 'comparison_panel': 3357, 'evidence_panel': 4127, 'full_frame': 66127, 'progression_panel': 8842, 'quality_panel': 47238, 'roi_crop': 4127, 'temporal_collage': 22220} | subjects: 47239 | skipped: 80475
Final splits remain pending; preparation is allowed, training/evaluation is not yet ready.


,kind,planned_views
0,bbox_overlay,4126
1,combined_panel,24164
2,comparison_panel,3357
3,evidence_panel,4127
4,full_frame,66127
5,progression_panel,8842
6,quality_panel,47238
7,roi_crop,4127
8,temporal_collage,22220


,kind,selection_scope,reason,records,explanation
0,combined_panel,annotation,no_additional_available_visual_evidence,23075,No usable bounding box or temporal neighbor is available to add another view. The original remains available; the combined panel is unavailable.
1,comparison_panel,annotation,no_verified_normal_reference_in_same_video,910,No distinct explicitly verified normal reference is available in the same video. Only the comparison panel is unavailable.
2,progression_panel,segment,segment_has_fewer_than_two_available_verified_anchors,13378,Only the progression panel is unavailable: the segment has fewer than two distinct verified frames. Its original frames remain available.
3,roi_crop,annotation,bbox_not_spatially_usable,43112,The annotation has no spatially usable bounding box. The original remains available; bbox-dependent views cannot be generated for this annotation.


,kind,selection_scope,finding_segment_id,available_verified_frames,required_verified_frames,available_verified_frame_indices_json,reason,status
0,progression_panel,segment,0531325b64674948__normal_clean_mucosa__00298,1.0,2.0,[10000],segment_has_fewer_than_two_available_verified_anchors,skipped
1,progression_panel,segment,0531325b64674948__normal_clean_mucosa__00300,1.0,2.0,[10003],segment_has_fewer_than_two_available_verified_anchors,skipped
2,progression_panel,segment,bca26705313a4644__normal_clean_mucosa__00038,1.0,2.0,[10003],segment_has_fewer_than_two_available_verified_anchors,skipped
3,progression_panel,segment,df6b47bafe5143f5__normal_clean_mucosa__00381,1.0,2.0,[10003],segment_has_fewer_than_two_available_verified_anchors,skipped
4,progression_panel,segment,0531325b64674948__normal_clean_mucosa__00301,1.0,2.0,[10004],segment_has_fewer_than_two_available_verified_anchors,skipped
5,progression_panel,segment,eb0203196e284797__angiectasia__00007,1.0,2.0,[10009],segment_has_fewer_than_two_available_verified_anchors,skipped
6,progression_panel,segment,df6b47bafe5143f5__normal_clean_mucosa__00383,1.0,2.0,[10011],segment_has_fewer_than_two_available_verified_anchors,skipped
7,progression_panel,segment,eb0203196e284797__angiectasia__00009,1.0,2.0,[10012],segment_has_fewer_than_two_available_verified_anchors,skipped
8,progression_panel,segment,0531325b64674948__normal_clean_mucosa__00306,1.0,2.0,[10013],segment_has_fewer_than_two_available_verified_anchors,skipped
9,progression_panel,segment,bca26705313a4644__normal_clean_mucosa__00041,1.0,2.0,[10013],segment_has_fewer_than_two_available_verified_anchors,skipped


SSD run directory: /content/mmvqa_sector8_v2/runs/492756b96163af4292c294969a184fedc1d66dc07a8a5239261e554a8889a3aa


## 9. SSD generation — a cell that can be safely rerun

After a failure, fix the cause and rerun this cell. Valid products are verified and reused. If CONFIG or code changes, rerun loading and planning first. Do not delete the cache because of an isolated error.

Across the complete reference S7 snapshot, the verified plan contains **118,201 new JPEGs** and **66,127 references to originals**. These are planning counts, not a claim that the images were generated in this notebook before delivery. `max_subjects=100` reduces the products for a trial run while retaining the originals inventory in the manifest.

In [ ]:
RUN = s8_process(PLAN, CONFIG)
display(pd.DataFrame(sorted(RUN['summary']['kind_counts'].items()), columns=['kind', 'views']))
print('Execution:', {k:v for k,v in RUN['execution'].items() if k in ('reused','restored','generated')})
print('Final split assignment:', RUN['state']['split_assignment_status'])


Load/copy previous S8 metadata + SHA-256:   0%|          | 0/4 [00:00<?, ?step/s]

Retained 66,127 original references; checking 118,201 derived products.


Verify SSD SHA / restore previous products:   0%|          | 0/118201 [00:00<?, ?product/s]

Copy/reuse sources to SSD + SHA-256:   0%|          | 0/66118 [00:00<?, ?file/s]

In [ ]:
# One preview per generated kind, read directly from SSD.
previewed = set()
for record in RUN['records']:
    if record.get('materialized', True) and record['kind'] not in previewed:
        print(record['kind'])
        with Image.open(Path(RUN['run_dir']) / record['relative_path']) as opened:
            preview = opened.copy()
            preview.thumbnail((1100, 600))
            display(preview)
        previewed.add(record['kind'])
print('Manifest tables:', list(RUN['tables']))


## 10. Actual model input

The example selects an available `fullcrop` recipe and loads two PIL images in the specified order. Change `SELECTED_VARIANT` for other variants. `QUESTION` is only a demonstration question; it is not added to the benchmark as validated QA.

The `MODEL_INPUT` object can be adapted to the selected model's processor. Image duplicates are not required for each experiment. For `temporal_multi`, the order and roles identify the target/context; actual offsets come from the S7 window.

In [ ]:
SELECTED_VARIANT = 'fullcrop'
QUESTION = 'Describe the observable visual characteristics in the provided images.'
choices = [r for r in RUN['benchmark']['examples'] if r['variant'] == SELECTED_VARIANT]
if choices:
    EXAMPLE = choices[0]
    MODEL_INPUT = s8_load_model_input(EXAMPLE, QUESTION, CONFIG, RUN['run_dir'])
    print('Model-facing keys:', list(MODEL_INPUT))
    print('Question:', MODEL_INPUT['question'])
    print('Ordered roles:', MODEL_INPUT['image_roles'])
    print('Number of actual images:', len(MODEL_INPUT['images']))
    for role, pixels in zip(MODEL_INPUT['image_roles'], MODEL_INPUT['images']):
        print(role, pixels.size)
        display(pixels)
else:
    print('Variant unavailable in the selected subset; inspect coverage and skip reasons.')

display(pd.DataFrame(sorted(RUN['benchmark']['coverage']['examples_by_variant'].items()),
                     columns=['variant','example_count']))
# Expected until a downstream stage supplies validated QA and final splits:
try:
    s8_assert_benchmark_ready(RUN['benchmark'])
except RuntimeError as error:
    print('Preparation-only status:', error)


## 11. Drive export with file confirmation

This cell authenticates Drive access from Colab. It saves only S8 products and metadata under the project root. The API verifies each destination and retransmits only missing or different files. The final catalog becomes visible after all products/metadata are confirmed. Rerun the cell after an interruption; do not restart generation if only export failed.

`RUN['state']['complete']` describes the local result. Cloud confirmation is `DRIVE_RECEIPT['complete']`. The manifest may appear through the mount later than through the API; API confirmation does not depend on a readback through the DriveFS cache.

In [ ]:
DRIVE_RECEIPT = s8_export_drive(RUN['state'], RUN['run_dir'], CONFIG)
print({key: DRIVE_RECEIPT.get(key) for key in ('status','complete','total','verified','pending')})
assert DRIVE_RECEIPT['complete'] is True


## 12. Local directory for potential GitHub publication

Prepares Parquet/JSON/schemas and product hashes. JPEGs remain on Drive. `product_hash_diff.json` compares the previous S8 catalog with the current one; `metadata_hash_diff.json` compares against the previous local review directory. These comparisons do not claim that GitHub already contains a particular version.

No Git commands or GitHub uploads are executed. A later commit/PR can publish the files in this directory after your verification in Colab. The `.json.gz` and `.jsonl.gz` archives reduce metadata size and preserve deterministic bytes.

In [ ]:
GITHUB_REVIEW_DIR = s8_prepare_github_review(RUN, CONFIG)
oversized = [str(path.relative_to(GITHUB_REVIEW_DIR))
             for path in Path(GITHUB_REVIEW_DIR).rglob('*') if path.is_file() and path.stat().st_size >= 90*1024**2]
print('Prepared metadata directory:', GITHUB_REVIEW_DIR)
print('Files requiring sharding before a normal GitHub upload:', oversized)
print('No GitHub write was performed.')


## Checks performed before delivery

- Actual loading of the S7 snapshot: hashes, schema 3 catalog, all three tables, and column types.
- Planning across all 47,239 annotations and 66,127 files, without reading your images.
- 14 renderer tests, including explicit simulations of GPU retry/subbatch/spill behavior.
- 13 integration tests using synthetic images, including integrity, resumption, restoration in a new runtime, and simulated export.
- Separate tests for staging and API file confirmation; no transfer to your Drive during this verification.

**Still to be verified in Colab:** actual CUDA execution, rendering on your images, and real Drive authentication/transfers. S8 does not create the QA benchmark or final split assignments.

### GPU setup and progress update

- 18 GPU/bootstrap regression tests: duplicate package repair, restart safety after partial failure or interruption, complete error diagnostics, explicit hybrid permission, and the retained GPU-only restriction.
- 5 storage progress regression tests: bar closure, API pagination, authentication imports, upload ordering, idempotence, and corrupted-file rejection.
- 4 skipped-product tests: singleton segments, distinct frame counting, valid progression panels, and summary counts.
- The 13 integration tests still pass, including reruns, selective repair, restoration, source-hash rejection, and simulated interrupted Drive export.
- The full pinned S7 plan still contains 66,127 original references, 118,201 derived JPEGs, and 47,239 subjects. All 118 previously defined functions/classes remain in the notebook.
- Every code cell parses, notebook metadata validates, all runtime imports are in section 2, and the embedded local self-test executes after the centralized imports.

These checks use local/synthetic data and simulated GPU/cloud responses where needed. They do not claim a real Colab GPU run or a transfer to your Drive. Run the early GPU preflight in your Colab session before full processing.
